# 解説付きの版について

このノートは、`材料別_温度シフトファクターの検討_ARD対応Matern52＋線形＋ホワイトノイズ.ipynb`（元のノート）の**コードの各行に、Python 初心者向けの解説を付けた版**です。

- **`# ▶` で始まる行が、すぐ下の行の解説**です（灰色のコメントで、実行には影響しません）。もとからあるコメント（`#` の後ろ）は、元のノートの説明です。
- コードそのものは元のノートと同じです（生成するときに、構文が元のノートと同じことを確かめています）。**実行すると、元のノートと同じように、実行の始めにダイアログで選んだ学習結果フォルダに、同じ結果が保存されます（同じフォルダを選ぶと、上書きされます）。**
- 解説を付けていない行：空行、コメントだけの行、閉じかっこ `)`・`]`・`}` だけの行、複数行にまたがる文字列（`"""…"""` など）の 2 行目以降。
- このノートは、`04_補助スクリプト/build_mat_shift_nb_commented.py` で作っています。元のノートを作り直したときは、このスクリプトを実行し直してください（元のセルの中身が変わっていたら、エラーで知らせます。そのときは解説ファイル `04_補助スクリプト/mat_shiftfactor_nb_解説/` を直します）。

## Python の読み方の基本（このノートによく出る書き方）

| 書き方 | 意味 |
|---|---|
| `x = 5` | 変数 x に 5 を入れる（`=` は「等しい」ではなく「入れる」） |
| `# …` | コメント（メモ）。実行されない |
| `"文字"`・`'文字'` | 文字列（テキスト） |
| `f"{x} 行"` | f 文字列。`{}` の中の変数や式の値が、文字の中に埋め込まれる |
| `[1, 2, 3]` | リスト（順番のある入れ物）。`a[0]` で最初の要素（番号は 0 から） |
| `(1, 2)` | タプル（中身を変えられないリスト） |
| `{"PP": 1, "MIX": 2}` | 辞書（名前と値の組の入れ物）。`d["PP"]` で 1 が取れる |
| `import numpy as np` | 部品の集まり（ライブラリ）を読み込み、`np` という短い名前で使う |
| `def 名前(引数):` | 関数（処理のまとまり）を作る。字下げした行が中身。`return` で結果を返す |
| `名前(値)` | 関数を呼ぶ（実行する） |
| `for x in リスト:` | リストの要素を 1 つずつ x に入れて、字下げした行をくり返す |
| `if 条件:` ／ `elif` ／ `else:` | 条件が成り立つときだけ、字下げした行を実行する |
| `[式 for x in リスト]` | リスト内包表記。リストの各要素で式を計算して、新しいリストを作る |
| `lambda x: 式` | 名前のない小さな関数 |
| `a.b` | a の中の b（a が持つ値や関数） |
| `np.…` | NumPy（数値計算のライブラリ）の機能。配列（数の並び）をまとめて計算する |
| `pd.…`・`DataFrame` | pandas（表を扱うライブラリ）。`DataFrame` は Excel の表のようなもの。`df["列名"]` で列を取り出す |
| `plt.…`・`fig`・`ax` | matplotlib（グラフを描くライブラリ）。`fig` は図全体、`ax` はその中の 1 つのグラフ |
| `**kw` | 辞書 kw の中身を、名前付きの引数としてまとめて渡す |
| `a if 条件 else b` | 条件が成り立てば a、そうでなければ b |
| `with open(…) as f:` | ファイルを開き、字下げした行が終わったら自動で閉じる |

# 材料別の温度シフトファクターの検討（ARD Matern 5/2 ＋ 線形 ＋ ホワイトノイズ）

温度のデータ列（**加熱シリンダ温度 Tset**・**流動中樹脂温度 Tff**〔型内樹脂温度。発表資料では「金型内樹脂最高温度」〕）ごとに、温度のシフトファクターを推定して比べ、**どの列をシフトファクターに使うのが適しているか**を調べる。2026-10-27 に、学習用のノート（`../材料別_温度シフト機械学習コード/材料別_温度シフトGPR_ARD対応Matern52＋線形＋ホワイトノイズ.ipynb`）から、この部分を分けた。データ（5data で学習し、3data で予測検証）とモデル（GP のカーネル、シフトファクターの式、プロファイル尤度）は、学習用のノートと同じ。**この結果を見て、学習用のノートの設定 `SHIFT_COL` を決める**。

**比べるモデル**（設定 `SHIFT_COLS` の列ごとに 1 つと、基準の C。記号は、本体の検討の B_1 などとは別物）

| 列（`SHIFT_COLS` の書き方） | モデル | 式 | 結果のフォルダ |
|---|---|---|---|
| `"加熱シリンダ温度[℃]"` | **B_Tset** | ln η = ln a_v(Tset) ＋ f( ln γ̇ ＋ ln a_h(Tset), ln P3_3, ln Tff ) | `B_Tset_温度シフトに加熱シリンダ温度採用/` |
| `"流動中樹脂温度(℃)"` | **B_Tff** | ln η = ln a_v(Tff) ＋ f( ln γ̇ ＋ ln a_h(Tff), ln P3_3, Tset ) | `B_Tff_温度シフトに型内樹脂温度採用/` |
| （基準。いつも学習する） | **C** | ln η = f( ln γ̇, ln P3_3, ln Tff, Tset ) | `C_温度シフトなし/` |

どのモデルも、両方の温度を使っていて、違うのは**シフトに使う温度**だけ（シフトに使わないほうの温度は、GP の入力に入る）。f は GP（ARD Matern 5/2 ＋ 線形項 ＋ ホワイトノイズ）。**横のシフト** a_h は「温度が変わると、同じ粘度になるせん断速度が a_h 倍ずれる」、**縦のシフト** a_v は「同じせん断速度で、粘度が a_v 倍になる」を表す（Tref で 1）。PP・粉砕材混合 PP は**アレニウス型**（θ = E_h・E_v [kJ/mol]）、GPPS は **WLF 型**（A2 = 51.6 K、D2 = Tg〔仮置き 100 ℃〕を固定し、θ = A1_h・A1_v を推定）。

**比べ方**
- **尤度**：θ = 0（その温度をシフトに使わない）との対数周辺尤度の差 ΔLML。**3.0 以上**なら、2 つの係数を足した効果が、尤度比検定の 5% 基準（2·ΔLML > 5.99）で認められる（その列のシフトが、データで支持される）。
- **シフトファクターの大きさと向き**：低温ほど粘い向きか、温度ごとの曲線がシフトで重なるか（マスターカーブ）。
- **3data の予測**：RMSE（ln η）が、基準の C の 1.1 倍以内か。95% 予測区間に入る割合（被覆率）。

**読むときの注意**：P3_3 と Tff は、流れの結果として測られる量で、成形の前には分からない。Tff は、せん断発熱で射出速度とともに変わるので、Tff でのシフトは、温度の効果だけでなく、曲線の形の調整も兼ねうる。どちらの温度が適するかは、入力の取り方（`INPUT_MODE`）にも左右される（10-23 の結果：ショットごとの入力では B_Tset、条件平均の入力では B_Tff が支持された）。

## 使い方（このノートを一人で実行して、結果を判断するために）

**初めての人へ**：1 つ上のフォルダ（`材料別_温度シフトファクター検討/`）の説明書 `説明書_材料別温度シフトGPRのノート.pdf` に、3 つのノートの目的・関係・使えるデータ（学習ファイルの条件）・使い方をまとめてある。先に読む。

1. 最初の「設定エリア」のセルで、比べる列 `SHIFT_COLS`（既定は 2 つの温度の列）、入力の取り方 `INPUT_MODE`、材料 `MATERIALS` を決める。ほかの設定も、このセルだけを直して、最初から実行し直す。
2. メニューの「すべて実行」で、上から順に実行する。カーネルは、このプロジェクトの `.venv` を選ぶ。
3. **実行の始めに（「2. データ」のセルで）、材料ごとにダイアログが 3 回開く**：①学習データ（5data）の CSV、②検証データ（3data）の CSV、③学習結果フォルダ（**材料ごとに別のフォルダ**を選ぶ）。学習用のノートと同じ学習結果フォルダを選ぶと、結果が 1 か所にそろう（このノートの結果は、別のフォルダ `シフトファクターの検討/` に入るので、学習用のノートの結果は上書きしない）。全部選ぶと、あとは最後まで自動で進む（目安：3 材料で、ショットごとの入力は約 40 分、条件平均の入力は約 80 分。大部分は「3. プロファイル尤度」）。
   - ダイアログが見当たらないときは、VS Code の後ろに隠れているので、タスクバーで探す。キャンセルすると、そこで止まる。
   - 毎回同じファイルを使うときは、設定 `FILES` に書くと、ダイアログは出ない（プロジェクトの外のファイルは、ここに書かずにダイアログで選ぶ）。
4. 結果は、学習結果フォルダの **`<版>/全厚み総合/シフトファクターの検討/`** に保存される（版は `INPUT_MODE` で、`全ショット版` か `条件ごとの平均版`）。
   - このフォルダの直下：モデルを比べる表と図（`シフトファクターの推定.csv`〔θ と ΔLML〕、`シフトファクター.png`、`流動曲線_シフト前.png`、`シフトファクターの倍率.csv`、`ハイパーパラメータ.csv`、`予測検証_3data_指標.csv`、`予測検証_RMSEの比較.png`、`判定の一覧.csv`、`まとめ.csv`）
   - その下のモデルのフォルダ（`B_Tset_温度シフトに加熱シリンダ温度採用/` など）：プロファイル尤度、マスターカーブ、パリティ図、3data の予測値、学習済みモデル
   - 学習結果フォルダの直下の `データ確認/` には、データの概要と、一覧で除いた行を書く。
   - **図は、すべて 600 dpi の 1 枚だけ**（設定 `FIG_DPI`）。
5. 各節の最後に、**判定**が出る（`>> 判定：OK` か `>> 判定：★要確認`）。★ は「想定と違う、または結果を読むときに注意が要る点」で、エラーではない。例えば、ある列の ΔLML が基準に届かないときは、その列のシフトが支持されない、という結果。
6. 最後の「まとめ」に、材料ごとの **「学習用のノートの SHIFT_COL の候補」** が出る（参考。ΔLML が基準以上のシフトのうち、差が最も大きい列。どれも届かなければ、`None`〔C〕も候補）。3data の誤差も見て、決めるのはユーザー。
7. このノートが保存した学習済みモデルを、**厚みごとの比較のノート**（`材料別_温度シフトGPR_ARD対応Matern52＋線形＋ホワイトノイズ_厚みごとの比較.ipynb`）が読む（厚みごとに、温度シフトが効いているか、どちらの温度が適しているかを調べる）。そのノートを使うときは、`SHIFT_COLS` を既定（2 つの列）のままにする。
8. 外れ値の検知と P3_3 の性質は、学習用のノートで確かめる（このノートも、問題データ一覧 `05_材料ごとの機械学習データと予測検証データ/問題データ一覧.csv` の「除外」は、同じように除く）。
9. 用語や図の見方は、説明書 `05_材料ごとの機械学習データと予測検証データ/学習結果説明_材料別温度シフトGPR.pdf` にある（10-26 までの結果で書いたもの）。

## 1. 設定

In [ ]:
# セル1：設定エリア  ▼▼▼ 変えてよいのは、ここだけ ▼▼▼
# ▶ pathlib（フォルダやファイルの場所を扱う標準の道具）から Path を読み込む。下の REPO で、フォルダを上へたどるのに使う
from pathlib import Path      # （下の REPO で、フォルダをたどるのに使う）
# ▶ 比べるシフトファクターの列のリスト（学習ファイルの列名。学習用のノートの SHIFT_COL と同じ書き方）。列ごとに、シフトのあるモデルを作る。
# ▶ 基準として、シフトのない C も、いつも学習する
SHIFT_COLS = ["加熱シリンダ温度[℃]", "流動中樹脂温度(℃)"]   # 比べるシフトファクターの列（学習ファイルの列名。学習用のノートの SHIFT_COL と同じ書き方）。基準として、シフトなしの C も学習する
# ▶ シフトの形のリスト。"B" は、横（せん断速度の向き）と縦（粘度の向き）を、別々の係数でずらす形。"A" は、縦だけをずらす形（["B", "A"] にすると、両方の形で作って比べる）
STRUCTS = ["B"]               # シフトの形："B"＝横・縦を別々に推定／"A"＝縦だけ（対案。["B", "A"] で両方。A_Tset・A_Tff ができる）
# ▶ 図をノートの中にも表示するモデルの名前（ほかのモデルの図も、ファイルには保存する）
MAIN_KEY = "B_Tset"           # 図をノートにも表示するモデル（B_Tset・B_Tff・C など。ほかのモデルの図も保存する）
# ▶ 変数 SEED に 42 を入れる（= は「入れる」の意味）。乱数（でたらめな数）の出方を決める「たね」。同じ値なら、何度実行しても同じ結果になる
SEED = 42                     # 乱数シード
# ▶ ノイズ分散（同じ条件でも、測るたびに出るばらつき）の下限。1e-6 は 1×10⁻⁶（= 0.000001）の書き方
NOISE_LB = 1e-6               # ノイズ分散の下限（6月のノートは 1e-3）
# ▶ True（はい）か False（いいえ）を入れる。True なら、GPR に「直線の傾向」（線形項）を足して、学習の範囲の外でも傾向を延ばせるようにする
USE_LINEAR = True             # True：6月のカーネル（ARD Matern 5/2 ＋ ホワイトノイズ）に線形項（DotProduct）を足す（3data は射出速度の両端の外挿のため）／False：6月と同じ
# ▶ 入力の取り方。"shot"（既定）は、ショットごとの値をそのまま入れる（1 ショットを 1 つのデータとして学習する。結果は「全ショット版」のフォルダ）。
# ▶ "cond" は、条件ごとのショットの平均を入れる（結果は「条件ごとの平均版」のフォルダ）
INPUT_MODE = "shot"           # "shot"：γ̇・P3_3・Tff をショットごとの値で入れる（既定。結果は 全ショット版/）／"cond"：条件（設定温度・厚み・射出速度）ごとの平均で入れる（結果は 条件ごとの平均版/）
# ▶ 最終モデルの学習で、出発点を変えて最適化（一番合う値を探す計算）をやり直す回数。多いほど良い解に着きやすいが、時間がかかる
N_RESTARTS = 20               # 最終モデルの学習での、最適化のやり直し回数（LML が最大の解を採る）
# ▶ シフトの係数を調べる格子（下の SHIFT_CFG）の各点での、最適化のやり直しの回数（最初の 1 回に追加する回数）
PROFILE_RESTARTS = 1          # プロファイル尤度の格子の各点での、やり直し回数（0 にすると約半分の時間。尤度の面ががたつきやすい）
# ▶ True なら、問題データ一覧で「除外」とした行を、学習・検証のデータから除く
USE_EXCLUSIONS = True         # True：問題データ一覧（05 フォルダ）の「除外」を除く／False：全部使う（0 以下の値の行だけは、対数がとれないので除く）
# ▶ リポジトリ直下のフォルダの場所を、自動で探す。Path.cwd() は今いるフォルダ（このノートのフォルダ）、.parents はその上のフォルダの並び。
# ▶ 下から順に見て、「04_補助スクリプト」フォルダがある最初のフォルダを選ぶ（next(…) は、条件に合う最初の 1 つを取り出す）。str(…) で文字列にする
REPO = str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "04_補助スクリプト").is_dir()))   # リポジトリ直下（このノートの場所から上へたどって、04_補助スクリプト があるフォルダを探す）
# ▶ 調べる材料の名前のリスト（[ ] で囲んだ並び）。減らすと、その材料だけを計算する。材料ごとに、実行の始めにダイアログで、ファイルとフォルダを選ぶ
MATERIALS = ["PP", "MIX", "GPPS"]    # 調べる材料（PP：バージン PP、MIX：粉砕材混合 PP、GPPS）。材料ごとに、実行の始めにダイアログで、学習データ・検証データの CSV と、学習結果フォルダを選ぶ
# ▶ 辞書（{ 名前: 値, … }）。材料の短い名前（"PP" など）から、05 フォルダの中の材料フォルダの名前を引けるようにする（ダイアログを開いたときに、最初に見せるフォルダ）
MAT_FOLDERS = {"PP": "01_バージンPP再々実験", "MIX": "02_粉砕材混合PP実験", "GPPS": "03_GPPS実験"}   # 05 フォルダの中の材料フォルダ名（ダイアログの最初のフォルダ）
# ▶ ダイアログを使わずに、ファイルとフォルダを決めるときの辞書。空（{}）なら、すべてダイアログで選ぶ。
# ▶ 書くときは、材料ごとに "学習"（5data の CSV）・"検証"（3data の CSV）・"結果"（学習結果フォルダ）の場所を書く（r"…" は、\ をそのまま文字として扱う書き方）
FILES = {}                    # ダイアログを使わずに決めるとき：{"PP": {"学習": r"…\結合_03_pp_max_5data_外れ値なし.csv", "検証": r"…", "結果": r"…"}}（相対パスはリポジトリ直下から）。書いていないものは、ダイアログで選ぶ
# ▶ 材料ごとのシフトファクターの設定（辞書の中に辞書が入った形）。ここで、式の型・基準温度・係数を調べる範囲を決める（係数そのものは、データから推定する）
SHIFT_CFG = {                 # 材料ごとのシフトファクターの型、基準温度 Tref [℃]（None なら中央の設定温度）、尤度を調べる範囲（粗い格子：(下限, 上限, 刻み)。h が横、v が縦）
    # ▶ PP：アレニウス型、基準温度 210 ℃。横の係数 E_h は −20〜40 kJ/mol、縦の係数 E_v は −20〜60 kJ/mol を、5 刻みで調べる（( ) は (下限, 上限, 刻み) のタプル）
    "PP":   {"type": "arrhenius", "Tref": 210.0, "h_grid": (-20.0, 40.0, 5.0), "v_grid": (-20.0, 60.0, 5.0)},          # θ = E [kJ/mol]
    # ▶ 粉砕材混合 PP：PP と同じ設定
    "MIX":  {"type": "arrhenius", "Tref": 210.0, "h_grid": (-20.0, 40.0, 5.0), "v_grid": (-20.0, 60.0, 5.0)},
    # ▶ GPPS：WLF 型、基準温度 220 ℃。Tg（ガラス転移温度。仮に 100 ℃）と A2 = 51.6 は固定し、係数 A1（横・縦）を −20〜40 の範囲で調べる
    "GPPS": {"type": "wlf", "Tref": 220.0, "Tg": 100.0, "A2": 51.6, "h_grid": (-20.0, 40.0, 5.0), "v_grid": (-20.0, 40.0, 5.0)},   # θ = A1 [−]。D2 = Tg [℃]（仮置き。分かれば直す）、A2 [K] は固定
}
# ▶ 粗い格子で一番良かった点の周りを、細かく調べ直す範囲と刻み（±4 を 1 刻み）。構造 B 用
FINE_B = (4.0, 1.0)           # B の細かい格子：最良の点の ±4 を 1 刻み
# ▶ 構造 A（縦だけずらす形。STRUCTS に "A" を入れたときだけ使う）用の細かい格子（±2.5 を 0.5 刻み）
FINE_A = (2.5, 0.5)           # A の細かい格子：最良の点の ±2.5 を 0.5 刻み
# ▶ 一番良い点が細かい格子の端に来たとき、その点を中心に格子を取り直す回数の上限
MAX_REFINE = 4                # 最良の点が細かい格子の端に当たったとき、その点を中心に細かい格子を取り直す回数の上限
# ▶ 同時に計算する数（並列計算）。パソコンの CPU のコアの数に合わせる
N_JOBS = 8                    # プロファイル尤度を並列に計算する数（CPU の数に合わせる）
# ▶ 保存する図（PNG）の解像度。dpi は 1 インチあたりの点の数。600 なら、報告書・発表資料にそのまま使える細かさ（すべての図を、この解像度の 1 枚だけで保存する）
FIG_DPI = 600                 # 保存する図の解像度（すべての図を、この解像度の 1 枚だけで保存する）
# ▶ True なら、図をノートの中にも表示する（False なら、ファイルに保存するだけ）
SHOW_FIGS = True              # True：図をノートにも表示する（False：保存だけ）
# ▲▲▲ 設定エリア終了 ▲▲▲

In [ ]:
# セル2：ライブラリ、フォント
# ▶ Jupyter の命令（Python の文ではない）。グラフを、ノートの中（このセルの下）に表示する設定
%matplotlib inline
# ▶ os：ファイルやフォルダの場所を扱う部品（Python に最初から入っている標準ライブラリ）を読み込む
import os
# ▶ sys：Python 自体の情報（バージョンなど）を扱う部品を読み込む
import sys
# ▶ time：時間を測る部品を読み込む（計算にかかった時間の表示に使う）
import time
# ▶ NumPy（数値計算のライブラリ）を読み込み、np という短い名前で使えるようにする
import numpy as np
# ▶ pandas（表を扱うライブラリ）を読み込み、pd という短い名前で使えるようにする
import pandas as pd
# ▶ matplotlib（グラフを描くライブラリ）の pyplot を、plt という短い名前で使えるようにする
import matplotlib.pyplot as plt
# ▶ フォント（文字の書体）を扱う部品を、fm という名前で使えるようにする
import matplotlib.font_manager as fm
# ▶ 三角形の網目を扱う部品を、mtri という名前で使えるようにする（3次元グラフの面を描くのに使う）
import matplotlib.tri as mtri
# ▶ display：表（DataFrame）を、ノートの中に見やすく表示する関数を読み込む
from IPython.display import display

# ▶ ノートに表示するグラフの解像度を 80 dpi にする（rcParams は matplotlib の設定の辞書）
plt.rcParams["figure.dpi"] = 80
# ▶ pandas の表を表示するときの横幅（文字数）を 220 に広げる
pd.set_option("display.width", 220)
# ▶ 表を表示するときの列の数の上限を 40 にする（列が多くても、省略せずに表示する）
pd.set_option("display.max_columns", 40)
# ▶ try：下の字下げした行を試しに実行する。エラーが出たら、except の行へ進む
try:
    # ▶ 日本語フォント IPAexGothic を探す（fallback_to_default=False：見つからなければ、ほかのフォントで代用せずにエラーにする）
    fm.findfont("IPAexGothic", fallback_to_default=False)
# ▶ 上で「フォントが見つからない」エラー（ValueError）が出たときだけ、下の行を実行する
except ValueError:
    # ▶ raise：わざとエラーを出して止める。メッセージで、フォントの入れ方を知らせる
    raise RuntimeError("日本語フォント IPAexGothic が見つかりません。インストールしてからカーネルを再起動してください。")
# ▶ グラフの文字のフォントを IPAexGothic にする（その中にない文字は DejaVu Sans で表示する）
plt.rcParams["font.family"] = ["IPAexGothic", "DejaVu Sans"]
# ▶ グラフのマイナス記号を、文字化けしない普通の「-」で表示する
plt.rcParams["axes.unicode_minus"] = False
# ▶ print：画面に表示する。使っている Python の場所とバージョンを表示する（split()[0] は、空白で区切った最初の部分）
print("Python:", sys.executable, "/", sys.version.split()[0])

### 共通の部品

ファイルを選ぶダイアログ、データの読み込み、問題データ一覧の「除外」、結果の置き場所、シフトファクター、モデルの学習と予測、プロファイル尤度、評価、図の保存。中身は `04_補助スクリプト/mat_common.py` と同じ（学習用のノートと同じ）。

In [ ]:
# セル：共通の部品（読み込み・問題データ一覧・シフトファクター・モデル・評価・図）。04_補助スクリプト/mat_common.py の中身。直すときは、そのファイルを直して、ノートを作り直す
# ▶ glob：パターン（* など）に合うファイルを探す部品を読み込む
import glob
# ▶ os：ファイルやフォルダの場所を扱う部品を読み込む
import os
# ▶ re：正規表現（文字の並びの型で、文字を探したり取り出したりする道具）の部品を読み込む
import re
# ▶ time：時間を測る部品を読み込む
import time
# ▶ warnings：警告の表示を調整する部品を読み込む
import warnings

# ▶ NumPy（数値計算のライブラリ）を np という名前で読み込む
import numpy as np
# ▶ pandas（表を扱うライブラリ）を pd という名前で読み込む
import pandas as pd
# ▶ scikit-learn（機械学習のライブラリ）から、「最適化が収束しなかった」という警告の種類を読み込む
from sklearn.exceptions import ConvergenceWarning
# ▶ scikit-learn のガウス過程回帰（GPR）の本体を読み込む
from sklearn.gaussian_process import GaussianProcessRegressor
# ▶ GPR のカーネルの部品を読み込む：定数（C という名前で使う）、線形（DotProduct）、Matern、足し算（Sum）、ホワイトノイズ
from sklearn.gaussian_process.kernels import ConstantKernel as C, DotProduct, Matern, Sum, WhiteKernel
# ▶ 評価の指標：平均二乗誤差（MSE）と決定係数（R²）を計算する関数を読み込む
from sklearn.metrics import mean_squared_error, r2_score
# ▶ 標準化（平均 0・標準偏差 1 にそろえる）の道具を読み込む
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------------- 設定（ノートの設定エリアの値があれば、それを使う）
# ▶ globals() は、これまでに作った変数の辞書。.get("SEED", 42) は、設定エリアで SEED を作っていればその値を、なければ 42 を使う
# ▶ （この部品を、ノートの外のスクリプトから使うときのための書き方。下の行も同じ）
SEED = globals().get("SEED", 42)
# ▶ ノイズ分散の下限（設定エリアの値。なければ 1e-6）
NOISE_LB = globals().get("NOISE_LB", 1e-6)                  # ノイズ分散の下限（6月のノートは 1e-3）
# ▶ 最終モデルの学習の、最適化のやり直し回数（設定エリアの値。なければ 20）
N_RESTARTS = globals().get("N_RESTARTS", 20)                # 最終モデルの学習での、最適化のやり直し回数（ハイパーパラメータの局所解が多いので、6月の 5 回より多くする）
# ▶ プロファイル尤度の格子の各点での、やり直し回数（設定エリアの値。なければ 1）
PROFILE_RESTARTS = globals().get("PROFILE_RESTARTS", 1)     # プロファイル尤度の格子の各点での、やり直し回数（前の点の解も初期値に使う）
# ▶ 線形項を足すか（設定エリアの値。なければ True）
USE_LINEAR = globals().get("USE_LINEAR", True)              # True：6月のカーネルに線形項（DotProduct）を足す
# ▶ 問題データ一覧の「除外」を使うか（設定エリアの値。なければ True）
USE_EXCLUSIONS = globals().get("USE_EXCLUSIONS", True)      # 問題データ一覧の「除外」を使うか
# ▶ リポジトリ直下の場所（設定エリアの値〔自動で探した場所〕。なければ、環境変数 NB_REPO、それもなければ ".."）
REPO = globals().get("REPO", os.environ.get("NB_REPO", ".."))
# ▶ 並列計算の数（設定エリアの値。なければ 8）
N_JOBS = globals().get("N_JOBS", 8)
# ▶ 保存する図の解像度（設定エリアの値。なければ 600 dpi）。すべての図を、この解像度の 1 枚だけで保存する
FIG_DPI = globals().get("FIG_DPI", 600)                     # 保存する図の解像度。すべての図を、この解像度の 1 枚だけで保存する（報告書・発表資料用。2026-10-27 から）
# ▶ 図をノートに表示するか（設定エリアの値。なければ True）
SHOW_FIGS = globals().get("SHOW_FIGS", True)                # 図をノートに表示するか（False なら保存だけ）
# ▶ 入力の取り方（設定エリアの値。なければ "shot"＝ショットごと）
INPUT_MODE = globals().get("INPUT_MODE", "shot")            # "shot"：γ̇・P3_3・Tff をショットごとの値で入れる（6月のノートと同じ。既定）／"cond"：条件（設定温度・厚み・射出速度）ごとの平均で入れる（参考）
# ▶ 細かい格子を取り直す回数の上限（設定エリアの値。なければ 4）
MAX_REFINE = globals().get("MAX_REFINE", 4)                 # プロファイル尤度で、最良の点が細かい格子の端に当たったとき、その点を中心に細かい格子を取り直す回数の上限
# ▶ 構造 B の細かい格子の範囲と刻み（設定エリアの値。なければ ±4 を 1 刻み）
FINE_B = globals().get("FINE_B", (4.0, 1.0))               # 構造 B の細かい格子：最良の点の ±4 を 1 刻み
# ▶ 構造 A の細かい格子の範囲と刻み（設定エリアの値。なければ ±2.5 を 0.5 刻み）
FINE_A = globals().get("FINE_A", (2.5, 0.5))                # 構造 A の細かい格子：最良の点の ±2.5 を 0.5 刻み
# ▶ 外れ値の規則 f の基準 1：z（予測からのずれ ÷ 予測のばらつき）の大きさが 3.5 を超える
OUTLIER_Z = globals().get("OUTLIER_Z", 3.5)                 # 規則 f：|z| がこれを超え、かつ |残差|（ln η）が次の値を超える条件
# ▶ 外れ値の規則 f の基準 2：ずれ（ln η）の大きさが 0.1（粘度で約 10%）を超える
OUTLIER_RESID = globals().get("OUTLIER_RESID", 0.1)
# ▶ 問題の候補の規則 g の基準：P3_3 が、条件の中央値から 20% 以上離れたショット
P33_DEV = globals().get("P33_DEV", 0.2)                     # 規則 g：P3_3 が条件の中央値から、この割合以上離れたショットを候補にする
# ▶ 最適化の「収束しなかった」という警告を表示しない（値が探す範囲の端に張り付いたかは、ハイパーパラメータの表に記録する）
warnings.filterwarnings("ignore", category=ConvergenceWarning)   # 下限・上限への張り付きは、ハイパーパラメータの表に記録する

# ---------------------------------------------------------------- データの場所と列
# ▶ os.path.join：フォルダ名をつないで、パス（ファイルの場所）を作る。ここでは 05 フォルダの場所
DATA_ROOT = os.path.join(REPO, "05_材料ごとの機械学習データと予測検証データ")
# ▶ 問題データ一覧（CSV）の場所
PROBLEM_CSV = os.path.join(DATA_ROOT, "問題データ一覧.csv")
# ▶ 材料の短い名前 → フォルダ名の辞書（設定エリアにあれば、それを使う）
MAT_FOLDERS = globals().get("MAT_FOLDERS", {"PP": "01_バージンPP再々実験", "MIX": "02_粉砕材混合PP実験", "GPPS": "03_GPPS実験"})
# ▶ 材料の短い名前 → 表示用の名前の辞書（問題データ一覧の「材料」列の書き方に合わせる）
MAT_NAMES = {"PP": "PP", "MIX": "粉砕材混合PP", "GPPS": "GPPS"}     # 問題データ一覧の「材料」列の名前
# ▶ 組（5data・3data）→ データのフォルダ名の形。* は「何文字でもよい」の意味
GROUP_DIRS = {"5data": "学習データ_*", "3data": "検証データ_*"}      # 材料フォルダの中の、学習データと検証データのフォルダ（CSV が 1 つずつ）
# ▶ CSV の列名（日本語）→ プログラムの中で使う短い列名の辞書。例：樹脂粘度η34 → eta、せん断速度γ34 → gd
COLS = {"元ファイル": "file", "キャビ厚": "H", "射出速度[mm/s]": "v", "樹脂粘度η34(Pa・s)": "eta", "せん断速度γ34(1/s)": "gd",
        # ▶ （辞書の続き）圧力の 3 列（P3_1・P3_2・P3_3）の名前
        "流動中樹脂圧力P3_1(MPa)": "P31", "流動中樹脂圧力P3_2(MPa)": "P32", "流動中樹脂圧力P3_3(MPa)": "P33",
        # ▶ （辞書の続き）加熱シリンダ温度（ファイルによって [℃] と (℃) の 2 通りの書き方がある）と、流動中樹脂温度の名前
        "加熱シリンダ温度[℃]": "Tset", "加熱シリンダ温度(℃)": "Tset", "流動中樹脂温度(℃)": "Tff"}
# ▶ 必ずなければいけない列のリスト
REQUIRED = ["file", "H", "v", "eta", "gd", "P33", "Tff"]          # 加熱シリンダ温度の列がなければ、元ファイル名から取る
# ▶ 元ファイル名（例：001_190℃_020_0.5mm.csv）を読み取る正規表現。( ) の中が取り出す部分（ショット番号・設定温度・射出速度・厚み）
# ▶ \d は数字、{3} は 3 文字、+ は 1 文字以上、^ と $ は先頭と末尾。r"…" は、\ をそのままの文字として扱う文字列
RE_SRC = r"^(\d{3})_(\d+)℃_(\d{3})_([\d.]+)mm[A-Za-z]*\.csv$"     # ショット_設定温度_射出速度_厚み（GPPS は末尾に NR が付く）
# ▶ 気体定数 R = 8.314 J/(mol·K)（アレニウス型の式に使う）
R_GAS = 8.314                                                      # 気体定数 [J/mol/K]
# ▶ 空の辞書を作る。後で、(材料, 組) ごとに読み込んだ表を入れる
D = {}             # (材料, 組) → DataFrame。組は "5data"（学習）と "3data"（検証）
# ▶ 空の辞書。読んだファイルの情報（行数など）を入れる
FILE_INFO = {}     # (材料, 組) → 読んだファイルの情報
# ▶ 空の辞書。問題データ一覧の「除外」で除いた行を入れる
REMOVED = {}       # (材料, 組) → 一覧の「除外」で除いた行（元ファイルと ID）
# ▶ 空の辞書。0 以下の値で除いた行を入れる
INVALID = {}       # (材料, 組) → 0 以下の値（η・γ̇・P3_3）で対数がとれず、一覧に登録がないまま除いた行
# ▶ 空のリスト。各節の判定（OK／★要確認）を、順に記録していく
JUDGES = []        # 判定の記録（まとめの表に使う）
# ▶ 空の辞書。材料ごとに、選んだ学習データ・検証データの CSV と、学習結果フォルダの場所を入れる（下の choose_inputs で決める）
SOURCES = {}       # 材料 → {"5data": 学習データの CSV, "3data": 検証データの CSV, "結果": 学習結果フォルダ}（choose_inputs で決める）

# ---------------------------------------------------------------- 結果の置き場所（2026-10-27 からの構成）
# ▶ 学習結果フォルダの直下のフォルダの名前の辞書：入力の取り方（"cond"／"shot"）→「条件ごとの平均版」／「全ショット版」。図のタイトルにも使う
VERSION_DIRS = {"cond": "条件ごとの平均版", "shot": "全ショット版"}   # 学習結果フォルダの直下の分け方（入力の取り方 INPUT_MODE ごと）。図のタイトルと、表の「入力」の列にも使う
# ▶ その下のフォルダの名前を、2 つの変数にまとめて入れる（a, b = x, y の書き方）。「全厚み総合」は全厚みのデータで学習した結果、「厚みごと」は厚みごとの比較のノートの結果
THICK_TOTAL, THICK_EACH = "全厚み総合", "厚みごと"   # その下の分け方：全厚みの 5data で 1 つのモデルを学習した結果（学習用・シフトファクター検討用のノート）／厚みごとの比較のノートの結果
# ▶ 「全厚み総合」の下の、シフトファクター検討用のノートの結果のフォルダの名前
STUDY_DIR = "シフトファクターの検討"                  # 全厚み総合/ の下の、シフトファクター検討用ノートの結果のフォルダ
# ▶ 10-26 までの結果のフォルダの名前（入力の取り方 → 名前）。その結果は残してあり、発表資料・ポスターのスクリプトが、この名前で読む
MODE_JA = {"shot": "ショットごとの入力", "cond": "条件平均の入力"}   # 10-26 までの結果のフォルダの名前（<材料フォルダ>_学習結果/ の下。その結果は、この構成のまま残す。発表資料・ポスターのスクリプトが読む）
# ▶ 組（"5data"／"3data"）→（設定 FILES での名前, ダイアログに出す説明）の辞書
DATA_KINDS = {"5data": ("学習", "学習データ（5data）の CSV"), "3data": ("検証", "検証データ（3data）の CSV")}   # 組 → (設定 FILES での名前, ダイアログの説明)
# ▶ Windows で使えるファイルの場所（パス）の長さの上限（260 文字）
PATH_MAX = 260     # Windows のパスの長さの上限（文字）
# ▶ 学習結果フォルダの下に作る、最も長いファイルの場所の長さの目安（90 文字）。選んだフォルダの場所が深すぎないかを調べるのに使う
DEEPEST = 90       # 学習結果フォルダの下の、最も長いファイルの場所の長さ（文字）の目安（せん断速度_粘度_拡張版/ の英語の凡例の図など）


# ▶ 関数：ファイルの場所を、表に書きやすい形にする（リポジトリの中なら、リポジトリ直下からの相対パス）
def show_path(p):
    # ▶ 関数の説明（docstring）
    """表や記録に書く場所：リポジトリの中なら、リポジトリ直下からの相対パス。外なら、そのまま（絶対パス）"""
    # ▶ os.path.abspath：場所を、ドライブ名から始まる絶対パスにする。a は調べる場所、r はリポジトリ直下
    a, r = os.path.abspath(p), os.path.abspath(REPO)
    # ▶ try：次の行でエラーが出たら、except の行へ移る
    try:
        # ▶ os.path.commonpath：2 つの場所の共通の親フォルダ。それがリポジトリ直下なら、リポジトリの中にある（normcase は、大文字・小文字の違いをそろえる）
        inside = os.path.normcase(os.path.commonpath([a, r])) == os.path.normcase(r)
    # ▶ ValueError（値のエラー）が出たら（ドライブが違うと、共通の親フォルダがないので、エラーになる）
    except ValueError:                                       # ドライブが違う
        # ▶ リポジトリの外とする
        inside = False
    # ▶ 中なら、リポジトリ直下からの相対パス（os.path.relpath）を、外なら、絶対パスのままを返す
    return os.path.relpath(a, r) if inside else a


# ▶ 関数：設定や記録に書いた場所を、絶対パスにする
def full_path(p):
    # ▶ 関数の説明（docstring）
    """設定や記録に書いた場所 → 絶対パス（相対パスは、リポジトリ直下から）"""
    # ▶ os.path.isabs：絶対パスか。絶対パスならそのまま、相対パスなら、リポジトリ直下（REPO）につないでから、絶対パスにする
    return os.path.abspath(p if os.path.isabs(p) else os.path.join(REPO, p))


# ▶ 関数：ダイアログで、ファイル（kind = "file"）かフォルダ（kind = "dir"）を選ぶ。title はダイアログの題、initialdir は最初に開くフォルダ
def ask_path(title, kind="file", initialdir=None):
    # ▶ 関数の説明（docstring）
    """ダイアログで、CSV ファイル（kind = "file"）かフォルダ（kind = "dir"）を選ぶ（tkinter）。選んだ場所を返す。
    キャンセルしたら止める（例外）。ダイアログは、ほかのウィンドウの手前に出す（VS Code の後ろに隠れたら、タスクバーで探す）"""
    # ▶ tkinter（Python に付いている、ウィンドウを作る道具）を tk という名前で読み込む
    import tkinter as tk
    # ▶ tkinter から、ファイルやフォルダを選ぶダイアログの部品を読み込む
    from tkinter import filedialog
    # ▶ これからダイアログを開くことを表示する（flush=True：すぐに画面に出す）
    print(f"ダイアログを開きます：{title}", flush=True)
    # ▶ ダイアログの親になるウィンドウを作る
    root = tk.Tk()
    # ▶ そのウィンドウを隠す（ダイアログだけを見せる）
    root.withdraw()                                          # 空の親ウィンドウは見せない
    # ▶ ダイアログを、ほかのウィンドウより手前に出す（"-topmost"：いちばん手前）
    root.attributes("-topmost", True)                       # ダイアログを、ほかのウィンドウの手前に出す
    # ▶ ウィンドウの設定を、すぐに反映させる
    root.update()
    # ▶ 最初に開くフォルダ。指定があって、そのフォルダがあればそれ、なければ None（ダイアログに任せる）
    start = initialdir if initialdir and os.path.isdir(initialdir) else None
    # ▶ try：この中で何があっても、最後に finally の行を実行する
    try:
        # ▶ ファイルを選ぶとき
        if kind == "file":
            # ▶ ファイルを開くダイアログを出し、選んだファイルの場所を p に入れる（filetypes：ダイアログで見せるファイルの種類。最初は CSV ファイル）
            p = filedialog.askopenfilename(parent=root, title=title, initialdir=start, filetypes=[("CSV ファイル", "*.csv"), ("すべてのファイル", "*.*")])
        # ▶ フォルダを選ぶとき
        else:
            # ▶ フォルダを選ぶダイアログを出し、選んだフォルダの場所を p に入れる（mustexist=False：まだないフォルダの名前も受け付ける）
            p = filedialog.askdirectory(parent=root, title=title, initialdir=start, mustexist=False)
    # ▶ finally：ダイアログを閉じた後（エラーのときも）、必ず実行する
    finally:
        # ▶ 親のウィンドウを片付ける
        root.destroy()
    # ▶ 何も選ばれなかったら（キャンセルすると、p が空になる）
    if not p:
        # ▶ エラーで止める（raise）。止まった理由と、どうすればよいかを表示する
        raise RuntimeError(f"選ばれなかったので止めました（{title}）。もう一度、このセルから実行してください")
    # ▶ 選んだ場所を、Windows の書き方（\ 区切り）にそろえて返す
    return os.path.normpath(p)


# ▶ 関数：材料ごとに、学習データ・検証データの CSV と、学習結果フォルダを決める。files は設定 FILES、need_data が False なら学習結果フォルダだけを決める
def choose_inputs(mats, files=None, need_data=True):
    # ▶ 関数の説明（docstring）
    """材料ごとに、学習データ（5data）・検証データ（3data）の CSV と、学習結果フォルダを決めて、SOURCES に記録する（実行の始めに、まとめて聞く）。
    files（ノートの設定 FILES）：{材料: {"学習": CSV, "検証": CSV, "結果": フォルダ}}。書いてあるものはそれを使い、ないものはダイアログで選ぶ（相対パスは、リポジトリ直下から）。
    need_data = False なら、学習結果フォルダだけを決める（データの場所は、学習済みモデルの記録から取る。data_from_model）。
    同じ学習結果フォルダを 2 つの材料で選ぶと、結果が上書きされるので止める。決めたものの表を返す"""
    # ▶ files が None（省略）なら、空の辞書にする（a or b は、a が空なら b）
    files = files or {}
    # ▶ 決めたものの表を作るための、空のリスト
    rows = []
    # ▶ 材料を 1 つずつ mat に入れてくり返す
    for mat in mats:
        # ▶ その材料の設定 FILES の中身（なければ空の辞書）を f に、SOURCES の中のその材料の辞書（なければ作る。setdefault）を src に入れる
        f, src = files.get(mat, {}), SOURCES.setdefault(mat, {})
        # ▶ ダイアログで最初に開くフォルダ（05 フォルダの中の、その材料のフォルダ）
        start = os.path.join(DATA_ROOT, MAT_FOLDERS.get(mat, ""))   # ダイアログの最初のフォルダ（05 フォルダの材料フォルダ）
        # ▶ 学習データ（5data）と検証データ（3data）について、くり返す（grp は組、name は FILES での名前、what はダイアログの説明）
        for grp, (name, what) in DATA_KINDS.items():
            # ▶ FILES に書いてなくて、データも要らないとき（need_data が False）は
            if not (f.get(name) or need_data):
                # ▶ とばす
                continue
            # ▶ FILES に書いてあれば、その場所（絶対パスにする）。なければ、ダイアログで選ぶ（【PP】のように、材料の名前を題に付ける）
            p = full_path(f[name]) if f.get(name) else ask_path(f"【{MAT_NAMES[mat]}】{what} を選んでください", "file", start)
            # ▶ そのファイルがなければ
            if not os.path.isfile(p):
                # ▶ エラーで止める（FileNotFoundError：ファイルが見つからないエラー）
                raise FileNotFoundError(f"{MAT_NAMES[mat]}：{what} が見つかりません（{p}）")
            # ▶ 決めたファイルの場所を、SOURCES に記録する
            src[grp] = p
        # ▶ 学習結果フォルダ：FILES に書いてあれば、その場所。なければ、ダイアログで選ぶ（次の行が、ダイアログの題）
        d = full_path(f["結果"]) if f.get("結果") else ask_path(
            # ▶ ダイアログの題（f"…" の中の {…} に、フォルダの名前が入る）と、フォルダを選ぶ指定（"dir"）と、最初に開くフォルダ
            f"【{MAT_NAMES[mat]}】学習結果フォルダを選んでください（この下に「{VERSION_DIRS['shot']}」「{VERSION_DIRS['cond']}」などを作ります）", "dir", start)
        # ▶ フォルダがなければ作る
        os.makedirs(d, exist_ok=True)
        # ▶ 学習結果フォルダの場所を、SOURCES に記録する
        src["結果"] = d
        # ▶ フォルダの場所の文字数に、その下に作るファイルの場所の目安（DEEPEST）を足して、Windows の上限を超えそうなら
        if len(os.path.abspath(d)) + DEEPEST > PATH_MAX:
            # ▶ 注意を表示する（次の行に続く。2 つの文字列を並べると、つながる）
            print(f"★ 注意：{MAT_NAMES[mat]} の学習結果フォルダの場所が長い（{len(os.path.abspath(d))} 文字）ので、中に作るファイルの場所が、"
                  # ▶ （注意の文の続き）
                  f"Windows の上限 {PATH_MAX} 文字を超えるおそれがあります。もっと浅いフォルダを選ぶと安全です")
        # ▶ 表の 1 行（材料、決めたデータの場所、学習結果フォルダ）を rows に足す（**{…} は、辞書の中身を、ここに広げる書き方）
        rows.append({"材料": MAT_NAMES[mat], **{f"{DATA_KINDS[g][0]}データ": show_path(src[g]) for g in DATA_KINDS if src.get(g)}, "学習結果フォルダ": show_path(d)})
    # ▶ すべての材料の学習結果フォルダの場所を、比べられる形（絶対パスで、大文字・小文字をそろえる）にしたリスト
    roots = [os.path.normcase(os.path.abspath(SOURCES[m]["結果"])) for m in mats]
    # ▶ 2 回以上出てくる場所（2 つの材料で同じフォルダ）を集める（{…} は集合。同じものは 1 つになる）
    dup = sorted({r for r in roots if roots.count(r) > 1})
    # ▶ 同じフォルダがあれば
    if dup:
        # ▶ エラーで止める（結果が上書きされてしまうため）
        raise ValueError("2 つ以上の材料で、同じ学習結果フォルダが選ばれました（結果が上書きされます）。材料ごとに別のフォルダを選んでください：" + "、".join(dup))
    # ▶ 決めたものの表を返す
    return pd.DataFrame(rows)


# ▶ 関数：学習済みモデル（joblib の中身 obj）に記録した、データの場所を使う
def data_from_model(mat, obj):
    # ▶ 関数の説明（docstring）
    """学習済みモデル（joblib の中身 obj）に記録した、学習データ・検証データの場所を SOURCES に入れる（設定 FILES で決めたものは、そのまま使う）。
    記録の場所にファイルがなければ、ダイアログで選ぶ"""
    # ▶ SOURCES の中の、その材料の辞書（なければ作る）
    src = SOURCES.setdefault(mat, {})
    # ▶ 学習データと検証データについて、くり返す
    for grp, (name, what) in DATA_KINDS.items():
        # ▶ すでに決まっていれば（設定 FILES に書いてあったとき）
        if src.get(grp):
            # ▶ とばす
            continue
        # ▶ 学習済みモデルに記録された場所（"学習データのファイル" など）を読む（なければ None）
        rec = obj.get(f"{name}データのファイル")
        # ▶ 記録があれば、絶対パスにする
        p = full_path(rec) if rec else None
        # ▶ 記録がない、またはその場所にファイルがなければ
        if not (p and os.path.isfile(p)):
            # ▶ そのことを表示する
            print(f"【{MAT_NAMES[mat]}】学習済みモデルに記録された{what} が見つかりません（{rec}）")
            # ▶ ダイアログで選ぶ
            p = ask_path(f"【{MAT_NAMES[mat]}】{what} を選んでください", "file", os.path.join(DATA_ROOT, MAT_FOLDERS.get(mat, "")))
        # ▶ 決めたファイルの場所を、SOURCES に記録する
        src[grp] = p


# ▶ def：関数 result_root を作る。材料 mat の学習結果フォルダの場所を返す
def result_root(mat):
    # ▶ 関数の説明（docstring）
    """学習結果フォルダ（choose_inputs で決めたもの）"""
    # ▶ SOURCES に記録した学習結果フォルダ（なければ None）
    d = SOURCES.get(mat, {}).get("結果")
    # ▶ まだ決まっていなければ
    if not d:
        # ▶ エラーで止める
        raise RuntimeError(f"{MAT_NAMES[mat]}：学習結果フォルダが決まっていません（先に choose_inputs で決める）")
    # ▶ 学習結果フォルダの場所を返す
    return d


# ▶ 関数：学習結果フォルダの下の場所を返す。*parts は、いくつでも並べてよいフォルダの名前（例："全ショット版", "全厚み総合", "B_Tset"）。make が True なら、フォルダを作る
def result_dir(mat, *parts, make=True):
    # ▶ 関数の説明（docstring）
    """学習結果フォルダの下の場所。parts の中のモデルの記号（B_Tset など）は、フォルダの名前（MODEL_DIRS）に直す。
    make が True なら、フォルダがなければ作る（読むだけのときは False）。
    例：result_dir(mat, "データ確認")、result_dir(mat, VERSION_DIRS[INPUT_MODE], THICK_TOTAL, "B_Tset")"""
    # ▶ 学習結果フォルダに、parts の名前を順につなぐ。モデルの記号（B_Tset など）は、MODEL_DIRS のフォルダの名前に直す（.get(p, p)：辞書になければ、そのまま）
    d = os.path.join(result_root(mat), *[MODEL_DIRS.get(p, p) for p in parts])
    # ▶ make が True なら（読むだけのときは False にして、フォルダを作らない）
    if make:
        # ▶ フォルダを作る（exist_ok=True：すでにあってもエラーにしない）
        os.makedirs(d, exist_ok=True)
    # ▶ return：関数の結果として、フォルダの場所 d を返す
    return d


# ▶ 関数 path_of：材料と組（"5data"／"3data"）から、データの CSV ファイルの場所を返す
def path_of(mat, grp):
    # ▶ 関数の説明（docstring）
    """読む CSV：choose_inputs で決めたもの。決めていなければ（ノートの外のスクリプト）、今までどおり 05 フォルダの材料フォルダから探す"""
    # ▶ choose_inputs で、その材料・組のファイルを決めてあれば
    if SOURCES.get(mat, {}).get(grp):
        # ▶ その場所を返す
        return SOURCES[mat][grp]
    # ▶ 決めていなければ（ノートの外のスクリプト）、05 フォルダの材料フォルダで、パターンに合うファイルを探す（glob.glob：探して、リストで返す。sorted で名前の順に並べる）
    fs = sorted(glob.glob(os.path.join(DATA_ROOT, MAT_FOLDERS[mat], GROUP_DIRS[grp], "*.csv")))
    # ▶ 見つかった CSV がちょうど 1 つでなければ、エラーで止める（len はリストの要素の数）
    assert len(fs) == 1, f"{MAT_FOLDERS[mat]}/{GROUP_DIRS[grp]} に CSV が 1 つだけあるはず（見つかったもの: {fs}）"
    # ▶ 見つかったファイル（リストの最初の要素。番号は 0 から）を返す
    return fs[0]


# ---------------------------------------------------------------- 問題データ一覧
# ▶ 関数：問題データ一覧の CSV を読む（path を省略すると、PROBLEM_CSV を読む）
def read_problem_list(path=None):
    # ▶ 関数の説明（docstring）
    """問題データ一覧.csv（05 フォルダ）を読む（空欄は空文字）"""
    # ▶ pd.read_csv：CSV を表（DataFrame）として読む。encoding="utf-8-sig"：BOM 付きの UTF-8、dtype=str：全部を文字として読む、fillna("")：空欄を空の文字 "" にする
    # ▶ path or PROBLEM_CSV：path が None（省略）なら PROBLEM_CSV を使う
    return pd.read_csv(path or PROBLEM_CSV, encoding="utf-8-sig", dtype=str).fillna("")


# ▶ 関数：表 df の各行が、問題データ一覧の行（区分が kinds のどれか）に当てはまるかを調べる
def problem_mask(df, mat, grp, plist, kinds):
    # ▶ 関数の説明（docstring）
    """区分が kinds のどれかで、再測定が「済み」でない一覧の行のうち、この材料・組・条件に当てはまる行の真偽値と、その ID を返す。
    「ショット」列は、空欄なら条件全体、数字ならそのショット（元ファイル名の先頭 3 桁）。"2;3" のように複数書ける"""
    # ▶ 行の数だけ False（当てはまらない）を並べた配列を作る（後で、当てはまる行を True にする）
    mask = np.zeros(len(df), bool)
    # ▶ 行ごとに空のリストを作る（当てはまった一覧の ID を入れる）。_ は「使わない変数」に使う慣習の名前
    ids = [[] for _ in range(len(df))]
    # ▶ 問題データ一覧を 1 行ずつ r に入れてくり返す（iterrows は、行番号と行の組を順に返す。行番号は使わないので _ で受ける）
    for _, r in plist.iterrows():
        # ▶ 区分が kinds にない行、再測定が「済み」の行、新しいデータ用でない行（「データ」列が 新 か空欄 でない）は、とばす
        if r["区分"] not in kinds or r["再測定"] == "済み" or r["データ"] not in ("新", ""):
            # ▶ continue：この回の残りをとばして、次の行へ進む
            continue
        # ▶ 材料が違う行（「すべて」か空欄なら、全材料が対象）と、組が違う行も、とばす
        if r["材料"] not in ("すべて", "", MAT_NAMES[mat]) or r["組"] not in ("", grp):
            # ▶ 次の行へ進む
            continue
        # ▶ まず、全部の行を True（当てはまる候補）にする
        m = np.ones(len(df), bool)
        # ▶ 一覧に設定温度が書いてあれば（!= は「等しくない」）
        if r["設定温度"] != "":
            # ▶ 設定温度が一致する行だけを True のまま残す（&= は「かつ」で絞り込む。float は文字を数に変える。to_numpy は列を NumPy の配列にする）
            m &= df["Tset"].to_numpy() == float(r["設定温度"])
        # ▶ 一覧に厚みが書いてあれば
        if r["厚み"] != "":
            # ▶ 厚みが一致する行だけを残す（np.isclose は、小数の細かい誤差を許して比べる）
            m &= np.isclose(df["H"].to_numpy(), float(r["厚み"]))
        # ▶ 一覧に射出速度が書いてあれば
        if r["射出速度"] != "":
            # ▶ 射出速度が一致する行だけを残す
            m &= df["v"].to_numpy() == float(r["射出速度"])
        # ▶ 「ショット」列に番号が書いてあれば（列がなければ "" として扱う）
        if r.get("ショット", "") != "":
            # ▶ 「2;3」のような書き方を、; や 、 で区切って、数のリスト [2, 3] にする（re.split は正規表現で区切る。int は文字を整数に変える）
            shots = [int(s) for s in re.split(r"[;；,、]", r["ショット"]) if s.strip()]
            # ▶ ショット番号がそのリストに入っている行だけを残す（ショットの列がなければ、どの行も当てはまらない）
            m &= np.isin(df["shot"].to_numpy(), shots) if "shot" in df.columns else np.zeros(len(df), bool)
        # ▶ 当てはまった行を、結果の mask に加える（|= は「または」で足し合わせる）
        mask |= m
        # ▶ 当てはまった行の番号を、1 つずつ i に入れてくり返す（np.flatnonzero は True の位置の番号を返す）
        for i in np.flatnonzero(m):
            # ▶ その行の ID のリストに、一覧の ID を足す（append はリストの最後に足す）
            ids[i].append(r["ID"])
    # ▶ 当てはまりの配列と、行ごとの ID（複数なら ; でつないだ文字）を返す
    return mask, [";".join(x) for x in ids]


# ---------------------------------------------------------------- 読み込み
# ▶ 関数 load：1 つの材料・組の CSV を読み、形式を確かめ、除く行を除いて、表を返す
def load(mat, grp, plist=None, exclude=None):
    # ▶ 関数の説明（docstring）
    """結合ファイル（path_of：choose_inputs で選んだ CSV）を読み、形式を確かめ、問題データ一覧の「除外」を除く。
    加熱シリンダ温度の列がないファイル（PP の 3data）は、元ファイル名から設定温度を取る。列があれば、ファイル名と一致するかを確かめる。
    η・γ̇・P3_3 が 0 以下の行は対数がとれないので、一覧に登録がなくても除き、INVALID に記録する（判定で ★）。
    列 sens には「感度確認」の ID が入る（なければ空文字）。圧力は Pa に直してから ln をとる。Tff の ln は K で"""
    # ▶ exclude を省略したら、設定エリアの USE_EXCLUSIONS を使う（a if 条件 else b の書き方）
    exclude = USE_EXCLUSIONS if exclude is None else exclude
    # ▶ CSV ファイルの場所を探す
    path = path_of(mat, grp)
    # ▶ CSV を表として読む
    raw = pd.read_csv(path, encoding="utf-8-sig")
    # ▶ COLS の辞書にない列名を集める（想定外の列があれば気づけるように）
    unknown = [c for c in raw.columns if c not in COLS]
    # ▶ 想定外の列があれば、エラーで止める（assert not リスト：リストが空でなければ止める）
    assert not unknown, f"想定にない列がある: {unknown}（mat_common.py の COLS に足す）"
    # ▶ 列名を、COLS の辞書で短い名前に付け替える
    df = raw.rename(columns=COLS)
    # ▶ 必要な列のうち、ない列を集める
    missing = [c for c in REQUIRED if c not in df.columns]
    # ▶ ない列があれば、エラーで止める
    assert not missing, f"必要な列がない: {missing}（{path}）"
    # ▶ 必要な列に欠損（空欄）が 1 つもないことを確かめる（isna は空欄なら True。sum を 2 回で、表全体の空欄の数）
    assert df[REQUIRED].isna().sum().sum() == 0, f"欠損がある（{path}）"
    # ▶ 元ファイル名から、正規表現で 4 つの部分（ショット・設定温度・射出速度・厚み）を取り出して、表 p にする（列の番号 0〜3）
    p = df["file"].str.extract(RE_SRC)
    # ▶ 全部の行で取り出せた（ファイル名の形が合っている）ことを確かめる
    assert p.notna().all().all(), f"元ファイル名の形式に合わない行がある（{path}）"
    # ▶ ファイル名の射出速度・厚みが、列の値と一致することを確かめる（np.allclose は、全部がほぼ等しいか）
    assert np.allclose(p[2].astype(float), df["v"]) and np.allclose(p[3].astype(float), df["H"]), "元ファイル名と、厚み・射出速度の列が一致しない"
    # ▶ 加熱シリンダ温度の列があるか（True／False）を覚えておく
    has_tset = "Tset" in df.columns
    # ▶ 列があれば
    if has_tset:
        # ▶ ファイル名の設定温度と、列の値が一致することを確かめる
        assert np.allclose(p[1].astype(float), df["Tset"].astype(float)), "元ファイル名と、加熱シリンダ温度の列が一致しない"
    # ▶ else：列がなければ
    else:
        # ▶ ファイル名から取った設定温度を、新しい列 Tset にする（astype(float) は小数の数に変える）
        df["Tset"] = p[1].astype(float)
    # ▶ 設定温度を、小数の数にそろえる
    df["Tset"] = df["Tset"].astype(float)
    # ▶ ファイル名の先頭 3 桁を、ショット番号（整数）の列 shot にする
    df["shot"] = p[0].astype(int)
    # ▶ 問題データ一覧を渡されていなければ、ここで読む
    plist = read_problem_list() if plist is None else plist
    # ▶ 読んだ行の数を覚えておく
    n_raw = len(df)
    # ▶ 「除外」を使う設定なら
    if exclude:
        # ▶ 一覧の「除外」に当てはまる行（drop が True の行）と、その ID を調べる（関数は 2 つの値を返すので、2 つの変数で受ける）
        drop, ids = problem_mask(df, mat, grp, plist, ("除外",))
        # ▶ 除いた行の元ファイル名と ID を表にして、記録する（df.loc[drop, "file"] は、drop が True の行の file 列）
        REMOVED[(mat, grp)] = pd.DataFrame({"元ファイル": df.loc[drop, "file"].to_numpy(), "ID": np.array(ids, dtype=object)[drop]})
        # ▶ 除外の行を除く（~ は True と False を反対にする）。reset_index で、行番号を 0 から振り直す
        df = df[~drop].reset_index(drop=True)
    # ▶ 「除外」を使わない設定なら
    else:
        # ▶ 空の表を記録する
        REMOVED[(mat, grp)] = pd.DataFrame(columns=["元ファイル", "ID"])
    # ▶ η・γ̇・P3_3 のどれかが 0 以下の行を探す（対数がとれないため）。any(axis=1) は「行の中のどれか 1 つでも」
    bad = (df[["eta", "gd", "P33"]] <= 0).any(axis=1).to_numpy()
    # ▶ その行を記録する
    INVALID[(mat, grp)] = df.loc[bad, ["file", "Tset", "H", "v", "shot", "eta", "gd", "P33"]].reset_index(drop=True)
    # ▶ その行を除く
    df = df[~bad].reset_index(drop=True)
    # ▶ 一覧の「感度確認」に当てはまる行の ID を、列 sens に入れる（1 つ目の戻り値は使わないので _ で受ける）
    _, df["sens"] = problem_mask(df, mat, grp, plist, ("感度確認",))
    # ▶ 感度確認の行なら True になる列 is_sens を作る
    df["is_sens"] = df["sens"] != ""
    # ▶ 条件の名前の列 cond を作る（例："190_0.5_20"＝設定温度_厚み_射出速度）。文字どうしは + でつなげる
    df["cond"] = df["Tset"].astype(int).astype(str) + "_" + df["H"].astype(str) + "_" + df["v"].astype(int).astype(str)
    # ▶ η・γ̇・厚み・射出速度の自然対数（ln）の列を、まとめて作る（np.log が ln）
    df["ln_eta"], df["ln_gd"], df["ln_H"], df["ln_v"] = np.log(df["eta"]), np.log(df["gd"]), np.log(df["H"]), np.log(df["v"])
    # ▶ P3_3 を MPa から Pa に直して（× 10⁶）、ln をとる
    df["ln_P33"] = np.log(df["P33"] * 1e6)                 # CSV の圧力は MPa。Pa に直してから ln をとる
    # ▶ 流動中樹脂温度を K に直して（+ 273.15）、ln をとる
    df["ln_Tff"] = np.log(df["Tff"] + 273.15)               # 流動中樹脂温度 [K] の ln（6月のノートと同じ）
    # ▶ γ̇・P3_3・Tff のそれぞれについて、下の行をくり返す
    for c in ["gd", "P33", "Tff"]:                          # 条件ごとの平均（INPUT_MODE = "cond" の入力、図の予測線、外れ値の検知に使う）
        # ▶ 条件（cond）ごとに ln の平均をとり、その条件の各行に書き込む（groupby は「条件ごとに分ける」、transform("mean") は平均を各行に戻す）
        # ▶ 列名の末尾 _cm は condition mean（条件平均）の略
        df[f"ln_{c}_cm"] = df.groupby("cond")[f"ln_{c}"].transform("mean")
    # ▶ Tff の条件平均を、℃ に戻した列（np.exp は ln の逆）
    df["Tff_cm"] = np.exp(df["ln_Tff_cm"]) - 273.15
    # ▶ 組の名前（"5data"／"3data"）の列を作る
    df["grp"] = grp
    # ▶ 読んだファイルの情報を辞書にして、記録する（場所。show_path で、リポジトリの中なら、リポジトリ直下から見た場所にする。以下の数行も、この辞書の続き）
    FILE_INFO[(mat, grp)] = {"材料": MAT_NAMES[mat], "組": grp, "ファイル": show_path(path), "読んだ行数": n_raw,
                             # ▶ （続き）一覧で除いた行数、0 以下で除いた行数、使う行数
                             "一覧で除外": n_raw - len(df) - int(bad.sum()), "0以下で除外（未登録）": int(bad.sum()), "使う行数": len(df),
                             # ▶ （続き）条件の数と、1 条件あたりのショット数の最小〜最大（groupby('cond').size() は条件ごとの行数）
                             "条件数": df["cond"].nunique(), "ショット/条件": f"{df.groupby('cond').size().min()}〜{df.groupby('cond').size().max()}",
                             # ▶ （続き）設定温度の水準（「・」でつなぐ）と、設定温度を列から取ったか、ファイル名から取ったか
                             "設定温度": "・".join(str(int(t)) for t in sorted(df["Tset"].unique())), "設定温度の取り方": "列" if has_tset else "元ファイル名",
                             # ▶ （続き）厚みと射出速度の水準（{h:g} は、余計な 0 を付けない数の書き方）
                             "厚み": "・".join(f"{h:g}" for h in sorted(df["H"].unique())), "射出速度": "・".join(str(int(v)) for v in sorted(df["v"].unique())),
                             # ▶ （続き）感度確認の行の数
                             "感度確認の行": int(df["is_sens"].sum())}
    # ▶ 読み込んで整えた表を返す
    return df


# ▶ 関数：材料のリストを受け取り、全部の材料の 5data（学習）と 3data（検証）を読む
def load_all(mats, exclude=None):
    # ▶ 関数の説明（docstring）
    """材料ごとに 5data（学習）と 3data（検証）を読む。学習と検証で条件が重ならないことも確かめる"""
    # ▶ 問題データ一覧を、1 回だけ読む
    plist = read_problem_list()
    # ▶ 辞書内包表記：材料と組の組み合わせごとに load で読み、(材料, 組) をキーにした辞書にまとめる
    out = {(mat, grp): load(mat, grp, plist, exclude) for mat in mats for grp in ["5data", "3data"]}
    # ▶ 材料ごとにくり返す
    for mat in mats:
        # ▶ 学習と検証の条件名の集合（set）の、共通部分（& は両方にあるもの）
        both = set(out[(mat, "5data")]["cond"]) & set(out[(mat, "3data")]["cond"])
        # ▶ 学習と検証で同じ条件があれば、エラーで止める
        assert not both, f"{MAT_NAMES[mat]}：学習と検証で条件が重なっている: {sorted(both)}"
    # ▶ 読んだ表の辞書を返す
    return out


# ▶ 関数：検証データ te の各行に、学習データ tr の射出速度の範囲と比べた区分（低速側・高速側・範囲内）を付ける
def add_groups(tr, te):
    # ▶ 関数の説明（docstring）
    """検証の行を、学習データの射出速度の範囲より遅い側（低速側）・速い側（高速側）・範囲内に分ける（列 group）"""
    # ▶ 元の表を変えないように、写しを作る
    te = te.copy()
    # ▶ 学習データの射出速度の最小と最大（2 つの変数に同時に入れる書き方）
    vmin, vmax = tr["v"].min(), tr["v"].max()
    # ▶ np.where(条件, A, B)：条件が成り立つ行は A、そうでない行は B。最小より遅ければ「低速側」、最大より速ければ「高速側」、それ以外は「範囲内」
    te["group"] = np.where(te["v"] < vmin, "低速側", np.where(te["v"] > vmax, "高速側", "範囲内"))
    # ▶ 区分を付けた表を返す
    return te


# ---------------------------------------------------------------- 温度のシフトファクター
# ▶ シフトファクターの設定の既定値（設定エリアに SHIFT_CFG がないときに使う。中身は設定エリアと同じ）
DEFAULT_SHIFT_CFG = {     # h_grid・v_grid：尤度を調べる粗い格子（下限, 上限, 刻み）。h が横、v が縦のシフトの係数
    # ▶ PP：アレニウス型、基準温度 210 ℃、横と縦の係数を調べる範囲（下限, 上限, 刻み）
    "PP": {"type": "arrhenius", "Tref": 210.0, "h_grid": (-20.0, 40.0, 5.0), "v_grid": (-20.0, 60.0, 5.0)},
    # ▶ 粉砕材混合 PP：PP と同じ
    "MIX": {"type": "arrhenius", "Tref": 210.0, "h_grid": (-20.0, 40.0, 5.0), "v_grid": (-20.0, 60.0, 5.0)},
    # ▶ GPPS：WLF 型、基準温度 220 ℃、Tg 100 ℃ と A2 = 51.6 は固定
    "GPPS": {"type": "wlf", "Tref": 220.0, "Tg": 100.0, "A2": 51.6, "h_grid": (-20.0, 40.0, 5.0), "v_grid": (-20.0, 40.0, 5.0)},
}
# ▶ 設定エリアの SHIFT_CFG があればそれを、なければ上の既定値を使う
SHIFT_CFG = globals().get("SHIFT_CFG", DEFAULT_SHIFT_CFG)


# ▶ ★温度のシフトファクターの式。温度 T_c [℃] と係数 theta から、ln a(T)（シフトファクターの自然対数）を計算して返す（cfg は材料の設定）
def shift_ln(T_c, theta, cfg):
    # ▶ 関数の説明（docstring）
    """ln a(T)（T は ℃。Tref で 0）。
    アレニウス型：ln a = (E/R)(1/T − 1/Tref)、theta = E [kJ/mol]。
    WLF 型：ln a = −A1·[(T − D2)/(A2 + T − D2) − (Tref − D2)/(A2 + Tref − D2)]、theta = A1 [−]、D2 = Tg、A2 は固定"""
    # ▶ 温度を数の配列にして、K（絶対温度）に直す
    T = np.asarray(T_c, float) + 273.15
    # ▶ 基準温度 Tref を K に直す
    Tr = float(cfg["Tref"]) + 273.15
    # ▶ アレニウス型なら（== は「等しいか」の比較）
    if cfg["type"] == "arrhenius":
        # ▶ ln a = (E/R)(1/T − 1/Tref) を計算して返す。theta（E）は kJ/mol なので、1000 倍して J/mol にする。T = Tref で 0（a = 1）になる
        return float(theta) * 1000.0 / R_GAS * (1.0 / T - 1.0 / Tr)
    # ▶ WLF 型の場合：D2（Tg を K に直した値）と A2 を取り出す
    D2, A2 = float(cfg["Tg"]) + 273.15, float(cfg["A2"])
    # ▶ ln a = −A1·[(T − D2)/(A2 + T − D2) − (Tref − D2)/(A2 + Tref − D2)] を計算して返す（theta が A1）。基準温度の値を引いているので、T = Tref で 0 になる
    return -float(theta) * ((T - D2) / (A2 + T - D2) - (Tr - D2) / (A2 + Tr - D2))


# ▶ 関数：係数の名前と単位を返す
def theta_names(cfg):
    # ▶ 関数の説明（docstring）
    """(横の名前, 縦の名前, 単位)"""
    # ▶ アレニウス型なら ("E_h", "E_v", "kJ/mol")、WLF 型なら ("A1_h", "A1_v", "−") を返す（h は横、v は縦）
    return ("E_h", "E_v", "kJ/mol") if cfg["type"] == "arrhenius" else ("A1_h", "A1_v", "−")


# ---------------------------------------------------------------- モデル
# ▶ モデル名 → 日本語の説明の辞書（図のタイトルなどに使う）
MODEL_JA = {"B_Tset": "B_Tset（設定温度 Tset で横・縦にシフト）", "B_Tff": "B_Tff（流動中樹脂温度 Tff で横・縦にシフト）",
            # ▶ （辞書の続き）構造 A（縦だけずらす）のモデル
            "A_Tset": "A_Tset（Tset で縦だけシフト）", "A_Tff": "A_Tff（Tff で縦だけシフト）",
            # ▶ （辞書の続き）基準のモデル C
            "C": "C（シフト構造なし。4 変数を並べた通常の GPR）"}
# ▶ モデル名 → 結果のフォルダの名前（記号_学習方法の短い説明）の辞書。型内樹脂温度は、流動中樹脂温度 Tff のこと
MODEL_DIRS = {"B_Tset": "B_Tset_温度シフトに加熱シリンダ温度採用", "B_Tff": "B_Tff_温度シフトに型内樹脂温度採用",
              # ▶ （辞書の続き）縦だけずらす構造 A のモデル
              "A_Tset": "A_Tset_縦だけの温度シフトに加熱シリンダ温度採用", "A_Tff": "A_Tff_縦だけの温度シフトに型内樹脂温度採用",
              # ▶ （辞書の続き）シフトのないモデル C
              "C": "C_温度シフトなし"}     # モデルの結果のフォルダの名前（記号_学習方法の短い説明。型内樹脂温度＝流動中樹脂温度 Tff。2026-10-27 のユーザー依頼）
# ▶ 列名 → 図に書く記号の辞書
COL_JA = {"ln_gd": "ln γ̇", "ln_P33": "ln P3_3", "ln_Tff": "ln Tff", "Tset": "Tset"}


# ▶ ★関数：設定 SHIFT_COL（シフトファクターに使うデータ列）から、モデルの名前（B_Tset・B_Tff・C など）を決める。struct はシフトの形（"B"／"A"）
def shift_key(col, struct="B"):
    # ▶ 関数の説明（docstring）
    """シフトファクターに使うデータ列 → モデルの記号。col：学習ファイルの列名（"加熱シリンダ温度[℃]"・"加熱シリンダ温度(℃)"・"流動中樹脂温度(℃)"）か、
    短い名前（"Tset"・"Tff"）。None（または空の文字）なら、シフトなしの "C"。struct："B"（横・縦を別々にずらす）か "A"（縦だけ）。温度でない列はエラー"""
    # ▶ 列が None（値なし）か、空の文字なら（str(…) は文字にする、.strip() は前後の空白を除く）
    if col is None or str(col).strip() == "":
        # ▶ シフトのないモデル "C" を返す
        return "C"
    # ▶ 学習ファイルの列名を、短い名前（"Tset"・"Tff" など）に直す（COLS の辞書で引く。辞書にない名前は、そのまま）
    t = COLS.get(str(col).strip(), str(col).strip())
    # ▶ 温度の列（"Tset"・"Tff"）でなければ
    if t not in ("Tset", "Tff"):
        # ▶ エラーで止める（シフトファクターの式は温度の式なので、温度の列しか使えない）
        raise ValueError(f"シフトファクターに使える列は、温度の列（加熱シリンダ温度・流動中樹脂温度）だけです（指定された列：{col}）")
    # ▶ シフトの形が "A" か "B" でなければ
    if struct not in ("A", "B"):
        # ▶ エラーで止める（\" は、文字列の中の " の書き方）
        raise ValueError(f"シフトの形は \"B\"（横・縦を別々）か \"A\"（縦だけ）です（指定：{struct}）")
    # ▶ 形と温度をつないだ名前を返す（例："B" と "Tset" → "B_Tset"）
    return f"{struct}_{t}"


# ▶ 関数：比べるモデルの名前の一覧を作る。cols は列のリスト、structs は形のリスト（("B",) は、要素が 1 つのタプル）
def model_keys(cols, structs=("B",)):
    # ▶ 関数の説明（docstring）
    """比べるモデルの記号の一覧（シフトの形 structs と、列 cols の組み合わせ。同じものは 1 つにする）"""
    # ▶ 空のリストを作る
    out = []
    # ▶ 形を 1 つずつ s に入れてくり返す
    for s in structs:
        # ▶ 列を 1 つずつ c に入れてくり返す
        for c in cols:
            # ▶ その形と列のモデルの名前
            k = shift_key(c, s)
            # ▶ まだリストになければ（同じ名前を 2 回入れない）
            if k not in out:
                # ▶ リストに足す
                out.append(k)
    # ▶ モデルの名前の一覧を返す
    return out


# ▶ ★関数：モデル名から、GP に入れる列と、シフトに使う温度の列を決める
def layout(key):
    # ▶ 関数の説明（docstring）
    """GP の入力の列（シフト前の名前）と、シフトに使う温度の列（C は None）。
    B_Tset・A_Tset：(ln γ̇ + ln a_h(Tset), ln P3_3, ln Tff)、B_Tff・A_Tff：(ln γ̇ + ln a_h(Tff), ln P3_3, Tset)、C：(ln γ̇, ln P3_3, ln Tff, Tset)"""
    # ▶ モデル C なら
    if key == "C":
        # ▶ 4 つの列（ln γ̇、ln P3_3、ln Tff、Tset）をそのまま入れ、シフトはしない（None は「なし」）
        return ["ln_gd", "ln_P33", "ln_Tff", "Tset"], None
    # ▶ モデル名を _ で分けて、2 番目（"Tset" か "Tff"）を取り出す（例："B_Tff".split("_") → ["B", "Tff"]）
    temp = key.split("_")[1]
    # ▶ Tset でずらすモデルは (ln γ̇, ln P3_3, ln Tff) を入れて Tset でずらす。Tff でずらすモデル（B_Tff）は (ln γ̇, ln P3_3, Tset) を入れて Tff でずらす
    return (["ln_gd", "ln_P33", "ln_Tff"], "Tset") if temp == "Tset" else (["ln_gd", "ln_P33", "Tset"], "Tff")


# ▶ 関数：図のタイトルに使うモデルの名前を作る
def model_title(key, mode=None):
    # ▶ 関数の説明（docstring）
    """図のタイトルに使うモデルの名前（版〔入力の取り方〕つき）"""
    # ▶ モデルの説明と、版の名前（入力の取り方：全ショット版・条件ごとの平均版）を「・」でつないで返す
    return f"{MODEL_JA[key]}・{VERSION_DIRS[INPUT_MODE if mode is None else mode]}"


# ▶ 関数：GP の入力の、図に書く名前のリストを作る
def input_labels(key):
    # ▶ 入力の列と、シフトの温度の列を受け取る
    cols, tcol = layout(key)
    # ▶ 各列の記号を並べたリストを作る
    lab = [COL_JA[c] for c in cols]
    # ▶ シフトがあり（is not None は「None でない」）、構造 B（名前が B で始まる）なら
    if tcol is not None and key.startswith("B"):
        # ▶ 1 つ目の入力の名前を「ln γ̇ + ln a_h(温度)」にする（横にずらした後のせん断速度）
        lab[0] = f"ln γ̇ + ln a_h({tcol})"
    # ▶ 名前のリストを返す
    return lab


# ▶ ★関数：GPR のカーネル（データ点どうしが「どれだけ似た値になるか」の決め方）を作る。d は入力の数
def kernel_for(d, linear=None):
    # ▶ 関数の説明（docstring）
    """6月のノートのカーネル（ARD Matern 5/2 ＋ ホワイトノイズ）。linear=True なら線形項（DotProduct）を足す（射出速度の両端の外挿のため）"""
    # ▶ linear を省略したら、設定エリアの USE_LINEAR を使う
    linear = USE_LINEAR if linear is None else linear
    # ▶ 振幅（定数カーネル C。0.001〜1000 で探す）× Matern 5/2 カーネル。滑らかな曲線を表す部分
    # ▶ ARD：入力ごとに長さスケール（どれだけ離れると値が変わるか）を持つ。初期値 1、0.01〜100 で探す
    k = C(1.0, (1e-3, 1e3)) * Matern(length_scale=[1.0] * d, length_scale_bounds=(1e-2, 1e2), nu=2.5)
    # ▶ 線形項を足す設定なら
    if linear:
        # ▶ 線形項（振幅 × DotProduct）を足す。直線的な傾向を表し、学習の範囲の外でも傾向が延びる
        k = k + C(1.0, (1e-3, 1e3)) * DotProduct(sigma_0=1.0, sigma_0_bounds=(1e-5, 1e3))
    # ▶ ホワイトノイズ（ショットごとのばらつき。NOISE_LB〜10 で探す）を足して、できたカーネルを返す
    return k + WhiteKernel(noise_level=0.1, noise_level_bounds=(NOISE_LB, 1e1))


# ▶ 関数：入力の取り方に応じて、使う列の名前を返す
def mode_col(df, c, mode):
    # ▶ 関数の説明（docstring）
    """入力の列名。mode = "cond" で、条件平均の列（<名前>_cm）があればそれを使う（予測用に組み立てた表には _cm がないので、そのまま）"""
    # ▶ 条件平均で入れる設定で、条件平均の列（名前_cm）があればその名前を、なければ元の名前を返す
    return f"{c}_cm" if mode == "cond" and f"{c}_cm" in df.columns else c


# ▶ ★関数：データの表から、GP に入れる入力の行列と、出力をずらす量を作る（シフトファクターを当てはめる所）。th が横、tv が縦の係数
def design(df, key, cfg, th=0.0, tv=0.0, mode=None):
    # ▶ 関数の説明（docstring）
    """データ → (標準化の定数を決めるための入力 X0〔シフト前〕、実際の入力 Xs〔シフト後〕、出力のシフト量 ysh = ln a_v(T))。
    mode（既定 INPUT_MODE）が "cond" なら、γ̇・P3_3・Tff は条件ごとの平均を使う"""
    # ▶ 入力の取り方（省略したら、設定エリアの INPUT_MODE）
    mode = INPUT_MODE if mode is None else mode
    # ▶ 入れる列と、シフトの温度の列を決める
    cols, tcol = layout(key)
    # ▶ 入力の列を横に並べて、行列（行＝データ、列＝入力）を作る（np.column_stack）。シフトする前の値
    X0 = np.column_stack([df[mode_col(df, c, mode)].to_numpy(float) for c in cols])
    # ▶ シフト後の入力 Xs は、まず X0 の写し。出力のずらし量 ysh は、まず全部 0（np.zeros）
    Xs, ysh = X0.copy(), np.zeros(len(df))
    # ▶ シフトのあるモデルなら
    if tcol is not None:
        # ▶ シフトに使う温度（Tset か Tff。条件平均の設定なら、その条件平均）を取り出す
        T = df[mode_col(df, tcol, mode)].to_numpy(float)
        # ▶ 横のシフト：1 列目（ln γ̇）に ln a_h(T) を足す（[:, 0] は「全部の行の、0 番目の列」）
        Xs[:, 0] = X0[:, 0] + shift_ln(T, th, cfg)
        # ▶ 縦のシフト：出力（ln η）をずらす量 ln a_v(T) を計算する
        ysh = shift_ln(T, tv, cfg)
    # ▶ シフト前の入力、シフト後の入力、出力のずらし量の 3 つを返す
    return X0, Xs, ysh


# ▶ class：データと関数をまとめた「型」を作る。ここでは、最適化の途中の損失を記録する道具
class LossRecorder:
    # ▶ 説明（docstring）
    """6月のノートのカスタムオプティマイザと同じ（L-BFGS-B）。各ステップの損失（負の対数周辺尤度）を、最適化 1 回ごとに runs に記録する"""

    # ▶ __init__：この型の物を作ったときに、最初に実行される関数（self は、その物自身）
    def __init__(self):
        # ▶ 最適化 1 回ごとの記録を入れる、空のリスト
        self.runs = []

    # ▶ __call__：この物を関数のように呼んだときの処理。GPR の学習が、最適化の道具としてこれを呼ぶ
    def __call__(self, obj_func, initial_theta, bounds):
        # ▶ SciPy の最適化の関数 minimize を読み込む
        from scipy.optimize import minimize
        # ▶ この 1 回の最適化の、損失の記録
        hist = []
        # ▶ L-BFGS-B 法で、損失（負の対数周辺尤度）が最小になるハイパーパラメータを、範囲 bounds の中で探す（jac=True：傾きも使って探す）
        res = minimize(obj_func, initial_theta, bounds=bounds, method="L-BFGS-B", jac=True,
                       # ▶ （続き）1 歩進むごとに、その点の損失を計算して hist に足す（lambda は名前のない小さな関数）
                       callback=lambda xk: hist.append(float(obj_func(xk, eval_gradient=False))))
        # ▶ この回の記録を runs に足す
        self.runs.append(hist)
        # ▶ 見つけたハイパーパラメータと、そのときの損失を返す
        return res.x, res.fun


# ▶ ★関数 fit_gp：GPR を学習する。key はモデル名、tr は学習データ、th・tv はシフトの係数（横・縦。ここでは決まった値として与える）
# ▶ restarts は最適化のやり直しの回数、theta0 はハイパーパラメータの初期値、record=True なら損失の推移も記録する
def fit_gp(key, tr, cfg, th=0.0, tv=0.0, restarts=None, theta0=None, record=False, mode=None):
    # ▶ 関数の説明（docstring）
    """学習。標準化・中心化は、シフトの量によらない定数（シフト前の入力の平均・標準偏差、ln η の平均。学習データだけで計算）。
    出力の自動標準化（normalize_y）は使わない（シフトの量ごとに尤度が比べられなくなるため）。theta0：カーネルの初期値（最初の最適化はここから、
    残りの restarts 回は、探索範囲の中の乱数の初期値から。対数周辺尤度が最大の解を採る）。record=True なら、損失の推移を戻り値の loss_runs に入れる。
    mode：入力の取り方（既定 INPUT_MODE。"shot" か "cond"）。予測にも同じ取り方を使う"""
    # ▶ 入力の取り方（省略したら、設定エリアの INPUT_MODE）
    mode = INPUT_MODE if mode is None else mode
    # ▶ シフト前の入力 X0、シフト後の入力 Xs、出力のずらし量 ysh を作る
    X0, Xs, ysh = design(tr, key, cfg, th, tv, mode)
    # ▶ 入力の列ごとの平均と標準偏差（axis=0 は「列ごと」）。標準化に使う。シフト前の値で計算するので、係数によらない一定の値になる
    mu_x, sd_x = X0.mean(axis=0), X0.std(axis=0)
    # ▶ 目的変数（ln η）を配列にする
    y = tr["ln_eta"].to_numpy()
    # ▶ ln η の平均（中心化に使う一定の値）
    y0 = float(y.mean())
    # ▶ 入力の数（shape[1]＝行列の列の数）に合わせたカーネルを作る
    k = kernel_for(X0.shape[1])
    # ▶ 初期値が渡されていれば
    if theta0 is not None:
        # ▶ カーネルのハイパーパラメータを、その初期値にした写しを作る
        k = k.clone_with_theta(theta0)
    # ▶ 損失を記録する設定なら、記録の道具（LossRecorder）を作る。そうでなければ None
    rec = LossRecorder() if record else None
    # ▶ GPR の本体を作る。カーネル、やり直しの回数（省略なら N_RESTARTS）、乱数のたねを指定する
    gp = GaussianProcessRegressor(kernel=k, n_restarts_optimizer=N_RESTARTS if restarts is None else restarts, random_state=SEED,
                                  # ▶ （続き）最適化の方法：記録する設定なら記録の道具、そうでなければ標準の L-BFGS-B（"fmin_l_bfgs_b"）
                                  optimizer=rec if record else "fmin_l_bfgs_b")
    # ▶ ★学習する。入力はシフト後の値を標準化したもの、出力は ln η から縦のずらし量 ln a_v と平均を引いたもの
    # ▶ ここで、カーネルのハイパーパラメータ（振幅・長さスケールなど）が、対数周辺尤度が最大になるように決まる
    gp.fit((Xs - mu_x) / sd_x, y - ysh - y0)
    # ▶ 最適化の方法を標準に戻す（記録の道具のままだと、ファイルに保存できないため）
    gp.optimizer = "fmin_l_bfgs_b"                          # 保存（joblib）できるように戻す
    # ▶ 学習結果（GPR、標準化の定数、モデル名、設定、係数など）を辞書にまとめる。予測のときに、学習と同じ変換をするため
    out = {"gp": gp, "mu_x": mu_x, "sd_x": sd_x, "y0": y0, "key": key, "cfg": dict(cfg), "th": float(th), "tv": float(tv), "cols": layout(key)[0], "mode": mode}
    # ▶ 損失を記録する設定なら
    if record:
        # ▶ 損失の記録を、リストにして入れる
        out["loss_runs"] = [list(h) for h in rec.runs]
    # ▶ 学習結果の辞書を返す
    return out


# ▶ 関数：fit_gp を並列計算で動かすための入口（並列の各計算は別々に動くので、設定を引数で渡す）
def fit_job(key, tr, cfg, th, tv, restarts, theta0, mode, record=False):
    # ▶ 関数の説明（docstring）
    """fit_gp を並列に動かすための仕事（joblib から呼ぶ。設定は引数で渡す）"""
    # ▶ 警告を表示しない
    warnings.simplefilter("ignore")
    # ▶ fit_gp を呼んで、その結果を返す
    return fit_gp(key, tr, cfg, th, tv, restarts, theta0, record, mode)


# ▶ ★関数：学習したモデル fit で、表 df の各行の粘度を予測する
def predict(fit, df):
    # ▶ 関数の説明（docstring）
    """予測平均（ln η）、予測標準偏差（ノイズ込み）、潜在関数の標準偏差、ノイズの標準偏差"""
    # ▶ 学習のときと同じ係数で入力をずらし、縦のずらし量を計算する（入力の取り方が記録されていなければ "shot"）。1 つ目の戻り値は使わないので _ で受ける
    _, Xs, ysh = design(df, fit["key"], fit["cfg"], fit["th"], fit["tv"], fit.get("mode", "shot"))
    # ▶ 入力を学習のときと同じ定数で標準化して、GP の予測の平均 mu と標準偏差 sd を求める（return_std=True で標準偏差も返す）
    mu, sd = fit["gp"].predict((Xs - fit["mu_x"]) / fit["sd_x"], return_std=True)
    # ▶ 学習したノイズ分散（kernel_ は学習後のカーネル。k2 は足し算の右側＝ホワイトノイズ）
    noise = fit["gp"].kernel_.k2.noise_level
    # ▶ 4 つを返す：予測平均（平均と縦のずらし量を足し戻した ln η）、予測標準偏差（ノイズ込み）、曲線そのものの標準偏差（ノイズの分を引く）、ノイズの標準偏差
    # ▶ np.maximum(…, 0.0) は、計算の誤差で負にならないようにする（** 2 は 2 乗）
    return mu + fit["y0"] + ysh, sd, np.sqrt(np.maximum(sd ** 2 - noise, 0.0)), float(np.sqrt(noise))


# ▶ 関数：学習したカーネルを、Matern の部分・線形の部分・ノイズの部分に分ける
def kernel_parts(gp):
    # ▶ 関数の説明（docstring）
    """学習したカーネルを、(Matern の部分〔振幅² × Matern〕, 線形の部分〔なければ None〕, ノイズ) に分ける"""
    # ▶ 学習後のカーネル
    k = gp.kernel_
    # ▶ 左側（k1）が足し算の形（Matern ＋ 線形）なら（isinstance は「その種類か」を調べる）
    if isinstance(k.k1, Sum):
        # ▶ Matern の部分、線形の部分、ノイズを返す
        return k.k1.k1, k.k1.k2, k.k2
    # ▶ 線形項がなければ、Matern の部分、None、ノイズを返す
    return k.k1, None, k.k2


# ▶ 関数：学習したハイパーパラメータを、表の 1 行（辞書）にまとめる
def kernel_row(fit):
    # ▶ 関数の説明（docstring）
    """ハイパーパラメータ（振幅、長さスケール、線形項、ノイズ、対数周辺尤度、下限・上限への張り付き）"""
    # ▶ 学習した GPR を取り出す
    gp = fit["gp"]
    # ▶ カーネルを 3 つの部分に分ける
    prod, lin, white = kernel_parts(gp)
    # ▶ Matern の長さスケール（入力ごと）を配列にする（np.atleast_1d は、数が 1 つでも配列にそろえる）
    ls = np.atleast_1d(prod.k2.length_scale)
    # ▶ 係数の名前（例：E_h・E_v）
    th_n, tv_n, _ = theta_names(fit["cfg"])
    # ▶ モデル名と係数（横の係数は構造 B だけ、縦の係数は C 以外。ない場合は np.nan＝「数がない」印）
    row = {"model": fit["key"], th_n: fit["th"] if fit["key"].startswith("B") else np.nan, tv_n: fit["tv"] if fit["key"] != "C" else np.nan,
           # ▶ （続き）対数周辺尤度（LML）と、振幅（定数カーネルの値の平方根）
           "LML": float(gp.log_marginal_likelihood_value_), "amplitude": float(np.sqrt(prod.k1.constant_value)),
           # ▶ （続き）ノイズの分散と標準偏差
           "noise_var": float(white.noise_level), "noise_sd": float(np.sqrt(white.noise_level)),
           # ▶ （続き）ノイズが下限に張り付いたか、長さスケールが範囲の端（0.01 か 100）に張り付いたか（bool は True／False にする）
           "noise_at_lower_bound": bool(white.noise_level < NOISE_LB * 1.05), "ls_at_bound": bool(np.any(ls > 99) or np.any(ls < 0.0105))}
    # ▶ 線形項があれば
    if lin is not None:
        # ▶ 線形項の振幅と σ0 を足す（update は、辞書に項目を足す）
        row.update({"lin_amplitude": float(np.sqrt(lin.k1.constant_value)), "lin_sigma0": float(lin.k2.sigma_0)})
    # ▶ 入力ごとの長さスケールを、「ls_入力名」の項目として足す（zip は 2 つのリストを組にして並べる）
    row.update({f"ls_{c}": float(l) for c, l in zip(input_labels(fit["key"]), ls)})
    # ▶ 1 行分の辞書を返す
    return row


# ---------------------------------------------------------------- プロファイル尤度（本体の手順5と同じ考え方）
# ▶ 関数：下限・上限・刻みから、格子の点の並びを作る
def grid_of(lo, hi, step):
    # ▶ np.arange で lo から hi まで step 刻みの数を作る（上限も含めるため、hi に刻みの半分を足す）。np.round で小数の誤差を 6 桁で丸める
    return np.round(np.arange(lo, hi + step / 2.0, step), 6)


# ▶ 関数：格子の点（係数の組）を順に学習して、それぞれの LML を返す（プロファイル尤度の 1 列分）
def prof_chain(key, tr, cfg, points, restarts, mode):
    # ▶ 関数の説明（docstring）
    """格子の点 points = [(θ_h, θ_v), …] を順に学習して LML を返す。前の点の解を初期値にする（局所解でがたつくのを減らす）。
    theta はその点で学習したカーネルの値（最終モデルの学習の初期値に使う）"""
    # ▶ 警告を表示しない
    warnings.simplefilter("ignore")
    # ▶ 結果のリストと、前の点の解（最初はなし）
    out, theta = [], None
    # ▶ 係数の組 (横, 縦) を、1 つずつ取り出してくり返す
    for th, tv in points:
        # ▶ その係数に固定して、GPR を学習する（前の点の解を初期値にする）
        f = fit_gp(key, tr, cfg, th, tv, restarts, theta, mode=mode)
        # ▶ 学習したハイパーパラメータを、次の点の初期値として覚える
        theta = f["gp"].kernel_.theta
        # ▶ 係数と、そのときの LML などを記録する
        out.append({"th": float(th), "tv": float(tv), "LML": float(f["gp"].log_marginal_likelihood_value_),
                    # ▶ （続き）ノイズの標準偏差と、ハイパーパラメータ（tolist はリストに変える）
                    "noise_sd": float(np.sqrt(f["gp"].kernel_.k2.noise_level)), "theta": theta.tolist()})
    # ▶ 記録のリストを返す
    return out


# ▶ ★関数：プロファイル尤度で、シフトの係数を探す。粗い格子で全体を調べ、良い点の周りを細かく調べ直す
def run_profile(key, tr, cfg, n_jobs=None, restarts=None, mode=None):
    # ▶ 関数の説明（docstring）
    """粗い格子で全体を見て、最良の点の近くを細かい格子で計算し直す。構造 A は θ_h = 0 の 1 次元、B は (θ_h, θ_v) の 2 次元。
    細かい格子は、最良の点を中心に取る。計算した結果、最良の点が動いたら（細かい格子の端に当たったとき）、新しい最良の点を中心に取り直す
    （最良の点が動かなくなるか、MAX_REFINE 回まで。斜めの尾根の上では、尤度がほとんど変わらないまま最良の点が動くことがある）。
    E_h ごとの列（chain）を並列に計算する。mode（既定 INPUT_MODE）は、並列の計算にも引数で渡す。
    返すもの：(表, 収束したか)。表の列：th, tv, LML, noise_sd, theta（カーネルの値。CSV には入れない）, stage（粗い／細かい）。
    収束したか：最良の点の周り（±細かい格子の幅）の点が、すべて計算済みか"""
    # ▶ 並列計算の道具（joblib）を読み込む
    from joblib import Parallel, delayed
    # ▶ 並列の数（省略なら N_JOBS）
    n_jobs = N_JOBS if n_jobs is None else n_jobs
    # ▶ やり直しの回数（省略なら PROFILE_RESTARTS）
    restarts = PROFILE_RESTARTS if restarts is None else restarts
    # ▶ 入力の取り方（省略なら INPUT_MODE）
    mode = INPUT_MODE if mode is None else mode
    # ▶ モデル名の前半（"B" か "A"）
    struct = key.split("_")[0]
    # ▶ 横・縦の係数の粗い格子を作る（*cfg["h_grid"] は、タプルの中身を別々の引数として渡す書き方）
    hg, vg = grid_of(*cfg["h_grid"]), grid_of(*cfg["v_grid"])
    # ▶ 細かい格子の範囲と刻み（材料の設定に "fine" があればそれ、なければ構造に応じて FINE_A か FINE_B）
    rng, step = cfg.get("fine") or (FINE_A if struct == "A" else FINE_B)

    # ▶ 関数の中の関数：点のリストを、並列に計算して表にする（stage は「粗い」か「細かい」）
    def run(points, stage):
        # ▶ 横の係数ごとに、縦の係数を集める辞書
        by = {}
        # ▶ 点を 1 つずつ取り出して
        for th, tv in points:
            # ▶ 横の係数 th のリストがなければ空のリストを作り、そこに縦の係数を足す（setdefault）
            by.setdefault(float(th), []).append(float(tv))
        # ▶ 横の係数ごとに、縦の係数の小さい順に並べた点の列（chain）を作る
        chs = [[(th, tv) for tv in sorted(tvs)] for th, tvs in sorted(by.items())]
        # ▶ 各列を、並列に prof_chain で計算する（delayed は「後で並列に実行する仕事」にする書き方。min は小さい方）
        res = Parallel(n_jobs=min(n_jobs, len(chs)))(delayed(prof_chain)(key, tr, cfg, ch, restarts, mode) for ch in chs)
        # ▶ 各列の結果をつなげて、1 つの表にする
        t = pd.DataFrame([r for rr in res for r in rr])
        # ▶ どの段階（粗い／細かい）の計算かを、列に書く
        t["stage"] = stage
        # ▶ 表を返す
        return t

    # ▶ 粗い格子の全部の点を計算する（構造 A は横を 0 に固定、B は横と縦のすべての組み合わせ）
    pts = run([(0.0, v) for v in vg] if struct == "A" else [(h, v) for h in hg for v in vg], "粗い")
    # ▶ 「取り直しが収束したか」の印。まず False
    converged = False
    # ▶ 最大 MAX_REFINE 回くり返す（range(n) は 0, 1, …, n−1。回数だけを使うので _ で受ける）
    for _ in range(MAX_REFINE):
        # ▶ これまでで LML が最大の点（idxmax は、最大の行の番号。loc でその行を取り出す）
        b = pts.loc[pts["LML"].idxmax()]
        # ▶ その点の周りの、横の係数の細かい格子（構造 A なら 0 だけ）
        hs = [0.0] if struct == "A" else grid_of(b["th"] - rng, b["th"] + rng, step)
        # ▶ その点の周りの、縦の係数の細かい格子
        vs = grid_of(b["tv"] - rng, b["tv"] + rng, step)
        # ▶ 計算済みの点 (横, 縦) の集合
        have = set(zip(np.round(pts["th"], 6), np.round(pts["tv"], 6)))
        # ▶ 細かい格子の点のうち、まだ計算していない点のリスト
        new = [(h, v) for h in hs for v in vs if (round(float(h), 6), round(float(v), 6)) not in have]
        # ▶ まだの点があれば（空のリストは「偽」として扱われる）
        if new:
            # ▶ それを計算して、表の下につなげる（pd.concat は表をつなぐ。ignore_index=True で行番号を振り直す）
            pts = pd.concat([pts, run(new, "細かい")], ignore_index=True)
        # ▶ 計算した後の、LML が最大の点
        b2 = pts.loc[pts["LML"].idxmax()]
        # ▶ 最大の点が動かなければ（周りをすべて計算しても、まだ最良）
        if (b2["th"], b2["tv"]) == (b["th"], b["tv"]):       # 最良の点が動かなかった：周りがすべて計算済みで、最良
            # ▶ 収束した印を付ける
            converged = True
            # ▶ break：くり返しを途中で終える
            break
    # ▶ 同じ点が 2 回あれば LML の大きい方を残し、係数の順に並べ直す（sort_values は並べ替え、drop_duplicates は重複を除く）
    pts = pts.sort_values("LML", ascending=False).drop_duplicates(subset=["th", "tv"]).sort_values(["th", "tv"]).reset_index(drop=True)
    # ▶ 全部の点の表と、収束したかを返す
    return pts, converged


# 尤度比検定の目安：2 つの係数（横・縦）を足したときの 5% 基準。2·ΔLML > 5.99（自由度 2 のカイ二乗分布の 95% 点）→ ΔLML > 3.0
# ▶ 尤度比検定の目安：最良の係数と係数 0 の LML の差が 3.0 以上なら、シフトの効果が 5% 基準で認められる（2 × 3.0 = 6.0 > 5.99）
LRT_DLML = 3.0


# ▶ ★関数：プロファイル尤度の表から、最良の係数と、そのおおよその 95% 区間を取り出す
def profile_best(pts, key, cfg, converged=True, lml_zero=None):
    # ▶ 関数の説明（docstring）
    """最良の θ と、LML が最大から 1.92 以内の範囲（おおよその 95% 区間）。at_edge：最良の点が粗い格子の端。refine_converged：細かい格子の取り直しが収束したか。
    範囲は、計算した点の中だけで決まる（斜めの尾根の上では、実際の範囲はもっと長い）。尾根に沿った方向は決まりにくく、差 θ_v − θ_h（diff_*）が最もよく決まる"""
    # ▶ LML が最大の行（最良の係数）
    r = pts.loc[pts["LML"].idxmax()]
    # ▶ LML が最大から 1.92 以内の点（おおよその 95% 区間に入る点）
    inside = pts[pts["LML"] >= r["LML"] - 1.92]
    # ▶ 粗い格子（最良の点が端に当たったかを調べるため）
    hg, vg = grid_of(*cfg["h_grid"]), grid_of(*cfg["v_grid"])
    # ▶ 係数が 0（シフトなし）の点
    z = pts[(pts["th"] == 0.0) & (pts["tv"] == 0.0)]
    # ▶ 係数 0 の点の LML（その点がなければ、渡された lml_zero か nan）。iloc[0] は最初の行
    lz = float(z["LML"].iloc[0]) if len(z) else (np.nan if lml_zero is None else lml_zero)
    # ▶ 95% 区間の中の点の、差（縦の係数 − 横の係数）
    diff = inside["tv"] - inside["th"]
    # ▶ 最良の点が、粗い格子の端にあるか（端なら、範囲を広げる必要がある）
    edge = bool(r["tv"] in (vg.min(), vg.max()) or (key.startswith("B") and r["th"] in (hg.min(), hg.max())))
    # ▶ 結果を辞書で返す：最良の横・縦の係数、その差、最良の LML
    return {"th_best": float(r["th"]), "tv_best": float(r["tv"]), "diff_best": float(r["tv"] - r["th"]), "LML_best": float(r["LML"]),
            # ▶ （続き）係数 0 の LML、その差 ΔLML、横の係数の 95% 区間（最小〜最大）
            "LML_at_0": lz, "dLML_vs_0": float(r["LML"]) - lz, "th_lo": float(inside["th"].min()), "th_hi": float(inside["th"].max()),
            # ▶ （続き）縦の係数と、差の 95% 区間
            "tv_lo": float(inside["tv"].min()), "tv_hi": float(inside["tv"].max()), "diff_lo": float(diff.min()), "diff_hi": float(diff.max()),
            # ▶ （続き）区間の中の点の数、計算した点の数、端に当たったか、取り直しが収束したか
            "n_in_95": int(len(inside)), "n_points": int(len(pts)), "at_edge": edge, "refine_converged": bool(converged),
            # ▶ （続き）最良の点で学習したハイパーパラメータ（最終モデルの学習の初期値に使う）
            "theta_best": np.asarray(r["theta"], float) if "theta" in pts.columns else None,
            # ▶ （続き）係数 0 の点で学習したハイパーパラメータ
            "theta_zero": np.asarray(z["theta"].iloc[0], float) if (len(z) and "theta" in pts.columns) else None}


# ---------------------------------------------------------------- 評価
# ▶ 関数：予測の当たり具合の指標を計算する（y は実測の ln η、mu は予測平均、sd は予測標準偏差）
def evaluate(y, mu, sd):
    # ▶ 関数の説明（docstring）
    """ln η の指標：R2、RMSE、95% 予測区間の被覆率、NLPD、予測標準偏差の平均。R2_real は粘度の実数値での R2（6月のノートと同じ）"""
    # ▶ 実測と予測の差を、予測の標準偏差で割った値
    z = (y - mu) / sd
    # ▶ R²（決定係数。データが 1 つなら nan）と、RMSE（誤差の 2 乗の平均の平方根。ln η で 0.1 なら、粘度で約 10%）
    return {"R2": r2_score(y, mu) if len(y) > 1 else np.nan, "RMSE": float(np.sqrt(mean_squared_error(y, mu))),
            # ▶ （続き）被覆率（実測が 95% 予測区間＝予測平均 ± 1.96 × 標準偏差 に入った割合）と、NLPD（予測の分布から見た実測の起こりにくさ。小さいほど良い）
            "coverage95": float(np.mean(np.abs(z) <= 1.96)), "NLPD": float(np.mean(0.5 * np.log(2 * np.pi * sd ** 2) + 0.5 * z ** 2)),
            # ▶ （続き）予測標準偏差の平均と、粘度の実数値（np.exp で ln を戻した値）での R²
            "mean_sd": float(np.mean(sd)), "R2_real": r2_score(np.exp(y), np.exp(mu)) if len(y) > 1 else np.nan}


# ▶ 関数：検証データの指標を、全体・低速側・高速側ごとに計算して、表の行のリストにする
def metrics_rows(mat, key, te, mu, sd, sd_f, extra=None):
    # ▶ 関数の説明（docstring）
    """検証の指標を、全体・低速側・高速側（と、感度確認の条件を除いたもの）ごとに返す"""
    # ▶ 集計する範囲の辞書。まず「全体」（全部の行が True）
    subsets = {"全体": np.ones(len(te), bool)}
    # ▶ 区分ごとに
    for g in ["低速側", "高速側", "範囲内"]:
        # ▶ その区分の行が 1 つでもあれば（any は「どれか 1 つでも True か」）
        if (te["group"] == g).any():
            # ▶ その区分の行を True にした配列を足す
            subsets[g] = (te["group"] == g).to_numpy()
    # ▶ 感度確認の行があれば
    if te["is_sens"].any():
        # ▶ 各範囲について、感度確認の行を除いた範囲も足す
        subsets.update({s + "（感度確認を除く）": m & ~te["is_sens"].to_numpy() for s, m in list(subsets.items())})
    # ▶ 結果の行のリスト
    rows = []
    # ▶ 実測の ln η
    y = te["ln_eta"].to_numpy()
    # ▶ 範囲の名前 s と、その行の印 m を 1 つずつ取り出す（items は、辞書の名前と値の組）
    for s, m in subsets.items():
        # ▶ 行が 1 つもなければ（True の数が 0）
        if m.sum() == 0:
            # ▶ とばす
            continue
        # ▶ 材料・モデル・範囲・行数・条件数と、指標を 1 行にする（**evaluate(…) は、返ってきた辞書の中身を、ここに並べる書き方）
        rows.append({"材料": MAT_NAMES[mat], "モデル": key, "範囲": s, "行数": int(m.sum()), "条件数": int(te.loc[m, "cond"].nunique()),
                     # ▶ （続き）曲線そのものの標準偏差の平均と、追加の項目（extra）
                     **evaluate(y[m], mu[m], sd[m]), "mean_sd_latent": float(np.nanmean(sd_f[m])) if np.isfinite(sd_f[m]).any() else np.nan, **(extra or {})})
    # ▶ 行のリストを返す
    return rows


# ▶ 関数：判定を表示して記録する（ok が True なら「OK」、False なら「★要確認」）
def judge(ok, text, mat=None, section=""):
    # ▶ 関数の説明（docstring）
    """結果の判定を 1 行で表示し、記録する。ok が True なら「OK」、False なら「★要確認」"""
    # ▶ 材料が指定されていれば、[材料名] を頭に付ける
    head = f"[{MAT_NAMES[mat]}] " if mat else ""
    # ▶ 判定を 1 行で表示する
    print(f">> 判定：{'OK' if ok else '★要確認'}　{head}{text}")
    # ▶ 判定を記録のリストに足す（最後の「まとめ」の表に使う）
    JUDGES.append({"節": section, "材料": MAT_NAMES[mat] if mat else "", "判定": "OK" if ok else "★要確認", "内容": text})


# ▶ 関数：表を CSV に保存する
def to_csv(df, path):
    # ▶ 行番号は書かず（index=False）、Excel で文字化けしない BOM 付き UTF-8 で保存する
    df.to_csv(path, index=False, encoding="utf-8-sig")


# ---------------------------------------------------------------- 外れ値の検知（本体の check_data.py と同じ規則。05 のデータ用）
# ▶ 関数：条件ごとの平均の表を作る
def cond_means(df):
    # ▶ 関数の説明（docstring）
    """条件ごとの平均（ln η と ln γ̇ をショットで平均した点）"""
    # ▶ 条件ごとに分ける（sort=False：元の順番のまま）
    g = df.groupby("cond", sort=False)
    # ▶ 条件ごとに集計する：設定温度・厚み・射出速度は最初の値（first）、ln η と ln γ̇ は平均（mean）。「名前=(列, 集計の方法)」の書き方
    t = g.agg(Tset=("Tset", "first"), H=("H", "first"), v=("v", "first"), ln_eta=("ln_eta", "mean"), ln_gd=("ln_gd", "mean"),
              # ▶ （続き）ショットの数（size）と組。reset_index で、条件名を普通の列に戻す
              n=("ln_eta", "size"), grp=("grp", "first")).reset_index()
    # ▶ 厚みの ln の列を足す
    t["ln_H"] = np.log(t["H"])
    # ▶ 表を返す
    return t


# ▶ 関数：外れ値の規則 f。条件平均が、ほかの条件から予測した値と大きく違う条件を拾う
def outlier_scan(df):
    # ▶ 関数の説明（docstring）
    """規則 f：条件平均が、ほかの条件から予測した値と大きく違う条件を拾う（本体の nb_common.outlier_scan と同じ）。
    条件平均の GPR（入力 ln γ̇（条件平均）・ln H・Tset、線形項あり）の、条件を 1 つずつ抜いた予測（閉じた式）との残差を、予測標準偏差で割って z とする。
    ※ この検知だけは、データの確認のための道具なので、本体と同じく H を使う（モデルの説明変数ではない）"""
    # ▶ 条件ごとの平均の表
    t = cond_means(df)
    # ▶ 入力（ln γ̇・ln H・Tset）と、出力（ln η）を配列にする
    X, y = t[["ln_gd", "ln_H", "Tset"]].to_numpy(), t["ln_eta"].to_numpy()
    # ▶ 入力を標準化し（StandardScaler）、出力から平均を引く
    Xs, yc = StandardScaler().fit_transform(X), y - y.mean()
    # ▶ 全部の条件で GPR を学習する（カーネルは線形項あり）
    gp = GaussianProcessRegressor(kernel=kernel_for(3, True), n_restarts_optimizer=N_RESTARTS, random_state=SEED).fit(Xs, yc)
    # ▶ 学習したカーネルの行列（条件どうしの共分散）の逆行列を計算する（np.linalg.inv）
    Kinv = np.linalg.inv(gp.kernel_(Xs))
    # ▶ 逆行列の対角の値
    d = np.diag(Kinv)
    # ▶ 条件を 1 つずつ抜いて予測したときの残差（実測 − 予測）と予測の標準偏差を、学習し直さずに一度に計算する公式（@ は行列の掛け算）
    t["resid"], t["sd"] = (Kinv @ yc) / d, np.sqrt(1.0 / d)       # 条件 i を抜いた予測：平均 = y_i − (K⁻¹y)_i / K⁻¹_ii、分散 = 1 / K⁻¹_ii
    # ▶ z = 残差 ÷ 予測の標準偏差
    t["z"] = t["resid"] / t["sd"]
    # ▶ |z| > 3.5 かつ |残差| > 0.1 の条件に印を付ける（abs は絶対値、& は「かつ」）
    t["flag"] = (t["z"].abs() > OUTLIER_Z) & (t["resid"].abs() > OUTLIER_RESID)
    # ▶ 必要な列だけを選んで返す
    return t[["cond", "grp", "Tset", "H", "v", "n", "resid", "sd", "z", "flag"]]


# ▶ 関数：ある条件（やショット）が、問題データ一覧の「除外」か「感度確認」に登録済みかを調べ、ID を返す
def registered(mat, grp, row, plist):
    # ▶ 関数の説明（docstring）
    """一覧の「除外」「感度確認」に、この条件（row に shot があれば、そのショット）が登録されているか（ID を返す）"""
    # ▶ その条件の、1 行だけの表を作る（ショットの指定がなければ −1）
    one = pd.DataFrame({"Tset": [float(row["Tset"])], "H": [float(row["H"])], "v": [float(row["v"])], "shot": [row.get("shot", -1)]})
    # ▶ ショットが空（nan）なら
    if pd.isna(one["shot"].iloc[0]):
        # ▶ −1（どのショットとも一致しない値）にする
        one["shot"] = -1
    # ▶ 一覧に当てはまるかを調べる
    _, ids = problem_mask(one, mat, grp, plist, ("除外", "感度確認"))
    # ▶ その行の ID（なければ ""）を返す
    return ids[0]


# ▶ 関数：問題の候補（見直しの候補）を、規則 a・b・c・e・g で拾う
def candidates(mat, pool, plist):
    # ▶ 関数の説明（docstring）
    """問題の候補（見直しの候補であって、外れ値と判定したものではない。一覧に載せるかはユーザーが決める）。
    a 温度の間で γ̇ が 1.3 倍を超えて違う／b 最低温度の粘度が最高温度より小さい（順序が逆）／c 同じ条件内の ln η の標準偏差 > 0.1／
    e 別のショットと値が完全に同じ行／g P3_3 が条件の中央値から P33_DEV 以上離れたショット"""
    # ▶ 候補を入れるリスト
    cand = []
    # ▶ 厚み・射出速度・設定温度ごとに、γ̇ と η の平均をとった表
    g = pool.groupby(["H", "v", "Tset"]).agg(gd=("gd", "mean"), eta=("eta", "mean"), grp=("grp", "first")).reset_index()
    # ▶ 最も低い設定温度と、最も高い設定温度
    t_lo, t_hi = pool["Tset"].min(), pool["Tset"].max()
    # ▶ 厚みと射出速度の組ごとに（s は、その組の温度ごとの行）
    for (h, v), s in g.groupby(["H", "v"]):
        # ▶ 温度が 3 水準そろっていなければ
        if len(s) < 3:
            # ▶ とばす
            continue
        # ▶ 温度ごとの γ̇ の ln
        lg = np.log(s["gd"].to_numpy())
        # ▶ 規則 a：温度の間で γ̇ が 1.3 倍を超えて違えば
        if np.exp(lg.max() - lg.min()) > 1.3:
            # ▶ 中央値から最も離れた温度の行（np.argmax は、最大の位置の番号）
            odd = s.iloc[np.argmax(np.abs(lg - np.median(lg)))]
            # ▶ 候補として記録する
            cand.append({"規則": "a 温度の間でせん断速度が1.3倍超違う", "組": odd["grp"], "設定温度": int(odd["Tset"]), "厚み": h, "射出速度": int(v), "ショット": "",
                         # ▶ （続き）内容の説明（γ̇ の最大/最小の比。:.2f は、小数 2 桁で表示する書き方）
                         "内容": f"γ̇ の最大/最小 = {np.exp(lg.max() - lg.min()):.2f}"})
        # ▶ 設定温度から粘度を引ける形にする（set_index は、列を行の名前にする）
        e = s.set_index("Tset")["eta"]
        # ▶ 規則 b：最も低い温度の粘度が、最も高い温度の粘度の 0.95 倍より小さければ（温度と粘度の順序が逆）
        if t_lo in e.index and t_hi in e.index and e[t_lo] / e[t_hi] < 0.95:
            # ▶ 候補として記録する
            cand.append({"規則": "b 粘度の順序が温度と逆", "組": s["grp"].iloc[0], "設定温度": int(t_lo), "厚み": h, "射出速度": int(v), "ショット": "",
                         # ▶ （続き）内容の説明（粘度の比）
                         "内容": f"η({int(t_lo)})/η({int(t_hi)}) = {e[t_lo] / e[t_hi]:.2f}"})
    # ▶ 条件ごとの ln η の標準偏差と、ショットの数
    sd = pool.groupby(["Tset", "H", "v"]).agg(std=("ln_eta", "std"), size=("ln_eta", "size"), grp=("grp", "first")).reset_index()
    # ▶ 規則 c：ショットが 2 つ以上で、標準偏差が 0.1 を超える条件を 1 つずつ
    for _, r in sd[(sd["size"] >= 2) & (sd["std"] > 0.1)].iterrows():
        # ▶ 候補として記録する
        cand.append({"規則": "c 同じ条件内のばらつきが大きい", "組": r["grp"], "設定温度": int(r["Tset"]), "厚み": r["H"], "射出速度": int(r["v"]), "ショット": "",
                     # ▶ （続き）内容の説明
                     "内容": f"ln η の標準偏差 = {r['std']:.3f}（{int(r['size'])} ショット）"})
    # ▶ 規則 e で比べる列（η・γ̇・圧力のうち、データにあって空欄のない列）
    key = [c for c in ["eta", "gd", "P31", "P32", "P33"] if c in pool.columns and pool[c].notna().all()]
    # ▶ 組（5data・3data）ごとに
    for grp, df in pool.groupby("grp"):
        # ▶ それらの列の値が、ほかの行と完全に同じ行（duplicated は重複の印。keep=False で、重複の全部の行に印を付ける）
        dup = df[df.duplicated(key, keep=False)]
        # ▶ 同じ値の行のまとまりごとに
        for _, gg in dup.groupby(key):
            # ▶ 同じファイルの行どうしなら（nunique は、異なる値の数）
            if gg["file"].nunique() < 2:
                # ▶ とばす
                continue
            # ▶ ファイル名の順で、最初の行
            r0 = gg.sort_values("file").iloc[0]
            # ▶ 規則 e：別のショットと値が完全に同じ行として、記録する
            cand.append({"規則": "e 別のショットと同じ値の行", "組": grp, "設定温度": int(r0["Tset"]), "厚み": r0["H"], "射出速度": int(r0["v"]), "ショット": str(int(r0["shot"])),
                         # ▶ （続き）内容の説明（どのファイルの、どの列が一致したか）
                         "内容": f"{' と '.join(gg['file'])} の {'・'.join(key)} が完全に一致"})
    # ▶ 条件ごとの P3_3 の中央値を、各行に書き込む
    med = pool.groupby("cond")["P33"].transform("median")
    # ▶ 条件ごとのショットの数を、各行に書き込む
    n = pool.groupby("cond")["P33"].transform("size")
    # ▶ 各ショットの P3_3 が、中央値から何割離れているか
    dev = (pool["P33"] / med - 1.0).abs()
    # ▶ 規則 g：ショットが 3 つ以上の条件で、中央値から 20% 以上離れたショットを 1 つずつ
    for _, r in pool[(n >= 3) & (dev >= P33_DEV)].iterrows():
        # ▶ 候補として記録する（規則の名前、組、設定温度、厚み、射出速度）
        cand.append({"規則": f"g P3_3 が条件の中央値から{int(P33_DEV * 100)}%以上離れている", "組": r["grp"], "設定温度": int(r["Tset"]), "厚み": r["H"], "射出速度": int(r["v"]),
                     # ▶ （続き）ショット番号と、内容の説明（P3_3 の値、条件の中央値、何 % 離れているか。r.name はその行の番号、:+.0f は符号付きで小数なしの書き方）
                     "ショット": str(int(r["shot"])), "内容": f"P3_3 = {r['P33']:.2f} MPa（条件の中央値 {med[r.name]:.2f}、{(r['P33'] / med[r.name] - 1) * 100:+.0f}%）"})
    # ▶ 候補のリストを表にする（列の順番を指定）
    out = pd.DataFrame(cand, columns=["規則", "組", "設定温度", "厚み", "射出速度", "ショット", "内容"])
    # ▶ 先頭（0 番目）に「材料」の列を入れる
    out.insert(0, "材料", MAT_NAMES[mat])
    # ▶ 各候補が、問題データ一覧に登録済みかを調べて、その ID を「一覧の ID」の列にする
    out["一覧の ID"] = [registered(mat, r["組"], {"Tset": r["設定温度"], "H": r["厚み"], "v": r["射出速度"],
                                                   # ▶ （続き）ショット番号があれば整数に、なければ nan にして渡す
                                                   "shot": int(r["ショット"]) if r["ショット"] != "" else np.nan}, plist) for _, r in out.iterrows()]
    # ▶ 候補の表を返す
    return out


# ---------------------------------------------------------------- Cross-WLF（6月のノートのセル10.5・11。D3 = 0）
# ▶ Cross-WLF モデルの係数の名前のリスト
CWLF_NAMES = ["n", "tau_star", "D1", "D2", "A1", "A2"]


# ▶ 関数：Cross-WLF モデルで ln η を計算する（比較用）。T_c は温度 [℃]、gd はせん断速度、ほかは係数
def cross_wlf_ln(T_c, gd, n, ln_tau, ln_D1, D2, A1, A2):
    # ▶ 関数の説明（docstring）
    """ln η（Cross-WLF。D3 = 0 なので T* = D2）。数値の安定のため、τ* と D1 は ln で受け取る。T は ℃、D2 は K"""
    # ▶ 温度（K に直したもの）と D2 の差。小さくなりすぎないように、1 以上にする
    diff = np.maximum(np.asarray(T_c, float) + 273.15 - D2, 1.0)
    # ▶ ゼロせん断粘度 η0 の ln（WLF 型の温度依存性）：ln η0 = ln D1 − A1(T − D2)/(A2 + T − D2)
    ln_eta0 = ln_D1 - A1 * diff / (A2 + diff)
    # ▶ Cross の式 η = η0 / (1 + (η0·γ̇/τ*)^(1−n)) の ln を計算して返す（np.logaddexp(0, x) は ln(1 + eˣ) を、桁あふれせずに計算する）
    return ln_eta0 - np.logaddexp(0.0, (1.0 - n) * (ln_eta0 + np.log(gd) - ln_tau))


# ▶ Cross-WLF の当てはめの出発点の数（設定エリアの値。なければ 12）
CWLF_STARTS = globals().get("CWLF_STARTS", 12)            # Cross-WLF の当てはめの初期値の数（局所解を避けるため、乱数の初期値を何度か試して、残差が最小のものを採る）


# ▶ 関数：Cross-WLF の係数を、学習データに合わせて求める。t_col は使う温度の列、fixed は固定する係数の辞書
def fit_cross_wlf(tr, t_col, fixed):
    # ▶ 関数の説明（docstring）
    """Cross-WLF の係数を、学習データ（ln η の最小二乗）で当てはめる。fixed：{名前: 値 または None（探索）}。
    初期値は、6月のノートに近い値と、探索範囲の中の乱数（SEED で固定）の CWLF_STARTS 個で、残差が最小のものを採る。
    返す辞書：係数（実数）、学習データでの残差の標準偏差"""
    # ▶ SciPy の、曲線の当てはめの関数 curve_fit を読み込む
    from scipy.optimize import curve_fit
    # ▶ 係数の出発点（τ* と D1 は桁が大きいので ln で扱う）
    init = {"n": 0.3, "tau_star": np.log(3e4), "D1": np.log(1e12), "D2": 263.15, "A1": 25.0, "A2": 51.6}
    # ▶ 係数を探す範囲の下限
    lo = {"n": 0.0, "tau_star": np.log(1e2), "D1": np.log(1e3), "D2": 150.0, "A1": 0.5, "A2": 10.0}
    # ▶ 係数を探す範囲の上限
    hi = {"n": 0.95, "tau_star": np.log(1e7), "D1": np.log(1e30), "D2": 450.0, "A1": 80.0, "A2": 200.0}
    # ▶ 固定する係数の値を、内部の形（τ* と D1 は ln）に直す
    conv = {k: (np.log(v) if k in ("tau_star", "D1") and v is not None else v) for k, v in fixed.items()}
    # ▶ 固定しない（None の）係数の名前のリスト。これを求める
    free = [k for k in CWLF_NAMES if conv.get(k) is None]

    # ▶ 関数の中の関数：求める係数の値（args）と、固定した係数を合わせて、6 つの係数の並びにする
    def full(args):
        # ▶ 固定した係数の辞書の写し
        vals = dict(conv)
        # ▶ 求める係数の値を、名前と組にして足す
        vals.update(dict(zip(free, args)))
        # ▶ 6 つの係数を、決まった順に並べて返す
        return [vals[k] for k in CWLF_NAMES]

    # ▶ 温度、せん断速度、ln η を配列にする
    T, gd, y = tr[t_col].to_numpy(float), tr["gd"].to_numpy(float), tr["ln_eta"].to_numpy()
    # ▶ 当てはめる関数（入力 X の 1 行目が温度、2 行目がせん断速度）。*args は、求める係数をまとめて受け取る書き方
    f = lambda X, *args: cross_wlf_ln(X[0], X[1], *full(args))
    # ▶ 乱数の発生器を、たね SEED で作る
    rs = np.random.RandomState(SEED)
    # ▶ 出発点のリスト：最初の値 1 つと、範囲の中の乱数（uniform は一様な乱数）の点
    starts = [[init[k] for k in free]] + [[rs.uniform(lo[k], hi[k]) for k in free] for _ in range(max(CWLF_STARTS - 1, 0))]
    # ▶ 一番良い結果を入れる変数（まだなし）
    best = None
    # ▶ 出発点を 1 つずつ試す
    for p0 in starts:
        # ▶ try：試しに実行する（当てはめが失敗することがあるため）
        try:
            # ▶ curve_fit で、誤差の 2 乗の和が最小になる係数を探す（bounds は探す範囲、maxfev は計算の回数の上限）
            p, _ = curve_fit(f, np.vstack([T, gd]), y, p0=p0, bounds=([lo[k] for k in free], [hi[k] for k in free]), maxfev=20000)
        # ▶ 失敗したら（Exception は、ほとんどのエラー）
        except Exception:
            # ▶ その出発点はとばす
            continue
        # ▶ 誤差の 2 乗の和（SSE）を計算する
        sse = float(np.sum((y - f(np.vstack([T, gd]), *p)) ** 2))
        # ▶ これまでで一番小さければ（or は「または」）
        if best is None or sse < best[0]:
            # ▶ 一番良い結果として覚える
            best = (sse, p)
    # ▶ すべての出発点で失敗していたら
    if best is None:
        # ▶ エラーで止める
        raise RuntimeError("Cross-WLF の当てはめがすべての初期値で失敗しました")
    # ▶ 一番良かった係数
    popt = best[1]
    # ▶ 固定した係数と合わせた、6 つの係数
    vals = full(popt)
    # ▶ 残差（実測 − Cross-WLF の値）
    resid = y - cross_wlf_ln(T, gd, *vals)
    # ▶ 探す範囲の下限か上限に張り付いた係数の名前を集める
    at_bound = [k for k, v in zip(free, popt) if np.isclose(v, lo[k], rtol=0, atol=1e-6 * max(1.0, abs(lo[k]))) or np.isclose(v, hi[k], rtol=0, atol=1e-6 * max(1.0, abs(hi[k])))]
    # ▶ 係数の名前と値の辞書
    out = dict(zip(CWLF_NAMES, vals))
    # ▶ τ* と D1 を、ln から元の値に戻す（np.exp）
    out["tau_star"], out["D1"] = float(np.exp(out["tau_star"])), float(np.exp(out["D1"]))
    # ▶ 係数、求めた係数の名前、張り付いた係数、残差の標準偏差、使った温度の列を辞書で返す
    return {"params": out, "free": free, "at_bound": at_bound, "resid_sd": float(np.sqrt(np.mean(resid ** 2))), "t_col": t_col}


# ▶ 関数：当てはめた Cross-WLF で、表 df の ln η を予測する
def predict_cross_wlf(cw, df):
    # ▶ 係数の辞書を取り出す
    p = cw["params"]
    # ▶ Cross-WLF の式で ln η を計算する
    mu = cross_wlf_ln(df[cw["t_col"]].to_numpy(float), df["gd"].to_numpy(float), p["n"], np.log(p["tau_star"]), np.log(p["D1"]), p["D2"], p["A1"], p["A2"])
    # ▶ 予測と、ばらつき（学習データの残差の標準偏差を、全行に同じ値で並べた配列。np.full）を返す
    return mu, np.full(len(df), cw["resid_sd"])


# ---------------------------------------------------------------- 予測の入力を組み立てる（依存性グラフ・3次元グラフ・保存したモデルの使い方）
# ▶ 説明変数の名前・単位・対数軸で描くか の辞書（依存性グラフに使う）
VAR_JA = {"gd": ("せん断速度 γ̇", "(1/s)", True), "P33": ("流動中樹脂圧力 P3_3", "(MPa)", True),
          # ▶ （辞書の続き）流動中樹脂温度と加熱シリンダ温度
          "Tff": ("流動中樹脂温度 Tff", "(℃)", False), "Tset": ("加熱シリンダ温度 Tset", "(℃)", False),
          # ▶ （辞書の続き）Tboth：設定温度と流動中樹脂温度を、同じだけ一緒に動かす場合
          "Tboth": ("Tset と Tff を同時に", "(℃。横軸は Tset)", False)}     # (名前, 単位, 対数軸か)。Tboth：設定温度を上げると、流動中樹脂温度も同じだけ上がるとして、2 つを一緒に動かす


# ▶ 関数：説明変数の値（γ̇・P3_3・Tff・Tset）から、predict に渡せる表を作る（予測線を描くときに使う）
def model_inputs(gd, P33, Tff, Tset):
    # ▶ 関数の説明（docstring）
    """説明変数の実数値（γ̇ [1/s]、P3_3 [MPa]、Tff [℃]、Tset [℃]。配列でも数でもよい）→ predict に渡せる表"""
    # ▶ 4 つの値の形をそろえ（np.broadcast_arrays。1 つの数は、配列と同じ長さに広げる）、1 列の配列にする（ravel）
    gd, P33, Tff, Tset = (np.asarray(a, float).ravel() for a in np.broadcast_arrays(gd, P33, Tff, Tset))
    # ▶ モデルが使う列（値と、その ln など）をそろえた表を返す
    return pd.DataFrame({"gd": gd, "ln_gd": np.log(gd), "P33": P33, "ln_P33": np.log(P33 * 1e6), "Tff": Tff, "ln_Tff": np.log(Tff + 273.15), "Tset": Tset})


# ▶ 関数：依存性グラフで、動かさない変数を固定する値を決める
def base_point(tr, cfg):
    # ▶ 関数の説明（docstring）
    """依存性グラフで固定する値：学習データの中央値（γ̇・P3_3 は対数の中央値）。設定温度は基準温度 Tref"""
    # ▶ γ̇ と P3_3 は、学習データの ln の中央値を元に戻した値（P3_3 は Pa から MPa に戻す）
    return {"gd": float(np.exp(tr["ln_gd"].median())), "P33": float(np.exp(tr["ln_P33"].median()) / 1e6),
            # ▶ （続き）Tff は ln の中央値を ℃ に戻した値、設定温度は基準温度 Tref
            "Tff": float(np.exp(tr["ln_Tff"].median()) - 273.15), "Tset": float(cfg["Tref"])}


# ▶ 関数：グラフで動かす説明変数の値の並び（格子）を作る。n は点の数、pad は両側に広げる割合
def var_grid(pool, var, n, pad=0.1):
    # ▶ 関数の説明（docstring）
    """動かす説明変数の格子（データの範囲を少し広げる。対数軸の変数は対数で等間隔。設定温度は ±10 ℃）"""
    # ▶ 設定温度（または Tset と Tff を一緒に動かす場合）なら
    if var in ("Tset", "Tboth"):
        # ▶ 設定温度の範囲の ±10 ℃ を、等間隔に n 点（np.linspace）
        return np.linspace(pool["Tset"].min() - 10.0, pool["Tset"].max() + 10.0, n)
    # ▶ その変数のデータの値
    x = pool[var].to_numpy(float)
    # ▶ 対数軸の変数なら（VAR_JA の 3 番目が True）
    if VAR_JA[var][2]:
        # ▶ 最小と最大の ln
        lo, hi = np.log(x.min()), np.log(x.max())
        # ▶ ln で等間隔に並べ（両側に pad の割合だけ広げる）、元の値に戻す
        return np.exp(np.linspace(lo - pad * (hi - lo), hi + pad * (hi - lo), n))
    # ▶ 普通の軸の変数なら、最小と最大
    lo, hi = x.min(), x.max()
    # ▶ 等間隔に n 点（両側に pad の割合だけ広げる）
    return np.linspace(lo - pad * (hi - lo), hi + pad * (hi - lo), n)


# ▶ 関数：重ね合わせの目安（マスターカーブの残差）を計算する
def superpose_rmse(t):
    # ▶ 関数の説明（docstring）
    """重ね合わせの目安：厚みごとに、条件平均の点（x = ln γ̇〔換算〕、y = ln η〔換算〕）を 3 次式で当てはめた残差の RMSE（全温度をまとめて）。
    温度のシフトで曲線が重なるほど小さくなる"""
    # ▶ 残差を集めるリスト
    res = []
    # ▶ 厚みごとに
    for _, g in t.groupby("H"):
        # ▶ 点が 6 つより少なければ
        if len(g) < 6:
            # ▶ とばす
            continue
        # ▶ 換算した点（x = ln γ̇、y = ln η）に、3 次式を当てはめる（np.polyfit）
        c = np.polyfit(g["x"], g["y"], 3)
        # ▶ 3 次式からの残差を、リストに足す（np.polyval は式の値を計算する。+= はリストの後ろにつなげる）
        res += list(g["y"] - np.polyval(c, g["x"]))
    # ▶ 残差の RMSE を返す（残差がなければ nan）
    return float(np.sqrt(np.mean(np.square(res)))) if res else np.nan


# ▶ ★関数：条件平均の点を、シフトファクターで換算する（マスターカーブの図と、重ね合わせの目安に使う）
def shifted_cond_means(df, key, cfg, th, tv):
    # ▶ 関数の説明（docstring）
    """条件平均の、換算した ln γ̇（x）と ln η（y）。key が None ならシフト前。Tff 版はショットごとの Tff でずらしてから平均する"""
    # ▶ ln γ̇ と ln η を配列にする
    x, y = df["ln_gd"].to_numpy(), df["ln_eta"].to_numpy()
    # ▶ モデルが指定されていれば（None ならシフトしない）
    if key is not None:
        # ▶ シフトに使う温度（layout(key)[1] が、温度の列の名前）
        T = df[layout(key)[1]].to_numpy(float)
        # ▶ 横のシフト：ln γ̇ に ln a_h(T) を足す（γ̇ × a_h にあたる）
        x = x + shift_ln(T, th, cfg)
        # ▶ 縦のシフト：ln η から ln a_v(T) を引く（η ÷ a_v にあたる）
        y = y - shift_ln(T, tv, cfg)
    # ▶ 換算した x・y を表に足して（assign）、条件ごとに平均した表を返す
    return df.assign(x=x, y=y).groupby("cond").agg(x=("x", "mean"), y=("y", "mean"), Tset=("Tset", "first"), H=("H", "first")).reset_index()


# ---------------------------------------------------------------- 図の保存と、色・記号
# ▶ 厚みごとの点の色（#d62728 などは色の番号。赤・青・黄・緑。6月のノートと同じ並び）
H_COLORS = {0.5: "#d62728", 1.0: "#1f77b4", 1.5: "#f0b400", 2.0: "#2ca02c"}       # 厚みごとの色（6月のノートと同じ並び）
# ▶ 厚みごとの点の形（o 丸、s 四角、^ 三角、D ひし形）
H_MARKS = {0.5: "o", 1.0: "s", 1.5: "^", 2.0: "D"}
# ▶ 設定温度ごとの色（低い順に、青・緑・赤…）
T_PALETTE = ["#1f77b4", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]               # 設定温度ごとの色（低い順）
# ▶ 設定温度ごとの点の形
T_MARKS = ["o", "s", "^", "D", "v"]


# ▶ 関数：設定温度の水準から、温度 → (色, 形) の辞書を作る
def t_style(levels):
    # ▶ 関数の説明（docstring）
    """設定温度の水準 → (色, 記号) の辞書（低い順に青・緑・赤）"""
    # ▶ 水準を小さい順に並べる
    lv = sorted(levels)
    # ▶ 辞書内包表記：i 番目の温度に、i 番目の色と形を割り当てる（enumerate は番号と値の組を返す。% は割った余りで、色が足りなければ最初に戻る）
    return {t: (T_PALETTE[i % len(T_PALETTE)], T_MARKS[i % len(T_MARKS)]) for i, t in enumerate(lv)}


# ▶ 関数：対数軸の目盛りの数字が重ならないように、目盛りの付け方を決める
def log_ticks(ax, axis="xy"):
    # ▶ 関数の説明（docstring）
    """対数軸の目盛りを、数字が重ならない形にする（軸の範囲を決めた後で呼ぶ）。範囲が 2.5 桁以下なら、1・2・5 × 10^n を普通の数で、
    それより広ければ、10 の累乗だけ。小さい目盛りには数字を付けない"""
    # ▶ matplotlib の目盛りの道具を、ticker という名前で読み込む
    import matplotlib.ticker as ticker
    # ▶ "x" と "y" の軸ごとに（文字列の 1 文字ずつ）
    for name in axis:
        # ▶ その軸を取り出す
        a = ax.xaxis if name == "x" else ax.yaxis
        # ▶ その軸の範囲（下限と上限）
        lo, hi = ax.get_xlim() if name == "x" else ax.get_ylim()
        # ▶ 範囲が正で、2.5 桁以下なら
        if hi > lo > 0 and np.log10(hi / lo) <= 2.5:
            # ▶ 1・2・5 × 10ⁿ の位置に目盛りを付ける
            a.set_major_locator(ticker.LogLocator(base=10.0, subs=(1.0, 2.0, 5.0)))
            # ▶ 目盛りの数字を、普通の数（例：200）で書く
            a.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{v:g}"))
        # ▶ 範囲が広ければ
        else:
            # ▶ 10ⁿ の位置だけに目盛りを付ける
            a.set_major_locator(ticker.LogLocator(base=10.0, subs=(1.0,)))
            # ▶ 目盛りの数字を、10ⁿ の形で書く
            a.set_major_formatter(ticker.LogFormatterSciNotation())
        # ▶ 小さい目盛りには数字を書かない
        a.set_minor_formatter(ticker.NullFormatter())


# ▶ 関数：対数軸の目盛りの数字が、隣どうしで重なるかを調べる（重なれば True を返す）
def tick_labels_overlap(fig, axis, gap=0.4):
    # ▶ 関数の説明（docstring）
    """軸 axis の大きい目盛りの数字のうち、グラフの範囲の中のものが、隣どうしで重なるか（間が、数字の高さの gap 倍より狭いか）。図を一度描いて確かめる"""
    # ▶ 図を一度描く（描かないと、数字の大きさと位置が決まらない）
    fig.canvas.draw()
    # ▶ 横軸か（True）、縦軸か（False）
    x = axis.axis_name == "x"
    # ▶ グラフの枠の、画面の上での位置と大きさ
    ab = axis.axes.get_window_extent()
    # ▶ グラフの枠の端（横軸なら左と右、縦軸なら下と上）
    lo, hi = (ab.x0, ab.x1) if x else (ab.y0, ab.y1)
    # ▶ 大きい目盛りの数字それぞれが占める四角（数字のない目盛りは除く）
    boxes = [t.get_window_extent() for t in axis.get_majorticklabels() if t.get_text()]
    # ▶ 四角の端（横軸なら左と右、縦軸なら下と上）を、小さい順に並べる
    spans = sorted((b.x0, b.x1) if x else (b.y0, b.y1) for b in boxes)
    # ▶ 中心がグラフの枠の中にある数字だけを残す（枠の外の目盛りの数字は、描かれないため）
    spans = [s for s in spans if lo - 1 <= (s[0] + s[1]) / 2 <= hi + 1]
    # ▶ 隣との間に要る隙間（数字の高さの gap 倍）
    pad = gap * max((b.height for b in boxes), default=0.0)
    # ▶ 隣どうしの間が、要る隙間より狭いところが 1 つでもあれば True
    return any(a[1] + pad > b[0] for a, b in zip(spans, spans[1:]))


# ▶ 関数：図を PNG で保存する（高解像度〔既定 600 dpi〕の 1 枚だけ。必要ならノートに表示する）
def save_fig(fig, out_dir, stem, show=None):
    # ▶ 関数の説明（docstring）
    """図を out_dir に PNG で保存する（解像度 FIG_DPI。既定 600 dpi の 1 枚だけ。報告書・発表資料にそのまま使える。2026-10-27 から、低い解像度の版は作らない）。
    show（既定 SHOW_FIGS）なら表示し、閉じる。保存したファイルの一覧を返す"""
    # ▶ グラフのライブラリを plt という名前で読み込む
    import matplotlib.pyplot as plt
    # ▶ 保存するフォルダを作る（すでにあってもよい）
    os.makedirs(out_dir, exist_ok=True)
    # ▶ 保存するファイルの場所（stem はファイル名の本体。.png を付ける）
    path = os.path.join(out_dir, f"{stem}.png")
    # ▶ 図を、解像度 FIG_DPI で保存する（bbox_inches="tight"：余白を詰める）
    fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
    # ▶ 図を表示する設定なら（show が指定されていればそれ、なければ SHOW_FIGS）
    if SHOW_FIGS if show is None else show:
        # ▶ 図をノートに表示する
        plt.show()
    # ▶ 図を閉じる（メモリを空ける）
    plt.close(fig)
    # ▶ 保存したファイルの場所を、リスト（[ ] で囲んだ並び。要素は 1 つ）で返す
    return [path]


# ▶ 依存性グラフ（6月のセル9-2 と同じ図）の、厚みごとの色・形・重なりの順（zorder：大きいほど手前）
DEP92_STYLE = {0.5: ("#d62728", "o", 14), 1.0: ("#1f77b4", "s", 13), 1.5: ("#ffd30ed9", "^", 12), 2.0: ("#41ea4f", "D", 11)}      # 6月のノートのセル9-2 の style_dict（色・形・zorder）
# ▶ 依存性グラフで動かす変数のリスト：(変数, 名前, 記号, 単位, 対数軸か, ファイル名)。せん断速度と P3_3
DEP92_VARS = [("gd", "せん断速度", r"$\gamma$", "(1/s)", True, "Shear_Rate"), ("P33", "流動中樹脂圧力", r"$P_{3\_3}$", "(MPa)", True, "P33"),
              # ▶ （リストの続き）流動中樹脂温度と加熱シリンダ温度
              ("Tff", "流動中樹脂温度", r"$T_{ff}$", "(℃)", False, "Tff"), ("Tset", "加熱シリンダ温度", r"$T_{set}$", "(℃)", False, "Tset")]      # (変数, 名前, 記号, 単位, 対数軸か, ファイル名)


# ▶ 関数：6月のノートのセル9-2 と同じ依存性グラフ（1 つの変数だけを動かし、ほかを固定したときの粘度の予測）を作る
# ▶ **kw は、名前付きの設定（figsize=… など）を、辞書 kw としてまとめて受け取る書き方
def dependency_plots_92(fit, mat, tr, out_dir, stem="依存性グラフ", **kw):
    # ▶ 関数の説明（docstring）
    """GPR6月ノートのセル9-2（目的変数に対する各説明変数の依存性グラフ）と同じ図を、学習データ（tr）と学習済みモデルで作る。
    説明変数ごとに 1 枚：その変数だけを動かし、ほかは学習データの平均（対数の変数は、対数の平均）に固定した、予測平均（赤線）と 95% 区間（帯）に、
    学習データ全部の点（色・形は厚み）を重ねる。図の大きさ・文字・点・目盛り（対数軸は 10 の累乗）・凡例・タイトルは、6月のセル9-2 と同じ。
    設定（kw）：figsize、dpi（保存の解像度。None なら FIG_DPI）、axis_fs・legend_fs・title_fs・tick_fs、plot_marker・legend_marker、show_xlabel・show_ylabel、ylim、
    xlim（変数 → (下限, 上限)）、n_grid、formats、show。保存したファイルの一覧を返す"""
    # ▶ グラフのライブラリを読み込む
    import matplotlib.pyplot as plt
    # ▶ 目盛りの道具を読み込む
    import matplotlib.ticker as ticker
    # ▶ 凡例に使う線の部品を読み込む
    from matplotlib.lines import Line2D
    # ▶ 凡例に使う四角の部品を読み込む
    from matplotlib.patches import Patch
    # ▶ 図の設定の既定値の辞書：大きさ、解像度、文字の大きさ、点の大きさ、軸名を出すか
    cfg = dict(figsize=(12, 10), dpi=None, axis_fs=24, legend_fs=24, title_fs=24, tick_fs=28, plot_marker=300, legend_marker=300, show_xlabel=False, show_ylabel=False,
               # ▶ （続き）縦軸の範囲、せん断速度の横軸の範囲、線の点の数、保存の形式、表示するか
               ylim=(0.1, 10000.0), xlim={"gd": (10.0, 100000.0)}, n_grid=100, formats=["png"], show=None)
    # ▶ 渡された設定（kw）で、既定値を上書きする
    cfg.update(kw)
    # ▶ 解像度（指定がなければ FIG_DPI）
    dpi = cfg["dpi"] or FIG_DPI
    # ▶ 動かさない変数を固定する値：学習データの平均（γ̇・P3_3・Tff は ln の平均を元に戻した値、設定温度は平均）
    base = {"gd": float(np.exp(tr["ln_gd"].mean())), "P33": float(np.exp(tr["ln_P33"].mean()) / 1e6), "Tff": float(np.exp(tr["ln_Tff"].mean()) - 273.15), "Tset": float(tr["Tset"].mean())}
    # ▶ 保存したファイルの場所を入れるリスト
    saved = []
    # ▶ 4 つの変数ごとに、図を 1 枚ずつ作る（タプルの中身を、6 つの変数で受け取る）
    for var, name, sym, unit, logx, fname in DEP92_VARS:
        # ▶ 図（fig）とグラフ（ax）を作る（plt.subplots。解像度は、保存するときに決める）
        fig, ax = plt.subplots(figsize=cfg["figsize"])
        # ▶ その変数の、学習データの値
        x = tr[var].to_numpy(float)
        # ▶ 横軸の範囲の指定（なければ None。辞書の get）
        lim = cfg["xlim"].get(var)
        # ▶ 範囲の指定がなければ
        if lim is None:                                           # 6月と同じ：データの範囲を、両側に 10% 広げる（対数軸の変数は、対数で）
            # ▶ データの最小と最大（対数軸の変数は ln で）
            lo, hi = (np.log(x.min()), np.log(x.max())) if logx else (x.min(), x.max())
            # ▶ 両側に 10% 広げる
            lo, hi = lo - 0.1 * (hi - lo), hi + 0.1 * (hi - lo)
            # ▶ その範囲に、等間隔（対数軸の変数は ln で等間隔）に点を並べる
            grid = np.exp(np.linspace(lo, hi, cfg["n_grid"])) if logx else np.linspace(lo, hi, cfg["n_grid"])
        # ▶ 範囲の指定があれば
        else:
            # ▶ 指定の範囲に、点を並べる
            grid = np.exp(np.linspace(np.log(lim[0]), np.log(lim[1]), cfg["n_grid"])) if logx else np.linspace(lim[0], lim[1], cfg["n_grid"])
        # ▶ 固定する値のうち、動かす変数だけを並べた値に置き換えて（{**base, var: grid}）、粘度を予測する
        mu, sd, _, _ = predict(fit, model_inputs(**{**base, var: grid}))
        # ▶ 95% 区間の下限と上限（予測平均 ± 1.96 × 標準偏差 を、exp で粘度に戻す）
        lower, upper = np.exp(mu - 1.96 * sd), np.exp(mu + 1.96 * sd)
        # ▶ 予測平均の線を、赤で描く（ax.plot）
        line = ax.plot(grid, np.exp(mu), color="red", linewidth=2, label="予測平均値", zorder=20)
        # ▶ 95% 区間を、薄い赤で塗る（fill_between は、2 本の線の間を塗る。alpha は透明度）
        fill = ax.fill_between(grid, lower, upper, color="red", alpha=0.25, label="95% 信頼区間", zorder=15)
        # ▶ 区間の下の縁に、細い線を描く
        ax.plot(grid, lower, color="red", linewidth=0.5, alpha=0.6, zorder=16)
        # ▶ 区間の上の縁に、細い線を描く
        ax.plot(grid, upper, color="red", linewidth=0.5, alpha=0.6, zorder=16)
        # ▶ 凡例に使う点と名前のリスト
        hs, labs = [], []
        # ▶ 厚みごとに
        for h in sorted(tr["H"].unique()):
            # ▶ その厚みの行の印
            m = np.isclose(tr["H"].to_numpy(), h)
            # ▶ その厚みの色・形・重なりの順（辞書にない厚みは灰色の ×）
            col, mk, z = DEP92_STYLE.get(float(h), ("gray", "x", 10))
            # ▶ 学習データの点を描く（ax.scatter。s は点の大きさ、edgecolors は縁の色）
            sc = ax.scatter(x[m], tr.loc[m, "eta"], color=col, s=cfg["plot_marker"], marker=mk, alpha=0.8, edgecolors="k", linewidths=0.5, zorder=z)
            # ▶ 凡例用に、点を覚える
            hs.append(sc)
            # ▶ 凡例用に、厚みの名前を覚える
            labs.append(f"{h:g} mm")
        # ▶ 凡例の点の大きさの倍率
        scale = np.sqrt(cfg["legend_marker"] / cfg["plot_marker"])
        # ▶ 凡例を、右上に描く（予測平均・95% 区間・見出し「キャビティ厚み」・厚みごとの点）
        ax.legend([line[0], fill, Patch(color="none", label="")] + hs, ["予測平均値", "95% 信頼区間", "キャビティ厚み"] + labs, loc="upper right", fontsize=cfg["legend_fs"],
                  # ▶ （続き）凡例の枠の不透明度・枠の色・点の大きさ
                  framealpha=1.0, edgecolor="black", markerscale=scale)
        # ▶ 目盛りの向き（内向き）・上と右にも目盛り・数字の大きさ
        ax.tick_params(axis="both", which="both", direction="in", top=True, right=True, labelsize=cfg["tick_fs"], pad=8)
        # ▶ グラフのタイトル（せん断速度の図だけ別のタイトル。\n は改行）
        ax.set_title("ガウス過程回帰による予測モデル\n（せん断速度と粘度の関係）" if var == "gd" else f"樹脂粘度の {name} 依存性", fontsize=cfg["title_fs"], fontweight="bold", pad=15)
        # ▶ 横軸の名前を出す設定なら
        if cfg["show_xlabel"]:
            # ▶ 横軸の名前を書く
            ax.set_xlabel(f"{name} {sym} {unit}", fontsize=cfg["axis_fs"], fontweight="bold", labelpad=15)
        # ▶ 縦軸の名前を出す設定なら
        if cfg["show_ylabel"]:
            # ▶ 縦軸の名前を書く
            ax.set_ylabel("樹脂粘度 (Pa·s)", fontsize=cfg["axis_fs"], fontweight="bold", labelpad=15)
        # ▶ 縦軸を対数軸にする
        ax.set_yscale("log")
        # ▶ 縦軸の範囲の指定があれば
        if cfg["ylim"] is not None:
            # ▶ 縦軸の範囲を決める（* はタプルの中身を、別々の引数として渡す）
            ax.set_ylim(*cfg["ylim"])
        # ▶ 縦軸の目盛りを、10ⁿ の位置に付ける
        ax.yaxis.set_major_locator(ticker.LogLocator(base=10.0, subs=(1.0,)))
        # ▶ 縦軸の目盛りの数字を、10ⁿ の形で書く
        ax.yaxis.set_major_formatter(ticker.LogFormatterSciNotation())
        # ▶ 横軸の範囲の指定があれば
        if lim is not None:
            # ▶ 横軸の範囲を決める
            ax.set_xlim(*lim)
        # ▶ 対数軸の変数なら
        if logx:
            # ▶ 横軸を対数軸にする
            ax.set_xscale("log")
            # ▶ 横軸の目盛りを、10ⁿ の位置に付ける
            ax.xaxis.set_major_locator(ticker.LogLocator(base=10.0, subs=(1.0,)))
            # ▶ 横軸の目盛りの数字を、10ⁿ の形で書く
            ax.xaxis.set_major_formatter(ticker.LogFormatterSciNotation())
            # ▶ 小さい目盛りには数字を書かない
            ax.xaxis.set_minor_formatter(ticker.NullFormatter())
            # ▶ 横軸の目盛りの向き・大きさ
            ax.tick_params(axis="x", which="both", direction="in", top=True, labelsize=cfg["tick_fs"], pad=8)
        # ▶ 大きい目盛りの位置に、実線の格子を描く
        ax.grid(True, which="major", linestyle="-", alpha=0.6, color="gray", zorder=1)
        # ▶ 小さい目盛りの位置に、点線の格子を描く
        ax.grid(True, which="minor", linestyle=":", alpha=0.4, zorder=1)
        # ▶ 図の中の配置を整える
        fig.tight_layout()
        # ▶ 保存するフォルダを作る
        os.makedirs(out_dir, exist_ok=True)
        # ▶ 保存の形式（png など）ごとに
        for fmt in cfg["formats"]:
            # ▶ ファイルの場所（例：依存性グラフ_Shear_Rate.png）
            path = os.path.join(out_dir, f"{stem}_{fname}.{fmt}")
            # ▶ 図を保存する
            fig.savefig(path, bbox_inches="tight", dpi=dpi)
            # ▶ 保存した場所を覚える
            saved.append(path)
        # ▶ 図を表示する設定なら
        if SHOW_FIGS if cfg["show"] is None else cfg["show"]:
            # ▶ 図をノートに表示する
            plt.show()
        # ▶ 図を閉じる
        plt.close(fig)
    # ▶ 保存したファイルの場所のリストを返す
    return saved


# ▶ 関数：予測線を描くための入力の表を作る（lg は ln γ̇ の並び、h は厚み、t は設定温度）
def flow_inputs(sub, lg, h, t):
    # ▶ 関数の説明（docstring）
    """予測線の入力（ln γ̇ = lg の点）。P3_3・Tff は、同じ設定温度・厚みの条件平均を ln γ̇ で内挿する（範囲の外は端の値のまま）"""
    # ▶ 同じ設定温度・厚みの条件ごとに、ln γ̇・ln P3_3・ln Tff の平均をとり、ln γ̇ の順に並べる
    cm = sub.groupby("cond").agg(lg=("ln_gd", "mean"), p=("ln_P33", "mean"), tf=("ln_Tff", "mean")).sort_values("lg")
    # ▶ 小さな関数 ip：ln γ̇ の位置で、その列の値を直線で内挿する（np.interp。範囲の外は端の値のまま）
    ip = lambda c: np.interp(lg, cm["lg"], cm[c])
    # ▶ Tff の ln を内挿する
    ln_tff = ip("tf")
    # ▶ 予測に使う列をそろえた表を返す（P3_3・Tff は内挿した値、設定温度と厚みは一定）
    return pd.DataFrame({"ln_gd": lg, "gd": np.exp(lg), "ln_P33": ip("p"), "ln_Tff": ln_tff, "Tff": np.exp(ln_tff) - 273.15, "Tset": float(t), "H": float(h)})


# ▶ せん断速度 ― 粘度の図の設定の既定値：レイアウト、帯の種類、点の種類、検証の点を描くか、横軸の範囲・延長
FLOW_DEFAULT = dict(layouts=["temperature", "thickness"], band="predict", points="cond", show_test=True, mark_sens=True, x_range=None, extend=1.3,
                    # ▶ （続き）軸の範囲（レイアウトごと）、図の大きさ、解像度
                    xlim={"thickness": None, "temperature": None}, ylim={"thickness": None, "temperature": None}, figsize=(12, 10), dpi=None,
                    # ▶ （続き）文字の大きさ、点の大きさ、線の太さ、タイトルを出すか
                    font={"axis": 24, "legend": 24, "title": 24, "tick": 28}, marker=300, marker_shot=70, line=2.0, show_title=True,
                    # ▶ （続き）軸名を出すか、保存の形式、線の点の数、表示するか
                    show_xlabel=False, show_ylabel=False, formats=["png"], n_grid=120, show=None)


# ▶ 総合図の、加熱シリンダ温度ごとの線・帯の色（低い温度から：赤・青・緑）
T_CURVE_COLORS = ["#d62728", "#1f77b4", "#2ca02c", "#9467bd", "#8c564b"]       # 総合図の、加熱シリンダ温度ごとの線・帯の色（低い温度から：赤・青・緑）。点の色は厚みごと（赤・青・黄・緑）なので、色は同じでも、点は形で見分ける


# ▶ 関数：6月のノートのセル9-2 と同じ仕様で、全厚み共通のせん断速度 ― 粘度の図を作る
def flow_plot_common(fit, mat, pool, train_conds, out_dir, stem, tr=None, **kw):
    # ▶ 関数の説明（docstring）
    """6月のノートのセル9-2 と同じ仕様（図の大きさ〔既定 12 × 10 インチ。6月は 15 × 10〕、文字の大きさ、点の大きさ〔既定 300〕、軸の目盛り〔両軸とも 10 の累乗〕、凡例の並べ方）で、
    全キャビティ厚みの**学習データ**の点（色・形は厚み）を重ね、**厚みによらない共通の平均関数と 95% 区間**を描く。検証（3data）の点は描かない。
    曲線は、せん断速度だけを動かし、ほかの入力を固定したもの：P3_3 は学習データ（tr）の中央値、Tff は、その設定温度の学習データの中央値、Tset はその温度。
    曲線の範囲は、その設定温度の学習データの範囲を extend 倍に延ばしたもの（x_range があればそれ）。軸の範囲は、common_xlim・common_ylim（None なら、データを含む 10 の累乗の範囲）。
    2 種類の図を保存する：①加熱シリンダ温度ごとに 1 枚（…_全厚み共通_<温度>C）、②全温度を 1 枚に重ね、温度ごとに線・帯の色を変えた総合図（…_全厚み共通_全温度）。
    保存したファイルの一覧を返す"""
    # ▶ グラフのライブラリを読み込む
    import matplotlib.pyplot as plt
    # ▶ 目盛りの道具を読み込む
    import matplotlib.ticker as ticker
    # ▶ 凡例に使う線の部品を読み込む
    from matplotlib.lines import Line2D
    # ▶ 凡例に使う四角の部品を読み込む
    from matplotlib.patches import Patch
    # ▶ 図の設定：既定値に、この図だけの設定を足し、渡された設定（kw）で上書きする（{**a, **b} は 2 つの辞書を合わせる書き方）
    cfg = {**FLOW_DEFAULT, "common_xlim": None, "common_ylim": None, "legend_ncol": 1, **kw}
    # ▶ 解像度（指定がなければ FIG_DPI）
    dpi = cfg["dpi"] or FIG_DPI
    # ▶ 予測線を作るデータ（指定がなければ pool）
    tr = pool if tr is None else tr
    # ▶ 文字の大きさの設定と、保存したファイルのリスト
    ft, saved = cfg["font"], []
    # ▶ 帯の名前（予測区間か信頼区間か）
    band_name = "95% 予測区間" if cfg["band"] == "predict" else "95% 信頼区間"
    # ▶ P3_3 を固定する値（ln の中央値を MPa に戻した値）
    p_med = float(np.exp(tr["ln_P33"].median()) / 1e6)
    # ▶ 設定温度の水準（小さい順）
    levels = sorted(tr["Tset"].unique())
    # ▶ 設定温度ごとの線の色の辞書
    tcol = {float(t): T_CURVE_COLORS[i % len(T_CURVE_COLORS)] for i, t in enumerate(levels)}
    # ▶ 温度ごとの予測線を入れる辞書
    curves = {}
    # ▶ 設定温度ごとに（sub はその温度の行）
    for t, sub in tr.groupby("Tset"):
        # ▶ 条件ごとの ln γ̇ の平均
        lgc = sub.groupby("cond")["ln_gd"].mean()
        # ▶ 予測線の範囲：x_range の指定があればそれ、なければデータの範囲を両側に extend 倍だけ延ばす
        lo, hi = (np.log(cfg["x_range"][0]), np.log(cfg["x_range"][1])) if cfg["x_range"] else (lgc.min() - np.log(cfg["extend"]), lgc.max() + np.log(cfg["extend"]))
        # ▶ その範囲に、ln γ̇ を等間隔に n_grid 点並べる
        lg = np.linspace(lo, hi, cfg["n_grid"])
        # ▶ その温度の Tff を固定する値（ln の中央値を ℃ に戻した値）
        tff_med = float(np.exp(sub["ln_Tff"].median()) - 273.15)
        # ▶ P3_3 と Tff を固定し、せん断速度だけを動かして、粘度を予測する
        mu, sd, sdf, _ = predict(fit, model_inputs(np.exp(lg), p_med, tff_med, float(t)))
        # ▶ 温度ごとに、せん断速度・予測平均・帯に使う標準偏差（予測区間ならノイズ込み、信頼区間なら曲線だけ）・固定した Tff を覚える
        curves[float(t)] = (np.exp(lg), mu, sd if cfg["band"] == "predict" else sdf, tff_med)

    # ▶ 関数の中の関数：学習データの点の座標（せん断速度, 粘度）と、点の大きさを返す
    def points_of(sub):
        # ▶ 関数の説明（docstring）
        """学習データの点（px, py, 面積）。cfg["points"] が "cond" なら条件平均、"shot" ならショットごと"""
        # ▶ 条件平均の点を描く設定なら
        if cfg["points"] == "cond":
            # ▶ 条件ごとに、ln γ̇ と ln η を平均する
            a = sub.groupby("cond").agg(lg=("ln_gd", "mean"), le=("ln_eta", "mean")).reset_index()
            # ▶ 平均を元の値に戻して（exp）、点の大きさと一緒に返す
            return np.exp(a["lg"].to_numpy()), np.exp(a["le"].to_numpy()), cfg["marker"]
        # ▶ そうでなければ、ショットごとの値と、ショット用の点の大きさを返す
        return sub["gd"].to_numpy(), sub["eta"].to_numpy(), cfg["marker_shot"]

    # ▶ 関数の中の関数：厚みごとに色・形を変えて点を描き、凡例用の記号のリストを返す（xs・ys には、軸の範囲を決めるための値を足していく）
    def thickness_points(ax, sub, xs, ys, z=14):
        # ▶ 関数の説明（docstring）
        """厚みごとに色・形を変えて、点を描く（6月のセル9-2 と同じ：alpha 0.8、黒い縁）。凡例用の記号の一覧を返す"""
        # ▶ 凡例用の記号のリスト
        hs = []
        # ▶ 厚みごとに
        for h in sorted(sub["H"].unique()):
            # ▶ その厚みの点の座標と大きさ
            px, py, ms = points_of(sub[np.isclose(sub["H"], h)])
            # ▶ その厚みの色と形（辞書にない厚みは、灰色の ×）
            col, mk = H_COLORS.get(float(h), "gray"), H_MARKS.get(float(h), "x")
            # ▶ 軸の範囲を決めるための x の値に足す
            xs += list(px)
            # ▶ 軸の範囲を決めるための y の値に足す
            ys += list(py)
            # ▶ 点を描く（alpha は透明度、edgecolors は縁の色、zorder は重なりの順）
            ax.scatter(px, py, s=ms, color=col, marker=mk, alpha=0.8, edgecolors="k", linewidths=0.5, zorder=z)
            # ▶ 凡例用の記号（線なしの点）を作って、リストに足す
            hs.append(Line2D([], [], color=col, ls="none", marker=mk, markersize=np.sqrt(cfg["marker"]), markeredgecolor="k", markeredgewidth=0.5, label=f"{h:g} mm"))
        # ▶ 凡例用の記号のリストを返す
        return hs

    # ▶ 関数の中の関数：軸の範囲を、10 の累乗の切りのよい値で決める（user に指定があれば、それを使う）
    def decade_lim(v, user):
        # ▶ 指定があれば
        if user is not None:
            # ▶ その範囲を返す
            return user
        # ▶ 最小の値の下の 10ⁿ（floor は切り捨て）から、最大の値の上の 10ⁿ（ceil は切り上げ）までを返す
        return 10.0 ** np.floor(np.log10(min(v))), 10.0 ** np.ceil(np.log10(max(v)))

    # ▶ 関数の中の関数：図を保存する（高解像度の版も）
    def save(fg, base):
        # ▶ 保存するフォルダを作る
        os.makedirs(out_dir, exist_ok=True)
        # ▶ 保存の形式ごとに
        for fmt in cfg["formats"]:
            # ▶ ファイルの場所
            path = os.path.join(out_dir, f"{base}.{fmt}")
            # ▶ 図を保存する
            fg.savefig(path, bbox_inches="tight", dpi=dpi)
            # ▶ 保存した場所を覚える
            saved.append(path)

    # ▶ 関数の中の関数：設定温度 t の予測平均の線と 95% 区間を、横軸の範囲 xl の端まで描く
    def draw_curve(ax, t, tff_med, col, lw, xl):
        # ▶ 関数の説明（docstring）
        """平均関数と 95% 区間を、グラフエリアの端（横軸の範囲 xl）まで描く（6月のセル9-2 と同じ。軸の範囲を変えても、常に端まで描く）"""
        # ▶ 横軸の範囲に、ln γ̇ を等間隔に並べる（400 点以上）
        lg = np.linspace(np.log(xl[0]), np.log(xl[1]), max(cfg["n_grid"], 400))
        # ▶ 粘度を予測する（P3_3 と Tff は固定）
        mu, sd, sdf, _ = predict(fit, model_inputs(np.exp(lg), p_med, tff_med, float(t)))
        # ▶ 帯に使う標準偏差と、せん断速度の値
        s, x = (sd if cfg["band"] == "predict" else sdf), np.exp(lg)
        # ▶ 95% 区間を塗る（予測平均 ± 1.96 × 標準偏差 を、exp で粘度に戻す）
        ax.fill_between(x, np.exp(mu - 1.96 * s), np.exp(mu + 1.96 * s), color=col, alpha=0.25, zorder=15)
        # ▶ 区間の下の縁に、細い線を描く
        ax.plot(x, np.exp(mu - 1.96 * s), color=col, lw=0.5, alpha=0.6, zorder=16)
        # ▶ 区間の上の縁に、細い線を描く
        ax.plot(x, np.exp(mu + 1.96 * s), color=col, lw=0.5, alpha=0.6, zorder=16)
        # ▶ 予測平均の線を描く
        ax.plot(x, np.exp(mu), color=col, lw=lw, zorder=20)

    # ▶ 関数の中の関数：軸・目盛り・凡例・タイトルを整えて保存する（draw が指定されていれば、軸の範囲を決めた後に、それで曲線を描く）
    def finish(fig, ax, xs, ys, base, title, handles, separate_legend=False, handles_en=None, draw=None):
        # ▶ 関数の説明（docstring）
        """軸・目盛り・凡例・タイトルを整えて保存する。separate_legend が True なら、グラフには凡例とタイトルを付けず、凡例だけを別の画像（base + "_凡例"）に保存する。
        draw：軸の範囲を決めた後で、曲線と帯をその範囲いっぱいに描く関数 draw(ax, 横軸の範囲)"""
        # ▶ 目盛りの向き（内向き）・上と右にも目盛り・数字の大きさ
        ax.tick_params(axis="both", which="both", direction="in", top=True, right=True, labelsize=ft["tick"], pad=8)
        # ▶ 横軸を対数軸にする
        ax.set_xscale("log")
        # ▶ 縦軸を対数軸にする
        ax.set_yscale("log")
        # ▶ 横軸の範囲を、10 の累乗の切りのよい値で決める
        ax.set_xlim(*decade_lim(xs, cfg["common_xlim"]))
        # ▶ 縦軸の範囲を、10 の累乗の切りのよい値で決める
        ax.set_ylim(*decade_lim(ys, cfg["common_ylim"]))
        # ▶ 曲線を描く関数が渡されていれば
        if draw is not None:
            # ▶ 今の軸の範囲を覚える
            xl, yl = ax.get_xlim(), ax.get_ylim()
            # ▶ 曲線を、横軸の端まで描く
            draw(ax, xl)
            # ▶ 描いたことで軸の範囲が変わらないよう、元の範囲に戻す
            ax.set_xlim(*xl)
            # ▶ （同じく縦軸）
            ax.set_ylim(*yl)
        # ▶ 横軸と縦軸のそれぞれについて
        for axis in (ax.xaxis, ax.yaxis):                        # 6月のセル9-2 と同じ：両軸とも 10 の累乗の目盛り
            # ▶ 目盛りを 10ⁿ の位置に付ける
            axis.set_major_locator(ticker.LogLocator(base=10.0, subs=(1.0,)))
            # ▶ 目盛りの数字を 10ⁿ の形で書く
            axis.set_major_formatter(ticker.LogFormatterSciNotation())
            # ▶ 小さい目盛りには数字を書かない
            axis.set_minor_formatter(ticker.NullFormatter())
        # ▶ 大きい目盛りの位置に、実線の格子を描く
        ax.grid(True, which="major", linestyle="-", alpha=0.6, color="gray", zorder=1)
        # ▶ 小さい目盛りの位置に、点線の格子を描く
        ax.grid(True, which="minor", linestyle=":", alpha=0.4, zorder=1)
        # ▶ 凡例を別の画像にしない設定なら（not は「〜でない」）
        if not separate_legend:
            # ▶ 凡例を、左下に描く
            ax.legend(handles=handles, loc="lower left", fontsize=ft["legend"], framealpha=1.0, edgecolor="black", ncol=cfg["legend_ncol"])
            # ▶ タイトルを出す設定で、タイトルがあれば
            if cfg["show_title"] and title:
                # ▶ タイトルを書く
                ax.set_title(title, fontsize=ft["title"], fontweight="bold", pad=15)
        # ▶ 横軸の名前を出す設定なら
        if cfg["show_xlabel"]:
            # ▶ 横軸の名前を書く（$…$ の中は数式の書き方。\dot{\gamma} は γ̇）
            ax.set_xlabel(r"せん断速度 $\dot{\gamma}$ (1/s)", fontsize=ft["axis"], fontweight="bold", labelpad=15)
        # ▶ 縦軸の名前を出す設定なら
        if cfg["show_ylabel"]:
            # ▶ 縦軸の名前を書く
            ax.set_ylabel(r"樹脂粘度 $\eta$ (Pa·s)", fontsize=ft["axis"], fontweight="bold", labelpad=15)
        # ▶ 図の中の配置を整える
        fig.tight_layout()
        # ▶ 図を保存する
        save(fig, base)
        # ▶ 図を表示する設定なら
        if SHOW_FIGS if cfg["show"] is None else cfg["show"]:
            # ▶ 図をノートに表示する
            plt.show()
        # ▶ 図を閉じる
        plt.close(fig)
        # ▶ 凡例を別の画像にする設定なら
        if separate_legend:                                       # 凡例だけの画像
            # ▶ 凡例だけを描く図を作る（6 × 6 インチ）
            lf = plt.figure(figsize=(6, 6))
            # ▶ 凡例を、図の中央に描く
            lf.legend(handles=handles, loc="center", fontsize=ft["legend"], framealpha=1.0, edgecolor="black", ncol=cfg["legend_ncol"])
            # ▶ 「…_凡例」の名前で保存する
            save(lf, base + "_凡例")
            # ▶ 図を閉じる
            plt.close(lf)
            # ▶ 英語の凡例も渡されていれば
            if handles_en is not None:                            # 凡例の日本語を英語にした版（…_凡例_English）
                # ▶ 英語の凡例の図を作る
                lf = plt.figure(figsize=(6, 6))
                # ▶ 英語の凡例を描く
                lf.legend(handles=handles_en, loc="center", fontsize=ft["legend"], framealpha=1.0, edgecolor="black", ncol=cfg["legend_ncol"])
                # ▶ 「…_凡例_English」の名前で保存する
                save(lf, base + "_凡例_English")
                # ▶ 図を閉じる
                plt.close(lf)

    # ① 加熱シリンダ温度ごと（曲線は赤。6月のセル9-2 と同じ）
    # ▶ 設定温度ごとに、全厚み共通の図を 1 枚ずつ作る（curves の値を 4 つの変数で受け取る）
    for t, (x, mu, s, tff_med) in curves.items():
        # ▶ 図とグラフを作る
        fig, ax = plt.subplots(figsize=cfg["figsize"])
        # ▶ 軸の範囲を決めるための値（予測線のせん断速度と粘度）
        xs, ys = list(x), list(np.exp(mu))                       # 軸の範囲を決めるための値（データの範囲の曲線と点）。曲線そのものは、finish が軸の端まで描く
        # ▶ その温度の学習データの点を、厚みごとに描く
        hs = thickness_points(ax, tr[tr["Tset"] == t], xs, ys)
        # ▶ 凡例の先頭：予測平均の線、95% 区間の帯、見出し「キャビティ厚み」（Patch(color="none") は見出し用の空の記号）
        head = [Line2D([], [], color="red", lw=cfg["line"], label="予測平均値"), Patch(facecolor="red", alpha=0.25, label=band_name), Patch(color="none", label="キャビティ厚み")]
        # ▶ グラフの右上に、固定した値（P3_3 と Tff）を書いた箱を置く（transform=ax.transAxes：位置をグラフの割合〔0〜1〕で指定）
        ax.text(0.98, 0.97, f"固定した値\nP3_3：{p_med:.2f} MPa\nTff：{tff_med:.1f} ℃", transform=ax.transAxes, ha="right", va="top", fontsize=ft["legend"] * 0.75,
                # ▶ （続き）箱の形（角の丸い白い箱、黒い縁）
                bbox=dict(boxstyle="round", facecolor="white", edgecolor="black"), zorder=30)
        # ▶ 軸・凡例を整えて保存する（ファイル名：…_全厚み共通_<温度>C）
        finish(fig, ax, xs, ys, f"{stem}_全厚み共通_{int(t)}C",
               # ▶ （続き）タイトル（材料・設定温度・モデル名）と、凡例の記号
               f"{MAT_NAMES[mat]}　せん断速度と粘度の関係（設定温度 {int(t)} ℃）\n{model_title(fit['key'], fit.get('mode'))}", head + hs,
               # ▶ （続き）曲線の描き方（lambda で、温度と Tff を固定した描画の関数を渡す。t=t は、その時の値を覚えておく書き方）
               draw=lambda a, xl, t=t, tm=tff_med: draw_curve(a, t, tm, "red", cfg["line"], xl))

    # ② 全温度を 1 枚に重ねた総合図（線・帯の色は温度、点の色・形は厚み）
    # ▶ 全温度をまとめた総合図を作る
    fig, ax = plt.subplots(figsize=cfg["figsize"])
    # ▶ 軸の範囲を決めるための値のリスト
    xs, ys = [], []
    # ▶ 温度ごとに
    for t, (x, mu, s, tff_med) in curves.items():
        # ▶ 予測線のせん断速度を足す
        xs += list(x)
        # ▶ 予測線の粘度を足す
        ys += list(np.exp(mu))
    # ▶ 全部の学習データの点を、厚みごとに描く
    hs = thickness_points(ax, tr, xs, ys)
    # ▶ 凡例：温度ごとの予測平均の線
    head = [Line2D([], [], color=tcol[t], lw=cfg["line"], label=f"予測平均値_{int(t)} ℃") for t in curves]
    # ▶ 凡例：温度ごとの 95% 区間の帯
    head += [Patch(facecolor=tcol[t], alpha=0.25, label=f"{band_name}_{int(t)} ℃") for t in curves]
    # ▶ 凡例：見出し「キャビティ厚み」
    head += [Patch(color="none", label="キャビティ厚み")]
    # ▶ 凡例の後ろに付ける注記：固定した値（P3_3）
    note = [Patch(color="none", label="固定した値（学習データの中央値）"), Patch(color="none", label=f"　P3_3：{p_med:.2f} MPa")]
    # ▶ （続き）温度ごとに固定した Tff
    note += [Patch(color="none", label=f"　Tff（{int(t)} ℃）：{v[3]:.1f} ℃") for t, v in curves.items()]
    # ▶ 帯の名前の英語版
    band_en = "95% prediction interval" if cfg["band"] == "predict" else "95% confidence interval"           # 凡例の英語版（hs の「0.5 mm」などは、そのまま）
    # ▶ 英語の凡例：温度ごとの予測平均の線
    head_en = [Line2D([], [], color=tcol[t], lw=cfg["line"], label=f"Predicted mean_{int(t)} °C") for t in curves]
    # ▶ 英語の凡例：温度ごとの 95% 区間の帯
    head_en += [Patch(facecolor=tcol[t], alpha=0.25, label=f"{band_en}_{int(t)} °C") for t in curves]
    # ▶ 英語の凡例：見出し「Cavity thickness」
    head_en += [Patch(color="none", label="Cavity thickness")]
    # ▶ 英語の注記：固定した値（P3_3）
    note_en = [Patch(color="none", label="Fixed values (median of training data)"), Patch(color="none", label=f"  P3_3: {p_med:.2f} MPa")]
    # ▶ （続き）温度ごとに固定した Tff
    note_en += [Patch(color="none", label=f"  Tff ({int(t)} °C): {v[3]:.1f} °C") for t, v in curves.items()]
    # ▶ 総合図を保存する（タイトルなし。凡例は別の画像に、日本語版と英語版の 2 つ）
    finish(fig, ax, xs, ys, f"{stem}_全厚み共通_全温度", None, head + hs + note, separate_legend=True, handles_en=head_en + hs + note_en,
           # ▶ （続き）曲線の描き方：温度ごとに、その温度の色で線と帯を描く
           draw=lambda a, xl: [draw_curve(a, t, v[3], tcol[t], cfg["line"], xl) for t, v in curves.items()])
    # ▶ 保存したファイルの場所のリストを返す
    return saved


# ▶ 関数：せん断速度 ― 粘度の学習結果グラフ（厚みごと・設定温度ごと）を作る
def flow_plot(fit, mat, pool, train_conds, out_dir, stem, **kw):
    # ▶ 関数の説明（docstring）
    """せん断速度 ― 粘度の学習結果グラフ（6月のノートのセル9-2 と同じ見せ方。本体の nb_common.hr_flow を、このデータ用に直したもの）。
    "temperature"：設定温度ごとに 1 枚（厚みごとの線）／"thickness"：厚みごとに 1 枚（設定温度ごとの線）。
    塗りつぶしの点は学習（5data）、白抜きは検証（3data）、× は問題データ一覧の「感度確認」。保存したファイルの一覧を返す"""
    # ▶ グラフのライブラリを読み込む
    import matplotlib.pyplot as plt
    # ▶ 目盛りの道具を読み込む
    import matplotlib.ticker as ticker
    # ▶ 凡例に使う線の部品を読み込む
    from matplotlib.lines import Line2D
    # ▶ 凡例に使う四角の部品を読み込む
    from matplotlib.patches import Patch
    # ▶ 図の設定：既定値を、渡された設定で上書きする
    cfg = {**FLOW_DEFAULT, **kw}
    # ▶ 解像度（指定がなければ FIG_DPI）
    dpi = cfg["dpi"] or FIG_DPI
    # ▶ 設定温度ごとの色と形の辞書
    ts = t_style(pool["Tset"].unique())
    # ▶ 厚み・温度ごとの予測線を入れる辞書
    curves = {}
    # ▶ 厚みと設定温度の組ごとに（sub はその組の行）
    for (h, t), sub in pool.groupby(["H", "Tset"]):
        # ▶ 条件ごとの ln γ̇ の平均
        lgc = sub.groupby("cond")["ln_gd"].mean()
        # ▶ 条件が 2 つより少なければ
        if len(lgc) < 2:
            # ▶ とばす
            continue
        # ▶ 予測線の範囲：x_range の指定があればそれ、なければデータの範囲を両側に extend 倍だけ延ばす
        lo, hi = (np.log(cfg["x_range"][0]), np.log(cfg["x_range"][1])) if cfg["x_range"] else (lgc.min() - np.log(cfg["extend"]), lgc.max() + np.log(cfg["extend"]))
        # ▶ その範囲に、ln γ̇ を等間隔に並べる
        lg = np.linspace(lo, hi, cfg["n_grid"])
        # ▶ 粘度を予測する（P3_3・Tff は、同じ厚み・温度の条件平均を内挿した値）
        mu, sd, sdf, _ = predict(fit, flow_inputs(sub, lg, h, t))
        # ▶ (厚み, 温度) ごとに、せん断速度・予測平均・帯の標準偏差を覚える
        curves[(float(h), float(t))] = (np.exp(lg), mu, sd if cfg["band"] == "predict" else sdf)
    # ▶ 保存したファイルのリストと、文字の大きさの設定
    saved, ft = [], cfg["font"]
    # ▶ 帯の名前
    band_name = "95% 予測区間" if cfg["band"] == "predict" else "95% 信頼区間"
    # ▶ レイアウト（"thickness"＝厚みごと、"temperature"＝設定温度ごと）ごとに
    for layout_ in cfg["layouts"]:
        # ▶ 厚みごとの図なら True
        by_h = layout_ == "thickness"
        # ▶ 図を分ける値（厚みごとの図なら厚み、そうでなければ温度）ごとに、1 枚ずつ作る（{…} は集合で、重複がなくなる）
        for gv in sorted({k[0] if by_h else k[1] for k in curves}):
            # ▶ 図とグラフを作る
            fig, ax = plt.subplots(figsize=cfg["figsize"])
            # ▶ 凡例の系列、検証の点や感度確認の点があったかの印、軸の範囲を決めるための値、後で描く線のリスト
            series, any_test, any_sens, xs, ys, pend = [], False, False, [], [], []
            # ▶ 図の中の系列（厚みごとの図なら温度、そうでなければ厚み）ごとに
            for sv in sorted({(k[1] if by_h else k[0]) for k in curves if (k[0] if by_h else k[1]) == gv}):
                # ▶ 厚みと温度
                h, t = (gv, sv) if by_h else (sv, gv)
                # ▶ 系列の色と形（厚みごとの図なら温度の色、そうでなければ厚みの色）
                col, mk = ts[t] if by_h else (H_COLORS.get(h, "gray"), H_MARKS.get(h, "x"))
                # ▶ その系列の予測線（データの範囲）
                x, mu, s = curves[(float(h), float(t))]             # データの範囲の曲線。軸の範囲を決めるのに使う（曲線そのものは、軸の範囲が決まってから、端まで描く）
                # ▶ その厚み・温度のデータの行
                sub = pool[np.isclose(pool["H"], h) & (pool["Tset"] == t)]
                # ▶ 線は、軸の範囲が決まった後で描くので、必要な情報を覚えておく
                pend.append((h, t, col, sub))
                # ▶ 条件平均の点を描く設定なら
                if cfg["points"] == "cond":
                    # ▶ 条件ごとに、ln γ̇ と ln η の平均と、感度確認の印をまとめる
                    a = sub.groupby("cond").agg(lg=("ln_gd", "mean"), le=("ln_eta", "mean"), sens=("is_sens", "max")).reset_index()
                    # ▶ 点の座標（平均を元の値に戻す）、感度確認の印、点の大きさ
                    px, py, sens, ms = np.exp(a["lg"].to_numpy()), np.exp(a["le"].to_numpy()), a["sens"].to_numpy().astype(bool), cfg["marker"]
                    # ▶ 学習データの条件なら True（isin は「リストに入っているか」）
                    trm = a["cond"].isin(train_conds).to_numpy()
                # ▶ ショットごとの点を描く設定なら
                else:
                    # ▶ ショットごとの座標、感度確認の印、点の大きさ
                    px, py, sens, ms = sub["gd"].to_numpy(), sub["eta"].to_numpy(), sub["is_sens"].to_numpy().astype(bool), cfg["marker_shot"]
                    # ▶ 学習データの条件なら True
                    trm = sub["cond"].isin(train_conds).to_numpy()
                # ▶ 軸の範囲を決めるための x の値に足す
                xs += list(x) + list(px)
                # ▶ 軸の範囲を決めるための y の値に足す
                ys += list(np.exp(mu)) + list(py)
                # ▶ 学習データの点を描く（塗りつぶし）
                ax.scatter(px[trm], py[trm], s=ms, color=col, marker=mk, alpha=0.85, edgecolors="k", linewidths=0.6, zorder=14)
                # ▶ 検証の点を描く設定で、検証の点があれば（~trm は学習でない点）
                if cfg["show_test"] and (~trm).any():
                    # ▶ 検証の点を白抜きで描く
                    ax.scatter(px[~trm], py[~trm], s=ms, facecolors="white", edgecolors=col, marker=mk, linewidths=2.5, zorder=14)
                    # ▶ 検証の点があった印
                    any_test = True
                # ▶ 感度確認の点に印を付ける設定で、その点があれば
                if cfg["mark_sens"] and sens.any():
                    # ▶ 感度確認の点に、黒い × を重ねる
                    ax.scatter(px[sens], py[sens], s=ms * 1.3, color="k", marker="x", linewidths=3, zorder=25)
                    # ▶ 感度確認の点があった印
                    any_sens = True
                # ▶ 凡例用に、系列を覚える
                series.append((sv, col, mk))
            # ▶ 凡例の見出し（厚みごとの図なら「設定温度」、そうでなければ「キャビティ厚み」）
            head = "キャビティ厚み" if not by_h else "設定温度"
            # ▶ 凡例：帯と見出し
            handles = [Patch(facecolor="gray", alpha=0.3, label=band_name), Patch(color="none", label=head)]
            # ▶ 凡例：系列ごとの線と点
            handles += [Line2D([], [], color=c, lw=cfg["line"], marker=m, markersize=np.sqrt(cfg["marker"]) * 0.8, markeredgecolor="k", markeredgewidth=0.6,
                               # ▶ （続き）系列の名前（「0.5 mm」や「190 ℃」）
                               label=(f"{sv:g} mm" if not by_h else f"{int(sv)} ℃")) for sv, c, m in series]
            # ▶ 検証の点があれば
            if any_test:
                # ▶ 凡例に「検証（3data。白抜き）」を足す
                handles.append(Line2D([], [], color="gray", ls="none", marker="o", markersize=np.sqrt(cfg["marker"]) * 0.8, markerfacecolor="white", markeredgewidth=2.5, label="検証（3data。白抜き）"))
            # ▶ 感度確認の点があれば
            if any_sens:
                # ▶ 凡例に「感度確認の条件」を足す
                handles.append(Line2D([], [], color="k", ls="none", marker="x", markersize=np.sqrt(cfg["marker"]) * 0.8, markeredgewidth=3, label="感度確認の条件"))
            # ▶ 帯を描かない設定なら
            if cfg["band"] not in ("predict", "latent"):
                # ▶ 凡例の最初（帯）を除く（[1:] は 2 番目から最後まで）
                handles = handles[1:]
            # ▶ 凡例を右上に描く
            ax.legend(handles=handles, loc="upper right", fontsize=ft["legend"], framealpha=1.0, edgecolor="black")
            # ▶ 目盛りの向き・大きさ
            ax.tick_params(axis="both", which="both", direction="in", top=True, right=True, labelsize=ft["tick"], pad=8)
            # ▶ 横軸を対数軸にする
            ax.set_xscale("log")
            # ▶ 縦軸を対数軸にする
            ax.set_yscale("log")
            # ▶ 横軸の範囲の指定（レイアウトごとの辞書なら、そのレイアウトの値）
            xl = cfg["xlim"][layout_] if isinstance(cfg["xlim"], dict) else cfg["xlim"]
            # ▶ 縦軸の範囲の指定
            yl = cfg["ylim"][layout_] if isinstance(cfg["ylim"], dict) else cfg["ylim"]
            # ▶ 横軸の範囲：指定がなければ、データの範囲を 1.4 倍広げる
            ax.set_xlim(*((min(xs) / 1.4, max(xs) * 1.4) if xl is None else xl))
            # ▶ 縦軸の範囲：指定がなければ、データの範囲を 1.5 倍広げる
            ax.set_ylim(*((min(ys) / 1.5, max(ys) * 1.5) if yl is None else yl))
            # ▶ 決めた軸の範囲を覚える
            xlim_now, ylim_now = ax.get_xlim(), ax.get_ylim()
            # ▶ 横軸の範囲の端から端まで、ln γ̇ を等間隔に並べる（予測線と帯を、グラフの端まで描くため）
            lg_full = np.linspace(np.log(xlim_now[0]), np.log(xlim_now[1]), max(cfg["n_grid"], 400))       # 平均関数と 95% 区間を、グラフエリアの端まで描く（軸の範囲を変えても追従）
            # ▶ 覚えておいた系列ごとに
            for h, t, col, sub in pend:
                # ▶ 粘度を予測する
                mu_f, sd_f, sdf_f, _ = predict(fit, flow_inputs(sub, lg_full, h, t))
                # ▶ 帯に使う標準偏差と、せん断速度の値
                s_f, x_f = (sd_f if cfg["band"] == "predict" else sdf_f), np.exp(lg_full)
                # ▶ 帯を描く設定なら
                if cfg["band"] in ("predict", "latent"):
                    # ▶ 95% 区間を塗る
                    ax.fill_between(x_f, np.exp(mu_f - 1.96 * s_f), np.exp(mu_f + 1.96 * s_f), color=col, alpha=0.22, zorder=15)
                # ▶ 予測平均の線を描く
                ax.plot(x_f, np.exp(mu_f), color=col, lw=cfg["line"], zorder=20)
            # ▶ 軸の範囲を、決めた値に戻す
            ax.set_xlim(*xlim_now)
            # ▶ （同じく縦軸）
            ax.set_ylim(*ylim_now)
            # ▶ 横軸と縦軸のそれぞれについて（auto は、範囲を自動で決めたか）
            for axis, auto in ((ax.xaxis, xl is None), (ax.yaxis, yl is None)):
                # ▶ その軸の範囲（下限と上限）
                lo_, hi_ = axis.get_view_interval()
                # ▶ 範囲を自動で決めた軸で、範囲が 2.5 桁以下なら、細かい目盛り（1・2・5 × 10ⁿ）にする
                fine = auto and np.log10(hi_ / lo_) <= 2.5                 # 範囲を自動で決めた軸で、2.5 桁以下なら 1・2・5 × 10ⁿ の目盛り（log_ticks と同じ基準）
                # ▶ 細かい目盛りにするなら
                if fine:
                    # ▶ 1・2・5 × 10ⁿ の位置に目盛りを付ける
                    axis.set_major_locator(ticker.LogLocator(base=10.0, subs=(1.0, 2.0, 5.0)))
                    # ▶ 目盛りの数字を、普通の数で書く
                    axis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{v:g}"))
                    # ▶ 図を描いて、それでも目盛りの数字が重なるなら、細かい目盛りをやめる
                    fine = not tick_labels_overlap(fig, axis)            # それでも目盛りの数字が重なるなら、10ⁿ だけにする
                # ▶ 範囲が広い軸、範囲を指定した軸、数字が重なった軸なら
                if not fine:
                    # ▶ 10ⁿ の位置だけに目盛りを付ける
                    axis.set_major_locator(ticker.LogLocator(base=10.0, subs=(1.0,)))
                    # ▶ 目盛りの数字を、10ⁿ の形で書く
                    axis.set_major_formatter(ticker.LogFormatterSciNotation())
                # ▶ 小さい目盛りには数字を書かない
                axis.set_minor_formatter(ticker.NullFormatter())
            # ▶ 大きい目盛りの位置に、実線の格子を描く
            ax.grid(True, which="major", linestyle="-", alpha=0.6, color="gray", zorder=1)
            # ▶ 小さい目盛りの位置に、点線の格子を描く
            ax.grid(True, which="minor", linestyle=":", alpha=0.4, zorder=1)
            # ▶ タイトルに入れる、図の区分の文字（例：厚み 1.0 mm）
            gtxt = f"厚み {gv:g} mm" if by_h else f"設定温度 {int(gv)} ℃"
            # ▶ タイトルを出す設定なら
            if cfg["show_title"]:
                # ▶ タイトルを書く（材料・区分・モデル名）
                ax.set_title(f"{MAT_NAMES[mat]}　せん断速度と粘度の関係（{gtxt}）\n{model_title(fit['key'], fit.get('mode'))}", fontsize=ft["title"], pad=15)
            # ▶ 横軸の名前を出す設定なら
            if cfg["show_xlabel"]:
                # ▶ 横軸の名前を書く
                ax.set_xlabel(r"せん断速度 $\dot{\gamma}$ (1/s)", fontsize=ft["axis"], labelpad=15)
            # ▶ 縦軸の名前を出す設定なら
            if cfg["show_ylabel"]:
                # ▶ 縦軸の名前を書く
                ax.set_ylabel(r"樹脂粘度 $\eta$ (Pa·s)", fontsize=ft["axis"], labelpad=15)
            # ▶ 図の中の配置を整える
            fig.tight_layout()
            # ▶ ファイル名（例：せん断速度_粘度_厚みごと_1mm、…_設定温度ごと_210C）
            base = f"{stem}_{'厚みごと' if by_h else '設定温度ごと'}_{(f'{gv:g}mm') if by_h else (str(int(gv)) + 'C')}"
            # ▶ 保存するフォルダを作る
            os.makedirs(out_dir, exist_ok=True)
            # ▶ 保存の形式ごとに
            for fmt in cfg["formats"]:
                # ▶ ファイルの場所
                path = os.path.join(out_dir, f"{base}.{fmt}")
                # ▶ 図を保存する
                fig.savefig(path, bbox_inches="tight", dpi=dpi)
                # ▶ 保存した場所を覚える
                saved.append(path)
            # ▶ 図を表示する設定なら
            if SHOW_FIGS if cfg["show"] is None else cfg["show"]:
                # ▶ 図をノートに表示する
                plt.show()
            # ▶ 図を閉じる
            plt.close(fig)
    # ▶ 保存したファイルの場所のリストを返す
    return saved

### 比べるモデルと、結果の置き場所

設定 `SHIFT_COLS`・`STRUCTS` から、比べるモデルを決める（列ごとのシフトのあるモデルと、基準の C）。結果は、学習結果フォルダの `<版>/全厚み総合/シフトファクターの検討/` にまとめる。

In [ ]:
# セル3：比べるモデルと、結果の置き場所
# ▶ 比べるモデルのリスト：設定 SHIFT_COLS の列と STRUCTS の形から作ったシフトのあるモデル（関数 model_keys は「共通の部品」のセルにある）と、最後に基準の C。
# ▶ （+ はリストをつなぐ。C が 2 回入らないように、先に C を除いておく）
KEYS = [k for k in model_keys(SHIFT_COLS, STRUCTS) if k != "C"] + ["C"]   # 比べるモデル：シフトファクターに使う列ごとのモデルと、基準の C（シフトなし）
# ▶ そのうち、シフトのあるモデル（"C" でないもの）だけのリスト。[k for k in … if …] は、条件に合う要素だけを集める書き方
SHIFT_KEYS = [k for k in KEYS if k != "C"]     # そのうち、シフトのあるモデル
# ▶ assert：MAIN_KEY が、比べるモデルのどれかでなければ、エラーで止める（, の後ろは、止まったときに表示する文）
assert MAIN_KEY in KEYS, f"MAIN_KEY（{MAIN_KEY}）は {KEYS} のどれか"


# ▶ 関数：このノートの結果を保存するフォルダの場所を返す。mat は材料、key はモデル（あれば、その下のモデルのフォルダ）、sub は、さらにその下のフォルダ
def out_dir(mat, key=None, sub=None):
    # ▶ 関数の説明（docstring）
    """このノートの結果の置き場所：<学習結果フォルダ>/<版>/全厚み総合/シフトファクターの検討/。
    key があれば、その下のモデルのフォルダ（sub があれば、さらにその下）"""
    # ▶ <学習結果フォルダ>/<版（全ショット版・条件ごとの平均版）>/全厚み総合/シフトファクターの検討 の場所。key・sub があれば、その下（*([key] if key else []) は、key があるときだけ、それを付け足す書き方）
    return result_dir(mat, VERSION_DIRS[INPUT_MODE], THICK_TOTAL, STUDY_DIR, *([key] if key else []), *([sub] if sub else []))


# ▶ 比べるモデル（"、" でつないだ名前）と、材料を表示する
print("比べるモデル：" + "、".join(model_title(k) for k in KEYS) + f"／材料：{MATERIALS}")
# ▶ 結果の置き場所（学習結果フォルダの中の、どのフォルダか）を表示する
print(f"結果の置き場所：<学習結果フォルダ>/{VERSION_DIRS[INPUT_MODE]}/{THICK_TOTAL}/{STUDY_DIR}/")

## 2. ファイルを選んで、データを読み込む

**このセルで、材料ごとにダイアログが 3 回開く**（①学習データ〔5data〕の CSV、②検証データ〔3data〕の CSV、③学習結果フォルダ。設定 `FILES` に書いたものは開かない）。読み方は、学習用のノートと同じ（問題データ一覧の「除外」の行と、η・γ̇・P3_3 が 0 以下の行を除く。検証の行は、学習データの射出速度の範囲より遅い側を**低速側**、速い側を**高速側**に分ける）。

**判定**：設定温度が学習・検証とも 3 水準ある／0 以下の値の行がすべて一覧に登録済み／基準温度 Tref が設定温度の範囲の中にある（外なら、材料とファイルの組み合わせを確かめる）／検証データに、学習データの射出速度の範囲の外の条件がある（ないなら、学習と検証のファイルを逆に選んだおそれがある）。

In [ ]:
# セル4：ファイルを選んで、読み込む
# ▶ 判定の記録を空にする（clear はリストの中身を全部消す。実行し直したときに、前の記録が残らないように）
JUDGES.clear()
# ▶ ★ここでダイアログが開く。材料ごとに、学習データ・検証データの CSV と、学習結果フォルダを決める（関数 choose_inputs は「共通の部品」のセルにある。設定 FILES に書いたものは、ダイアログを出さない）
CHOSEN = choose_inputs(MATERIALS, FILES)            # 材料ごとに、学習データ・検証データの CSV と、学習結果フォルダを決める（設定 FILES に書いていなければ、ダイアログで選ぶ）
# ▶ 決めたファイルとフォルダの表を、ノートに表示する
display(CHOSEN)
# ▶ 設定エリアの材料の、5data（学習）と 3data（検証）を全部読む（上で決めたファイル。関数 load_all は「共通の部品」のセルにある）
D = load_all(MATERIALS)
# ▶ 読んだファイルの情報を、材料・組ごとに 1 行ずつ並べた表にする
INFO = pd.DataFrame([FILE_INFO[(m, g)] for m in MATERIALS for g in ["5data", "3data"]])
# ▶ その表を、ノートに表示する
display(INFO)
# ▶ 学習データ、検証データ、シフトファクターの設定を、材料ごとに入れる空の辞書を 3 つ作る
TRAIN, TEST, CFG = {}, {}, {}
# ▶ 材料ごとに、下の字下げした行をくり返す
for mat in MATERIALS:
    # ▶ データの確認の結果を保存するフォルダ（学習結果フォルダの直下の「データ確認」）の場所。関数 result_dir は「共通の部品」のセルにある
    chk_dir = result_dir(mat, "データ確認")         # データの確認の結果（入力の取り方によらないので、学習結果フォルダの直下）
    # ▶ その材料の行だけを選んで（INFO["材料"] == … が True の行）、「データの概要.csv」に保存する
    to_csv(INFO[INFO["材料"] == MAT_NAMES[mat]], os.path.join(chk_dir, "データの概要.csv"))
    # ▶ 学習データ（5data）
    TRAIN[mat] = D[(mat, "5data")]
    # ▶ 検証データ（3data）に、低速側・高速側の区分の列を付ける
    TEST[mat] = add_groups(TRAIN[mat], D[(mat, "3data")])
    # ▶ その材料のシフトファクターの設定の写しを作る（dict(…) で写す。元の設定を書き換えないため）
    cfg = dict(SHIFT_CFG[mat])
    # ▶ 基準温度 Tref が None（指定なし）なら
    if cfg.get("Tref") is None:
        # ▶ 学習データの設定温度の水準を、小さい順に並べる（unique は重複を除く）
        lv = sorted(TRAIN[mat]["Tset"].unique())
        # ▶ 真ん中の水準を基準温度にする（// は割り算の整数部分。3 水準なら 1 番目＝2 つ目）
        cfg["Tref"] = float(lv[len(lv) // 2])
    # ▶ その材料の設定を覚える
    CFG[mat] = cfg
    # ▶ 5data と 3data で一覧の「除外」により除いた行を、組の列を付けて（assign）、1 つの表につなげる
    rem = pd.concat([REMOVED[(mat, g)].assign(組=g) for g in ["5data", "3data"]], ignore_index=True)
    # ▶ 0 以下の値で除いた行も、同じように 1 つの表にする
    inv = pd.concat([INVALID[(mat, g)].assign(組=g) for g in ["5data", "3data"]], ignore_index=True)
    # ▶ 一覧で除いた行を「一覧で除いた行.csv」に保存する
    to_csv(rem, os.path.join(chk_dir, "一覧で除いた行.csv"))
    # ▶ 検証データの区分ごとの行数の文（例：「低速側 18 行、高速側 54 行」）を作る（"、".join は、文字のリストを「、」でつなぐ）
    grp_txt = "、".join(f"{g} {int((TEST[mat]['group'] == g).sum())} 行" for g in ["低速側", "高速側", "範囲内"] if (TEST[mat]["group"] == g).any())
    # ▶ 学習・検証の行数と条件数を表示する（\n は改行。nunique は異なる値の数）
    print(f"\n【{MAT_NAMES[mat]}】学習 {len(TRAIN[mat])} 行・{TRAIN[mat]['cond'].nunique()} 条件、検証 {len(TEST[mat])} 行・{TEST[mat]['cond'].nunique()} 条件（{grp_txt}）")
    # ▶ 一覧で除いた行があれば（len が 0 でなければ「真」）
    if len(rem):
        # ▶ 除いた行のファイル名・組・ID を表示する
        print("  問題データ一覧の「除外」で除いた行：" + "、".join(f"{r['元ファイル']}（{r['組']}、{r['ID']}）" for _, r in rem.iterrows()))
    # ▶ 0 以下の値で除いた行があれば
    if len(inv):
        # ▶ そのファイル名を表示する
        print("  ★ 0 以下の値で除いた行（一覧に未登録）：" + "、".join(inv["file"]))
    # ▶ 学習と検証の、設定温度の水準の数
    n_t = (TRAIN[mat]["Tset"].nunique(), TEST[mat]["Tset"].nunique())
    # ▶ 設定温度を列から取ったか、ファイル名から取ったかの説明の文
    how = f"学習は{FILE_INFO[(mat, '5data')]['設定温度の取り方']}、検証は{FILE_INFO[(mat, '3data')]['設定温度の取り方']}から"
    # ▶ 判定：学習・検証とも、設定温度が 3 水準以上あるか（min は小さい方）
    judge(min(n_t) >= 3, f"学習・検証とも、設定温度が 3 水準以上ある（学習 {n_t[0]}・検証 {n_t[1]} 水準。設定温度は、{how}）", mat, "データ")
    # ▶ 判定：0 以下の値の行が、すべて問題データ一覧に登録済みか
    judge(len(inv) == 0, f"η・γ̇・P3_3 が 0 以下の行（対数がとれない）は、すべて問題データ一覧に登録済み（未登録のまま自動で除いた行 {len(inv)} 行）。未登録なら、一覧に「除外」で登録する", mat, "データ")
    # ▶ 判定：基準温度 Tref が、学習データの設定温度の範囲（最小〜最大）の中にあるか（a <= b <= c は「b が a 以上 c 以下」）
    judge(TRAIN[mat]["Tset"].min() <= cfg["Tref"] <= TRAIN[mat]["Tset"].max(),
          # ▶ （続き）判定の文。シフトファクターの型（アレニウス型か WLF 型か）も書く
          f"基準温度 Tref = {cfg['Tref']:g} ℃ が、学習データの設定温度の範囲の中にある（シフトファクターは{'アレニウス型' if cfg['type'] == 'arrhenius' else 'WLF 型'}）", mat, "データ")
    # ▶ 判定：検証データに、学習データの射出速度の範囲の外の条件（低速側・高速側）があるか（.any() は「1 つでも True があるか」。:g は、数を短く書く書式）
    judge((TEST[mat]["group"] != "範囲内").any(), f"検証データに、学習データの射出速度の範囲（{TRAIN[mat]['v'].min():g}〜{TRAIN[mat]['v'].max():g} mm/s）の外の条件がある"
          # ▶ （続き）判定の文。すべて範囲の中なら、学習と検証のファイルを逆に選んだおそれがある
          "（すべて範囲の中なら、学習データと検証データのファイルを逆に選んだおそれがある）", mat, "データ")

## 3. 列ごとのシフトファクターの推定（プロファイル尤度。時間がかかる）

シフトのあるモデルごとに、θ = (θ_h, θ_v)（アレニウス型は E_h・E_v [kJ/mol]、WLF 型は A1_h・A1_v）の格子の各点で GPR を学習し、対数周辺尤度（LML）を求める。粗い格子（設定 `SHIFT_CFG`）で全体を見て、最良の点を中心に細かい格子（`FINE_B`）を取って計算し直す（最良の点が動いたら、取り直す。最大 `MAX_REFINE` 回）。

**θ = 0 の意味**：シフトなし、つまり B_Tset なら「Tset を使わないモデル」、B_Tff なら「Tff を使わないモデル」。最良の LML と θ = 0 の LML の差 ΔLML が、その温度を（シフトの形で）入れる効果を表す。**列どうしを比べるときは、この差を比べる**（同じデータなので、LML どうしも比べられる）。

**尾根**：θ_h と θ_v は別々には決まりにくく、尤度の山が斜めの帯（尾根）になる。**差 θ_v − θ_h** が最もよく決まる。

**判定**：最良の θ が、粗い格子の端にも、細かい格子の端にもない（端なら `SHIFT_CFG` の範囲を広げる）。

### 3.1b 最良の θ と θ = 0 の、念入りな最適化

格子の各点では、最適化のやり直しが少ないので、LML が局所解で 1〜3 ずれることがある。そこで、**最良の θ と θ = 0 の 2 点だけ**を、最適化のやり直し `N_RESTARTS` 回で学習し直し、その 2 つから ΔLML を求める。最良の θ の学習が、そのまま最終モデルになる。

**判定**：ΔLML が 3.0 以上（その列のシフトが、尤度で支持される。小さければ、温度の効果はほかの入力が担っている）。

In [ ]:
# セル5：プロファイル尤度（目安：材料 1 つ・モデル 1 つで 5〜10 分）
# ▶ プロファイル尤度の格子の表と、最良の係数の結果を、(材料, モデル) ごとに入れる空の辞書
PROF, BEST = {}, {}
# ▶ 今の時刻（秒）を覚える（全体にかかった時間を測るため）
t_all = time.time()
# ▶ 材料ごとに、下の字下げした行をくり返す
for mat in MATERIALS:
    # ▶ 係数の名前と単位（アレニウス型なら E_h・E_v・kJ/mol、WLF 型なら A1_h・A1_v・−）
    thn, tvn, un = theta_names(CFG[mat])
    # ▶ シフトのあるモデル（B_Tset・B_Tff）ごとに
    for key in SHIFT_KEYS:
        # ▶ このモデルの計算を始めた時刻
        t0 = time.time()
        # ▶ ★プロファイル尤度で、シフトの係数を探す（係数の格子の各点で GPR を学習して、LML を記録する）。全部の点の表と、収束したかを受け取る
        pts, conv = run_profile(key, TRAIN[mat], CFG[mat])
        # ▶ 格子の表を覚える
        PROF[(mat, key)] = pts
        # ▶ ★最良の係数（LML が最大の点）と、その 95% 区間などを取り出して覚える（b = … で、同じ結果を b という短い名前でも使う）
        BEST[(mat, key)] = b = profile_best(pts, key, CFG[mat], conv)
        # ▶ 格子の表を「プロファイル尤度_格子.csv」に保存する（theta の列〔ハイパーパラメータ〕は除く）
        to_csv(pts.drop(columns="theta"), os.path.join(out_dir(mat, key), "プロファイル尤度_格子.csv"))
        # ▶ 点の数、かかった時間、最良の係数を表示する（隣り合う 2 つの f 文字列は、1 つにつながる）
        print(f"{MAT_NAMES[mat]} {key}：{len(pts)} 点、{time.time() - t0:.0f} 秒（細かい格子の取り直しが{'収束' if conv else '収束せず'}）。最良 {thn} = {b['th_best']:g}、{tvn} = {b['tv_best']:g} {un}"
              # ▶ （続き）係数の差（縦 − 横）とその範囲、格子での LML
              f"（差 {tvn} − {thn} = {b['diff_best']:g}〔{b['diff_lo']:g}〜{b['diff_hi']:g}〕）、格子の LML {b['LML_best']:.1f}")
# ▶ 全体にかかった時間（分）を表示する（シフトのあるモデルがなければ、計算しなかったことも書く）
print(f"合計 {(time.time() - t_all) / 60:.1f} 分" + ("" if SHIFT_KEYS else "（シフトのあるモデルがないので、計算なし）"))
# ▶ 材料ごとに
for mat in MATERIALS:
    # ▶ 係数の名前と単位
    thn, tvn, un = theta_names(CFG[mat])
    # ▶ モデルごとに
    for key in SHIFT_KEYS:
        # ▶ 最良の係数の結果
        b = BEST[(mat, key)]
        # ▶ 判定：最良の係数が、格子の端に当たっていないか（not は「〜でない」、and は「かつ」）
        judge((not b["at_edge"]) and b["refine_converged"], f"{key}：最良の θ が、粗い格子の端にも、細かい格子の端にもない（{thn} = {b['th_best']:g}、{tvn} = {b['tv_best']:g} {un}。取り直しが{'収束' if b['refine_converged'] else '収束せず'}）。端なら SHIFT_CFG の範囲を広げる。収束していなければ MAX_REFINE を増やす", mat, "尤度")

In [ ]:
# セル5b：最良の θ と θ = 0 の念入りな最適化（並列。目安：数分）
# ▶ 並列計算の道具（joblib）を読み込む
from joblib import Parallel, delayed
# ▶ 今の時刻（秒）を覚える（かかった時間を測るため）
t0 = time.time()
# ▶ 最終モデル、θ = 0（シフトなし）のモデル、損失の記録を入れる、空の辞書を 3 つ作る
FITS, ZERO, LOSS = {}, {}, {}
# ▶ 計算する仕事のリスト：材料 × モデル × 種類（"best"＝最良の係数、"zero"＝係数 0）のすべての組み合わせ
jobs = [(mat, key, kind) for mat in MATERIALS for key in SHIFT_KEYS for kind in ("best", "zero")]
# ▶ 仕事を並列に計算する（次の 3 行が、仕事の中身）
res = Parallel(n_jobs=min(N_JOBS, len(jobs)))(
    # ▶ 各仕事：fit_gp で学習する。係数は、best なら最良の値（横 th_best・縦 tv_best）、zero なら 0
    delayed(fit_job)(key, TRAIN[mat], CFG[mat], (BEST[(mat, key)]["th_best"] if kind == "best" else 0.0), (BEST[(mat, key)]["tv_best"] if kind == "best" else 0.0),
                     # ▶ （続き）やり直しは N_RESTARTS 回（20 回）、初期値は格子の計算で得たハイパーパラメータ、best のときだけ損失を記録する
                     N_RESTARTS, (BEST[(mat, key)]["theta_best"] if kind == "best" else BEST[(mat, key)]["theta_zero"]), INPUT_MODE, kind == "best")
    # ▶ （続き）仕事のリストの全部について。仕事がなければ（C だけなら）、計算しない（if jobs else []）
    for mat, key, kind in jobs) if jobs else []
# ▶ 仕事と結果を組にして（zip）、1 つずつ取り出す
for (mat, key, kind), f in zip(jobs, res):
    # ▶ 最良の係数での学習なら
    if kind == "best":
        # ▶ ★最終モデルとして覚える（シフトの係数は、ここで最良の値に固定される）
        FITS[(mat, key)] = f
        # ▶ 損失の記録を覚える
        LOSS[(mat, key)] = f["loss_runs"]
    # ▶ 係数 0 での学習なら
    else:
        # ▶ 比較用として覚える
        ZERO[(mat, key)] = f
# ▶ 仕事の数と、かかった時間（分）を表示する
print(f"念入りな最適化：{len(jobs)} 回、{(time.time() - t0) / 60:.1f} 分")
# ▶ 材料ごとに（シフトのあるモデルがなければ〔C だけなら〕、空のリスト [] になって、何もしない）
for mat in (MATERIALS if SHIFT_KEYS else []):       # シフトのあるモデルがなければ（C だけなら）、飛ばす
    # ▶ 係数の名前と単位
    thn, tvn, un = theta_names(CFG[mat])
    # ▶ モデルごとに
    for key in SHIFT_KEYS:
        # ▶ プロファイル尤度の結果（辞書）
        b = BEST[(mat, key)]
        # ▶ 念入りに学習した、最良の係数と係数 0 の LML
        lml_b, lml_z = float(FITS[(mat, key)]["gp"].log_marginal_likelihood_value_), float(ZERO[(mat, key)]["gp"].log_marginal_likelihood_value_)
        # ▶ 格子での値を、名前を変えて残しておく（「_grid」を付けた名前）
        b["LML_best_grid"], b["LML_at_0_grid"], b["dLML_vs_0_grid"] = b["LML_best"], b["LML_at_0"], b["dLML_vs_0"]
        # ▶ 念入りな学習と格子の値の、大きい方を採る（max）
        b["LML_best"], b["LML_at_0"] = max(lml_b, b["LML_best_grid"]), max(lml_z, b["LML_at_0_grid"])
        # ▶ ΔLML（最良 − 係数 0）を計算し直す
        b["dLML_vs_0"] = b["LML_best"] - b["LML_at_0"]
    # ▶ モデルごとに、推定の結果を 1 行にした表を作る（材料、モデル、入力の取り方〔版の名前〕、係数の名前、単位）
    t = pd.DataFrame([{"材料": MAT_NAMES[mat], "モデル": key, "入力": VERSION_DIRS[INPUT_MODE], "θ の名前": f"{thn}・{tvn}", "単位": un,
                       # ▶ （続き）結果の辞書のうち、ハイパーパラメータ（theta_best・theta_zero）以外の項目を並べる（辞書内包表記）
                       **{k: v for k, v in BEST[(mat, key)].items() if k not in ("theta_best", "theta_zero")}} for key in SHIFT_KEYS])
    # ▶ 「シフトファクターの推定.csv」に保存する
    to_csv(t, os.path.join(out_dir(mat), "シフトファクターの推定.csv"))
    # ▶ 主な列を選び、小数 2 桁に丸めて表示する（round(2)）
    display(t[["材料", "モデル", "θ の名前", "単位", "th_best", "th_lo", "th_hi", "tv_best", "tv_lo", "tv_hi", "diff_best", "diff_lo", "diff_hi", "LML_best", "LML_at_0", "dLML_vs_0", "dLML_vs_0_grid", "at_edge", "refine_converged"]].round(2))
    # ▶ モデルごとに
    for key in SHIFT_KEYS:
        # ▶ 推定の結果
        b = BEST[(mat, key)]
        # ▶ 判定の文に書く「ほかの入力」の名前（endswith は「〜で終わる」）
        other = "Tff・P3_3" if key.endswith("Tset") else "Tset・P3_3"
        # ▶ 念入りな最適化と格子での、LML と ΔLML を表示する（:+.1f は、符号付きで小数 1 桁）
        print(f"{MAT_NAMES[mat]} {key}：念入りな最適化の LML {b['LML_best']:.1f}（格子 {b['LML_best_grid']:.1f}）、θ = 0 は {b['LML_at_0']:.1f}（格子 {b['LML_at_0_grid']:.1f}）→ ΔLML {b['dLML_vs_0']:+.1f}（格子では {b['dLML_vs_0_grid']:+.1f}）")
        # ▶ 判定：ΔLML が 3.0 以上か（温度のシフトが、尤度比検定の 5% 基準で支持されるか）
        judge(b["dLML_vs_0"] >= LRT_DLML, f"{key}：温度のシフトが尤度で支持される（念入りに最適化した LML が、θ = 0〔{key.split('_')[1]} を使わない〕より {b['dLML_vs_0']:.1f} 大きい。基準 {LRT_DLML:g} 以上＝尤度比検定の 5% 基準）。小さければ、温度の効果はほかの入力（{other}）が担っている", mat, "尤度")

**図の見方**（モデルのフォルダの `プロファイル尤度.png`）：色は LML − 最大値（明るいほど尤度が高い）。赤線の内側が、おおよその 95% 区間（最大から 1.92 以内）。★ が最良の点、× が θ = 0（シフトなし）。帯が斜めに伸びるときは、θ_h と θ_v が別々には決まりにくく、差 θ_v − θ_h だけがよく決まる。

In [ ]:
# セル6：尤度の面
# ▶ 材料ごとに、下の字下げした行をくり返す
for mat in MATERIALS:
    # ▶ 係数の名前と単位
    thn, tvn, un = theta_names(CFG[mat])
    # ▶ モデルごとに
    for key in SHIFT_KEYS:
        # ▶ 格子の表と、最良の結果
        pts, b = PROF[(mat, key)], BEST[(mat, key)]
        # ▶ 各点の LML から最大値を引いた値（最良の点で 0、ほかは負）
        val = (pts["LML"] - b["LML_best"]).to_numpy()
        # ▶ 構造 A（係数が縦の 1 つだけ）なら、1 次元のグラフを描く
        if key.startswith("A"):
            # ▶ 図とグラフを作る
            fig, ax = plt.subplots(figsize=(9, 5.5))
            # ▶ 横軸 縦の係数、縦軸 LML − 最大値 を、点と線で描く（"o-" は丸と線）
            ax.plot(pts["tv"], val, "o-", color="#1f77b4")
            # ▶ −1.92 の位置に赤い破線を引く（これより上が、95% 区間の目安）
            ax.axhline(-1.92, color="red", ls="--", label="最大 − 1.92（95% 区間の目安）")
            # ▶ 係数 0 の位置に、縦線を引く
            ax.axvline(0, color="k", lw=0.6)
            # ▶ 横軸の名前
            ax.set_xlabel(f"縦のシフト {tvn} ({un})")
            # ▶ 縦軸の名前
            ax.set_ylabel("LML − 最大値")
            # ▶ 縦軸の範囲（下は −60 より下げない）
            ax.set_ylim(max(val.min(), -60), 2)
            # ▶ 凡例を描く
            ax.legend()
            # ▶ 点線の格子を描く
            ax.grid(True, ls=":", alpha=0.5)
        # ▶ 構造 B（横と縦の 2 つの係数）なら、等高線の図を描く
        else:
            # ▶ 横に 2 つのグラフが並んだ図を作る
            fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))
            # ▶ 細かい格子の範囲（±4）
            rng = (CFG[mat].get("fine") or FINE_B)[0]
            # ▶ 最良の点の近く（横・縦とも ±4 の範囲）の点を選ぶ（between は「範囲の中か」、& は「かつ」）
            near = pts[pts["th"].between(b["th_best"] - rng, b["th_best"] + rng) & pts["tv"].between(b["tv_best"] - rng, b["tv_best"] + rng)]
            # ▶ 左は全体、右は最良の点の近く、の 2 つのグラフを順に描く（lo は色の下限、ttl はタイトル）
            for ax, sub, lo, ttl in [(axes[0], pts, -60.0, "全体（粗い格子＋細かい格子）"), (axes[1], near, -15.0, "最良の点の近く（細かい格子）")]:
                # ▶ LML − 最大値（下限 lo より小さい値は lo にそろえる）
                v = np.maximum((sub["LML"] - b["LML_best"]).to_numpy(), lo)
                # ▶ 格子の点（横, 縦）を、三角形の網目でつなぐ（等高線を描くため）
                tri = mtri.Triangulation(sub["th"].to_numpy(), sub["tv"].to_numpy())
                # ▶ 色で塗り分けた等高線を描く（levels は 25 段階の区切り、cmap は色の並び）
                cs = ax.tricontourf(tri, v, levels=np.linspace(lo, 0, 25), cmap="viridis")
                # ▶ −1.92 の等高線を、赤い線で描く（その内側が 95% 区間の目安）
                ax.tricontour(tri, v, levels=[-1.92], colors="red", linewidths=2)
                # ▶ 色と値の対応を示す帯（カラーバー）を付ける
                fig.colorbar(cs, ax=ax, label="LML − 最大値")
                # ▶ 最良の点に、白い星印を描く
                ax.plot(b["th_best"], b["tv_best"], marker="*", color="white", ms=20, mec="k", ls="none", label=f"最良（{b['th_best']:g}, {b['tv_best']:g}）")
                # ▶ 係数 0 の点が、描く範囲の中にあれば
                if sub["th"].min() <= 0 <= sub["th"].max() and sub["tv"].min() <= 0 <= sub["tv"].max():
                    # ▶ そこに黒い × 印を描く
                    ax.plot(0, 0, marker="x", color="k", ms=12, mew=2.5, ls="none", label="θ = 0（シフトなし）")
                # ▶ 横軸の名前
                ax.set_xlabel(f"横のシフト {thn} ({un})")
                # ▶ 縦軸の名前
                ax.set_ylabel(f"縦のシフト {tvn} ({un})")
                # ▶ グラフのタイトル
                ax.set_title(f"{ttl}。赤線の内側：95% 区間の目安")
                # ▶ 凡例を左上に描く
                ax.legend(loc="upper left", fontsize=10)
        # ▶ 図全体のタイトル（材料・モデル・シフトファクターの型）
        fig.suptitle(f"{MAT_NAMES[mat]}　{model_title(key)}　プロファイル尤度（{'アレニウス型' if CFG[mat]['type'] == 'arrhenius' else 'WLF 型'}）")
        # ▶ モデルのフォルダに「プロファイル尤度.png」として保存する
        save_fig(fig, out_dir(mat, key), "プロファイル尤度")

## 4. 基準のモデル C の学習と、ハイパーパラメータ

シフトのあるモデルは、3.1b の念入りな学習が、そのまま最終モデル。基準の C（シフトなし。4 つの説明変数を並べた通常の GPR）は、ここで同じ回数（`N_RESTARTS`）で学習する。表は、全モデルのハイパーパラメータ（振幅、長さスケール、線形項、ノイズ）と対数周辺尤度（LML）。

**判定**：最終モデルの LML が、尤度の格子の最良以上／ノイズ分散が下限（`NOISE_LB`）に張り付いていない。

In [ ]:
# セル7：基準のモデル C の学習（シフトのあるモデルは 3.1b で済み）
# ▶ 今の時刻（秒）を覚える
t0 = time.time()
# ▶ シフトのないモデル（C）だけのリスト（KEYS のうち、SHIFT_KEYS にないもの。not in は「含まれない」）
NOSHIFT_KEYS = [k for k in KEYS if k not in SHIFT_KEYS]      # シフトのないモデル（C）
# ▶ 計算する仕事のリスト：材料 × シフトのないモデル（シフトのあるモデルだけなら、空のリスト）
jobs = [(mat, key) for mat in MATERIALS for key in NOSHIFT_KEYS]
# ▶ シフトのないモデルを、並列で学習する（係数は 0, 0、やり直し 20 回、損失を記録する）。仕事がなければ（if jobs else []）、何もしない
res = Parallel(n_jobs=min(N_JOBS, len(jobs)))(delayed(fit_job)(key, TRAIN[mat], CFG[mat], 0.0, 0.0, N_RESTARTS, None, INPUT_MODE, True) for mat, key in jobs) if jobs else []
# ▶ 仕事と結果を組にして、1 つずつ取り出す
for (mat, key), f in zip(jobs, res):
    # ▶ 最終モデルとして覚える
    FITS[(mat, key)] = f
    # ▶ 損失の記録を覚える
    LOSS[(mat, key)] = f["loss_runs"]
# ▶ 学習した回数と、かかった時間を表示する
print(f"シフトのないモデルの学習：{len(jobs)} 回、{(time.time() - t0) / 60:.1f} 分")
# ▶ ハイパーパラメータの表の行を入れる、空のリスト
KROWS = []
# ▶ 材料ごとに
for mat in MATERIALS:
    # ▶ モデルごとに
    for key in KEYS:
        # ▶ 最終モデル
        f = FITS[(mat, key)]
        # ▶ ハイパーパラメータを 1 行にして足す（**kernel_row(f) は、返ってきた辞書の中身を並べる書き方）
        KROWS.append({"材料": MAT_NAMES[mat], **kernel_row(f)})
        # ▶ 学習したカーネル（式の形と値）を表示する
        print(f"{MAT_NAMES[mat]} {key}：{f['gp'].kernel_}")
# ▶ 行のリストを表にする
KT = pd.DataFrame(KROWS)
# ▶ 小数 4 桁に丸めて表示する
display(KT.round(4))
# ▶ 材料ごとに
for mat in MATERIALS:
    # ▶ その材料の行を「ハイパーパラメータ.csv」に保存する（関数 out_dir は、このノートの結果の置き場所）
    to_csv(KT[KT["材料"] == MAT_NAMES[mat]], os.path.join(out_dir(mat), "ハイパーパラメータ.csv"))
    # ▶ モデルごとに
    for key in KEYS:
        # ▶ その材料・モデルの行（iloc[0] は最初の 1 行）
        r = KT[(KT["材料"] == MAT_NAMES[mat]) & (KT["model"] == key)].iloc[0]
        # ▶ シフトのあるモデルなら（格子の計算がある）
        if key in SHIFT_KEYS:
            # ▶ 格子での最良の LML
            lb = BEST[(mat, key)]["LML_best"]
            # ▶ 判定：最終モデルの LML が、格子の最良（から 0.5 引いた値）以上か（最適化がうまくいったか）
            judge(r["LML"] >= lb - 0.5, f"{key}：最終モデルの LML（{r['LML']:.1f}）が、尤度の格子の最良（{lb:.1f}）以上", mat, "学習")
        # ▶ 判定：ノイズ分散が、下限に張り付いていないか
        judge(not r["noise_at_lower_bound"], f"{key}：ノイズ分散（{r['noise_var']:.2e}）が下限 {NOISE_LB:g} に張り付いていない", mat, "学習")
        # ▶ 長さスケールが上限（100）か下限（0.01）に届いた入力の名前を集める（c[3:] は 4 文字目から。「ls_」を除いた名前）
        at = [c[3:] for c in KT.columns if c.startswith("ls_") and c != "ls_at_bound" and pd.notna(r[c]) and (r[c] > 99 or r[c] < 0.0105)]
        # ▶ そういう入力があれば
        if at:
            # ▶ （参考）として表示する
            print(f"  （参考）{MAT_NAMES[mat]} {key}：長さスケールが上限・下限に届いた入力：{'、'.join(at)}")

## 5. 列ごとのシフトファクターと、マスターカーブ

- **シフトファクターの図**（`シフトファクター.png`）：左が横のシフト ln a_h(T)、右が縦のシフト ln a_v(T)。**列（モデル）ごとに色を変えて重ねる**。線が最良の θ、帯が尤度の 95% 区間。点線は設定温度、薄い橙は Tff の範囲。B_Tset は設定温度、B_Tff は流動中樹脂温度で読む。ln a_v(T) > 0 は「その温度では、同じせん断速度で粘度が Tref より高い」。
- **マスターカーブ**（モデルのフォルダの `マスターカーブ.png`）：条件平均の点を、最良の θ でずらしたもの（横：γ̇·a_h、縦：η/a_v）。シフトが効いていれば、厚みごとに、温度の違う点が 1 本の曲線に近づく（シフト前は `流動曲線_シフト前.png`）。
- **重ね合わせの残差**：厚みごとに、全温度の条件平均の点を 3 次式で当てはめた残差の RMSE（ln η）。シフト前より小さければ、シフトで曲線が重なった。列どうしで比べると、どちらの温度で、曲線がよく重なるかが分かる。

**判定**：縦のシフトの向きが「低温ほど粘い」（最も低い設定温度で ln a_v > 0）／重ね合わせの残差が、シフト前より小さい。

In [ ]:
# セル8：シフトファクターとマスターカーブ
# ▶ シフトファクターの倍率の表の行を入れる、空のリスト
SHIFT_ROWS = []
# ▶ 材料ごとに、下の字下げした行をくり返す
for mat in MATERIALS:
    # ▶ その材料のシフトファクターの設定
    cfg = CFG[mat]
    # ▶ 係数の名前と単位
    thn, tvn, un = theta_names(cfg)
    # ▶ 学習データと検証データを、1 つの表にする
    pool = pd.concat([TRAIN[mat], TEST[mat]], ignore_index=True)
    # ▶ 学習データの条件名の集合
    train_c = set(TRAIN[mat]["cond"])
    # ▶ 設定温度の最低と最高
    tl, th_ = pool["Tset"].min(), pool["Tset"].max()
    # ▶ シフトのあるモデルがあるときだけ（C だけなら、シフトファクターの図はない）、下の字下げした行で、シフトファクターの図を描く
    if SHIFT_KEYS:                                  # シフトファクターの図（シフトのあるモデルがあるときだけ）
        # ▶ シフトファクターの曲線を描く温度の並び（Tff と設定温度の範囲を、両側に 5 ℃ 広げて 200 点）。この Tg は温度の並びの名前で、ガラス転移温度ではない
        Tg = np.linspace(min(pool["Tff"].min(), tl) - 5, max(pool["Tff"].max(), th_) + 5, 200)
        # ▶ 横に 2 つのグラフ（左：横のシフト、右：縦のシフト）が並んだ図を作る
        fig, axes = plt.subplots(1, 2, figsize=(16, 6.2))
        # ▶ モデルごとの線の色
        cols_ = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd"]
        # ▶ シフトのあるモデルと色を組にして、1 つずつ
        for key, col in zip(SHIFT_KEYS, cols_):
            # ▶ 格子の表と、最良の結果
            pts, b = PROF[(mat, key)], BEST[(mat, key)]
            # ▶ 95% 区間に入る格子の点（LML が最大から 1.92 以内）
            inside = pts[pts["LML"] >= b["LML_best"] - 1.92]
            # ▶ 左のグラフに横の係数（"th"）、右のグラフに縦の係数（"tv"）の曲線を描く
            for ax, w, nm in [(axes[0], "th", thn), (axes[1], "tv", tvn)]:
                # ▶ 構造 A は横のシフトがないので
                if w == "th" and key.startswith("A"):
                    # ▶ とばす
                    continue
                # ▶ 95% 区間の中の各係数で、ln a(T) の曲線を計算する（関数 shift_ln）
                cur = np.array([shift_ln(Tg, v, cfg) for v in inside[w]])
                # ▶ それらの曲線の最小〜最大を、薄く塗る（係数の不確かさの帯）
                ax.fill_between(Tg, cur.min(axis=0), cur.max(axis=0), color=col, alpha=0.18)
                # ▶ 最良の係数での ln a(T) の曲線を、太い線で描く（凡例に、係数の値と 95% 区間を書く）
                ax.plot(Tg, shift_ln(Tg, b[f"{w}_best"], cfg), color=col, lw=2.5, label=f"{key}（{nm} = {b[f'{w}_best']:g} {un}、95%：{b[f'{w}_lo']:g}〜{b[f'{w}_hi']:g}）")
            # ▶ 表の 1 行：材料、モデル、式の型、基準温度
            SHIFT_ROWS.append({"材料": MAT_NAMES[mat], "モデル": key, "型": "アレニウス" if cfg["type"] == "arrhenius" else "WLF", "Tref [℃]": cfg["Tref"],
                               # ▶ （続き）横・縦の最良の係数と単位
                               f"横 {thn}": b["th_best"], f"縦 {tvn}": b["tv_best"], "単位": un,
                               # ▶ （続き）縦の倍率：最低温度の a_v ÷ 最高温度の a_v（同じせん断速度で、低温の粘度が高温の何倍か）
                               f"縦の倍率 a_v({tl:g})/a_v({th_:g})": float(np.exp(shift_ln(tl, b["tv_best"], cfg) - shift_ln(th_, b["tv_best"], cfg))),
                               # ▶ （続き）横の倍率：最低温度の a_h ÷ 最高温度の a_h
                               f"横の倍率 a_h({tl:g})/a_h({th_:g})": float(np.exp(shift_ln(tl, b["th_best"], cfg) - shift_ln(th_, b["th_best"], cfg)))})
        # ▶ 2 つのグラフのそれぞれについて
        for ax, ttl in [(axes[0], "横のシフト ln a_h(T)（せん断速度の向き）"), (axes[1], "縦のシフト ln a_v(T)（粘度の向き）")]:
            # ▶ 設定温度の水準ごとに
            for t in sorted(pool["Tset"].unique()):
                # ▶ その温度の位置に、点線の縦線を引く
                ax.axvline(t, color="gray", ls=":", lw=1)
            # ▶ Tff のデータの範囲を、薄いオレンジで塗る（axvspan）
            ax.axvspan(pool["Tff"].min(), pool["Tff"].max(), color="orange", alpha=0.07, label="Tff の範囲（データ）")
            # ▶ ln a = 0（a = 1。基準温度と同じ）の位置に線を引く
            ax.axhline(0, color="k", lw=0.6)
            # ▶ 横軸の名前
            ax.set_xlabel("温度 T (℃)（B_Tset は設定温度、B_Tff は流動中樹脂温度）")
            # ▶ 縦軸の名前（タイトルの「（」より前から「のシフト 」を除いた文字。例：「横 ln a_h(T)」。split は区切る、replace は置き換える）
            ax.set_ylabel(ttl.split("（")[0].replace("のシフト ", " "))
            # ▶ グラフのタイトル
            ax.set_title(ttl)
            # ▶ 凡例を、空いている所に描く
            ax.legend(fontsize=9, loc="best")
            # ▶ 点線の格子を描く
            ax.grid(True, ls=":", alpha=0.5)
        # ▶ 図のタイトルに書く、式の型の説明
        typ = "アレニウス型" if cfg["type"] == "arrhenius" else f"WLF 型（D2 = Tg = {cfg['Tg']:g} ℃、A2 = {cfg['A2']:g} K）"
        # ▶ 図全体のタイトル
        fig.suptitle(f"{MAT_NAMES[mat]}　温度のシフトファクター（{typ}、Tref = {cfg['Tref']:g} ℃。帯：尤度の 95% 区間）")
        # ▶ このノートの結果のフォルダ（out_dir(mat)）に「シフトファクター.png」として保存する
        save_fig(fig, out_dir(mat), "シフトファクター")
    # マスターカーブ（シフト前と、モデルごと）
    # ▶ 厚みの水準
    Hs = sorted(pool["H"].unique())
    # ▶ グラフを並べる列の数（厚みが 4 つなら 2 列）
    ncol = 2 if len(Hs) == 4 else len(Hs)
    # ▶ 行の数（np.ceil は切り上げ）
    nrow = int(np.ceil(len(Hs) / ncol))
    # ▶ 設定温度ごとの色と形
    ts = t_style(pool["Tset"].unique())
    # ▶ 重ね合わせの残差を入れる辞書
    sup = {}
    # ▶ シフトなし（None）と、シフトのある各モデルについて
    for key in [None] + SHIFT_KEYS:
        # ▶ 最良の係数（シフトなしなら 0, 0。get の 2 つ目は、見つからないときの値）
        b = BEST.get((mat, key), {"th_best": 0.0, "tv_best": 0.0})
        # ▶ ★条件平均の点を、最良の係数でずらす（横：γ̇ × a_h、縦：η ÷ a_v）。(key or "") は、key が None なら "" にする書き方
        t = shifted_cond_means(pool, key, cfg, b["th_best"] if (key or "").startswith("B") else 0.0, b["tv_best"])
        # ▶ 重ね合わせの残差（温度ごとの曲線が、どれだけ 1 本に重なったか。小さいほどよく重なる）
        sup[key] = superpose_rmse(t)
        # ▶ 厚みごとのグラフを並べた図を作る（squeeze=False：常に 2 次元の並びで受け取る）
        fig, axes = plt.subplots(nrow, ncol, figsize=(7.5 * ncol, 5.6 * nrow), squeeze=False)
        # ▶ グラフと厚みを組にして、1 つずつ（axes.flat は、グラフの並びを 1 列にしたもの）
        for ax, h in zip(axes.flat, Hs):
            # ▶ 設定温度ごとに
            for tt, (col, mk) in ts.items():
                # ▶ その厚み・温度の条件の印
                m = (t["H"] == h) & (t["Tset"] == tt)
                # ▶ そのうち、学習の条件と、検証の条件
                trm, tem = m & t["cond"].isin(train_c), m & ~t["cond"].isin(train_c)
                # ▶ 学習の条件の点を描く（塗りつぶし。np.exp で ln を元の値に戻す）
                ax.scatter(np.exp(t.loc[trm, "x"]), np.exp(t.loc[trm, "y"]), color=col, marker=mk, s=60, edgecolors="k", linewidths=0.5, label=f"{int(tt)} ℃", zorder=3)
                # ▶ 検証の条件の点を、白抜きで描く
                ax.scatter(np.exp(t.loc[tem, "x"]), np.exp(t.loc[tem, "y"]), facecolors="white", edgecolors=col, marker=mk, s=60, linewidths=1.8, zorder=3)
            # ▶ 横軸を対数軸にする
            ax.set_xscale("log")
            # ▶ 縦軸を対数軸にする
            ax.set_yscale("log")
            # ▶ 対数軸の目盛りを整える
            log_ticks(ax)
            # ▶ グラフのタイトル（厚み）
            ax.set_title(f"厚み {h:g} mm")
            # ▶ 横軸の名前（シフト前は「せん断速度」、シフト後は「換算せん断速度 γ̇·a_h」）
            ax.set_xlabel(r"せん断速度 $\dot{\gamma}$ (1/s)" if key is None else r"換算せん断速度 $\dot{\gamma}\cdot a_h$ (1/s)")
            # ▶ 縦軸の名前（シフト前は「粘度」、シフト後は「換算粘度 η/a_v」）
            ax.set_ylabel(r"粘度 $\eta$ (Pa·s)" if key is None else r"換算粘度 $\eta / a_v$ (Pa·s)")
            # ▶ 点線の格子を描く
            ax.grid(True, which="both", ls=":", alpha=0.5)
            # ▶ 凡例を描く（見出し「設定温度」）
            ax.legend(fontsize=9, title="設定温度")
        # ▶ 使わなかったグラフの枠（厚みの数より多い分）について
        for ax in list(axes.flat)[len(Hs):]:
            # ▶ 枠を消す
            ax.axis("off")
        # ▶ 図のタイトル（シフト前か、マスターカーブか）
        ttl = "シフト前（実測の条件平均）" if key is None else f"マスターカーブ：{model_title(key)}（最良の θ でずらした条件平均）"
        # ▶ 図全体のタイトル（重ね合わせの残差も書く）
        fig.suptitle(f"{MAT_NAMES[mat]}　{ttl}\n塗り：学習（5data）、白抜き：検証（3data）。重ね合わせの残差 {sup[key]:.3f}")
        # ▶ シフト前の図は結果のフォルダ（out_dir(mat)）に「流動曲線_シフト前.png」、シフト後の図はモデルのフォルダ（out_dir(mat, key)）に「マスターカーブ.png」として保存する
        save_fig(fig, out_dir(mat) if key is None else out_dir(mat, key), "流動曲線_シフト前" if key is None else "マスターカーブ",
                 # ▶ （続き）ノートに表示するのは、シフト前と、主に見せるモデルだけ
                 show=SHOW_FIGS and key in (None, MAIN_KEY))
    # ▶ 表の行ごとに
    for r in SHIFT_ROWS:
        # ▶ この材料の行なら
        if r["材料"] == MAT_NAMES[mat]:
            # ▶ シフト後とシフト前の、重ね合わせの残差を足す
            r["重ね合わせの残差（シフト後）"], r["重ね合わせの残差（シフト前）"] = sup[r["モデル"]], sup[None]
    # ▶ シフトのあるモデルごとに
    for key in SHIFT_KEYS:
        # ▶ 最良の係数
        b = BEST[(mat, key)]
        # ▶ 判定：最低温度で ln a_v が正か（＝低温ほど粘度が高い向きにずれているか）
        judge(shift_ln(tl, b["tv_best"], cfg) > 0, f"{key}：縦のシフトの向きが「低温ほど粘い」（ln a_v({tl:g} ℃) = {float(shift_ln(tl, b['tv_best'], cfg)):+.3f}。{tvn} = {b['tv_best']:g} {un}）", mat, "シフト")
        # ▶ 判定：シフトで、重ね合わせの残差が小さくなったか（温度ごとの曲線が近づいたか）
        judge(sup[key] < sup[None], f"{key}：シフトで温度ごとの曲線が近づく（重ね合わせの残差 {sup[None]:.3f} → {sup[key]:.3f}）", mat, "シフト")
# ▶ 行のリストを表にする
SHIFT_TAB = pd.DataFrame(SHIFT_ROWS)
# ▶ 小数 3 桁に丸めて表示する
display(SHIFT_TAB.round(3))
# ▶ 材料ごとに（シフトのあるモデルがなければ〔C だけなら〕、表が空なので、空のリスト [] になって、何もしない）
for mat in (MATERIALS if len(SHIFT_TAB) else []):   # シフトのあるモデルがなければ（C だけなら）、表はない
    # ▶ その材料の行を、すべて空の列を除いて（dropna）、「シフトファクターの倍率.csv」に保存する
    to_csv(SHIFT_TAB[SHIFT_TAB["材料"] == MAT_NAMES[mat]].dropna(axis=1, how="all"), os.path.join(out_dir(mat), "シフトファクターの倍率.csv"))

## 6. 3data による予測検証（列どうしと、基準の C の比較）

5data で学習した各モデルで、3data（射出速度の両端。低速側 3・5 mm/s、高速側 280・320 mm/s）を予測する。3data で測った P3_3・Tff を入力に使う。
- **指標**（ln η）：RMSE（小さいほど良い。0.1 は粘度で約 10%）、R²、95% 予測区間に入る割合（被覆率）、NLPD。**R²（実数）**は、粘度の実数値で計算した R²。
- **パリティ図**（モデルのフォルダの `予測検証_3data_パリティ.png`）と、**RMSE の比較**（`予測検証_RMSEの比較.png`。全モデルを、全体・低速側・高速側で並べる）。

**判定**：シフトのあるモデルの RMSE（全体）が、C の 1.1 倍以内（シフトの構造を入れても悪くならない）／各モデルの被覆率が 0.8 以上。

**注意**：3data の誤差は、最適化の局所解（乱数シード）で大きく動くことがある（10-15 の確認。`04_補助スクリプト/mat_seed_check.py` で確かめられる）。差が小さいときは、尤度（ΔLML）も合わせて判断する。

In [ ]:
# セル9：3data による予測検証
# ▶ 予測値の表を入れる辞書と、指標の行を入れるリスト
PRED, MROWS = {}, []
# ▶ 材料ごとに、下の字下げした行をくり返す
for mat in MATERIALS:
    # ▶ 検証データ（3data）
    te = TEST[mat]
    # ▶ モデルごとに
    for key in KEYS:
        # ▶ ★最終モデルで、検証データの粘度を予測する（予測平均、予測標準偏差、曲線そのものの標準偏差、ノイズの標準偏差）
        mu, sd, sdf, nsd = predict(FITS[(mat, key)], te)
        # ▶ 検証データの主な列に、予測の列を足した表を作る（assign は列を足す）
        PRED[(mat, key)] = te[["file", "cond", "Tset", "H", "v", "group", "eta", "gd", "P33", "Tff", "ln_eta"]].assign(
            # ▶ （続き）予測の ln η、予測標準偏差、予測の粘度、95% 予測区間の下限と上限（np.exp で粘度に戻す）
            予測_ln_eta=mu, 予測標準偏差=sd, 潜在関数の標準偏差=sdf, 予測η=np.exp(mu), 下限95=np.exp(mu - 1.96 * sd), 上限95=np.exp(mu + 1.96 * sd))
        # ▶ モデルのフォルダに「予測検証_3data_予測値.csv」として保存する
        to_csv(PRED[(mat, key)], os.path.join(out_dir(mat, key), "予測検証_3data_予測値.csv"))
        # ▶ 全体・低速側・高速側ごとの指標を計算して、リストに足す（+= はリストの後ろにつなげる）
        MROWS += metrics_rows(mat, key, te, mu, sd, sdf, {"ノイズの標準偏差": nsd})
# ▶ 指標の表
MT = pd.DataFrame(MROWS)
# ▶ 区分ごとの色と形（低速側：青の丸、高速側：赤の四角、範囲内：緑の三角）
GCOL = {"低速側": ("#1f77b4", "o"), "高速側": ("#d62728", "s"), "範囲内": ("#2ca02c", "^")}
# ▶ 材料ごとに
for mat in MATERIALS:
    # ▶ その材料の指標の行
    t = MT[MT["材料"] == MAT_NAMES[mat]]
    # ▶ このノートの結果のフォルダ（out_dir(mat)）に「予測検証_3data_指標.csv」として保存する（学習用のノートでは、次のセルで、Cross-WLF の行を足して保存し直す）
    to_csv(t, os.path.join(out_dir(mat), "予測検証_3data_指標.csv"))
    # ▶ 見出しを表示する
    print(f"\n【{MAT_NAMES[mat]}】3data の ln η の RMSE")
    # ▶ 行＝モデル、列＝範囲 の RMSE の表を表示する（pivot_table は、縦長の表を縦横の表に組み替える）
    display(t.pivot_table(index="モデル", columns="範囲", values="RMSE", sort=False).round(4))
    # ▶ 見出しを表示する
    print("95% 予測区間に入る割合（被覆率）")
    # ▶ 被覆率の表を表示する
    display(t.pivot_table(index="モデル", columns="範囲", values="coverage95", sort=False).round(3))
    # ▶ 全体の R²・NLPD・予測標準偏差の平均などを表示する
    display(t[t["範囲"] == "全体"][["モデル", "R2", "R2_real", "NLPD", "mean_sd", "ノイズの標準偏差"]].round(4))
    # ▶ モデルごとに
    for key in KEYS:
        # ▶ 予測値の表
        p = PRED[(mat, key)]
        # ▶ そのモデルの「全体」の指標の行
        r = t[(t["モデル"] == key) & (t["範囲"] == "全体")].iloc[0]
        # ▶ 図とグラフを作る（ほぼ正方形）
        fig, ax = plt.subplots(figsize=(9.5, 9))
        # ▶ 区分ごとに
        for g, (col, mk) in GCOL.items():
            # ▶ その区分の行の印
            m = (p["group"] == g).to_numpy()
            # ▶ その区分の行があれば
            if m.any():
                # ▶ 横軸 実測の粘度、縦軸 予測の粘度 の点に、95% 予測区間を縦の線（エラーバー）で付けて描く（パリティプロット）
                ax.errorbar(p.loc[m, "eta"], p.loc[m, "予測η"], yerr=[p.loc[m, "予測η"] - p.loc[m, "下限95"], p.loc[m, "上限95"] - p.loc[m, "予測η"]],
                            # ▶ （続き）点の形・色・大きさ、線の太さ、線の端の横棒の長さ（capsize）、凡例の名前
                            fmt=mk, color=col, ms=6, elinewidth=1.2, capsize=3, alpha=0.8, label=f"{g}（{int(m.sum())} ショット）")
        # ▶ 両軸の範囲（実測と区間の最小〜最大を、1.3 倍広げる）
        lim = [min(p["eta"].min(), p["下限95"].min()) / 1.3, max(p["eta"].max(), p["上限95"].max()) * 1.3]
        # ▶ 「予測 = 実測」の対角線を、黒い破線で描く（点がこの線に近いほど、予測が当たっている）
        ax.plot(lim, lim, "k--", lw=1, label="予測 = 実測")
        # ▶ 横軸を対数軸にする
        ax.set_xscale("log")
        # ▶ 縦軸を対数軸にする
        ax.set_yscale("log")
        # ▶ 横軸の範囲を決める
        ax.set_xlim(lim)
        # ▶ 縦軸の範囲を決める
        ax.set_ylim(lim)
        # ▶ 縦と横の目盛りの間隔をそろえる
        ax.set_aspect("equal")
        # ▶ 対数軸の目盛りを整える
        log_ticks(ax)
        # ▶ 横軸の名前
        ax.set_xlabel(r"実測の粘度 $\eta$ (Pa·s)")
        # ▶ 縦軸の名前
        ax.set_ylabel(r"予測の粘度 $\eta$ (Pa·s)")
        # ▶ タイトル（RMSE・R²・被覆率も書く）
        ax.set_title(f"{MAT_NAMES[mat]}　{model_title(key)}\n3data：RMSE(ln η) {r['RMSE']:.3f}、R²(ln η) {r['R2']:.4f}、R²(実数) {r['R2_real']:.4f}、被覆率 {r['coverage95']:.2f}")
        # ▶ 破線の格子を描く
        ax.grid(True, which="both", ls="--", alpha=0.4)
        # ▶ 凡例を左上に描く
        ax.legend(loc="upper left")
        # ▶ モデルのフォルダに「予測検証_3data_パリティ.png」として保存する
        save_fig(fig, out_dir(mat, key), "予測検証_3data_パリティ", show=SHOW_FIGS and key == MAIN_KEY)
    # ▶ モデルの中に、基準の C（シフトなし）があれば、C と比べる（in は「含まれる」。C だけを学習したときは、比べるシフトのあるモデルがないので、下の判定は出ない）
    if "C" in KEYS:                                 # 基準の C（シフトなし）と比べる（C とシフトのあるモデルの両方を学習したとき）
        # ▶ 基準のモデル C の、全体の RMSE
        rc = float(t[(t["モデル"] == "C") & (t["範囲"] == "全体")]["RMSE"].iloc[0])
        # ▶ シフトのあるモデルごとに
        for key in SHIFT_KEYS:
            # ▶ そのモデルの、全体の RMSE
            rk = float(t[(t["モデル"] == key) & (t["範囲"] == "全体")]["RMSE"].iloc[0])
            # ▶ 判定：C の RMSE の 1.1 倍以内か
            judge(rk <= 1.1 * rc, f"{key} の 3data の RMSE（全体）{rk:.4f} が、C（{rc:.4f}）の 1.1 倍以内", mat, "検証")
    # ▶ モデルごとの、全体の被覆率（set_index で、モデル名から引ける形にする）
    cov = t[t["範囲"] == "全体"].set_index("モデル")["coverage95"]
    # ▶ 判定：どのモデルも、被覆率が 0.8 以上か
    judge(bool((cov >= 0.8).all()), "95% 予測区間に入る割合が、どのモデルも 0.8 以上（" + "、".join(f"{k} {v:.2f}" for k, v in cov.items()) + "）", mat, "検証")

In [ ]:
# セル10：3data の予測誤差の比較（モデルごと）
# ▶ 材料ごとに、下の字下げした行をくり返す
for mat in MATERIALS:
    # ▶ 検証の指標の表（MT。前のセルで作った）から、その材料の行を取り出す
    t = MT[MT["材料"] == MAT_NAMES[mat]]
    # ▶ 感度確認の範囲を除き（str.contains は「その文字を含むか」）、行＝モデル、列＝範囲 の RMSE の表にする
    pv = t[~t["範囲"].str.contains("感度")].pivot_table(index="モデル", columns="範囲", values="RMSE", sort=False)
    # ▶ 図とグラフを作る
    fig, ax = plt.subplots(figsize=(11, 5.5))
    # ▶ 棒の幅（0.8 をモデルの数で割る）
    w = 0.8 / len(pv.index)
    # ▶ モデルごとに（i は番号、k はモデル名、row はその行）
    for i, (k, row) in enumerate(pv.iterrows()):
        # ▶ 範囲ごとの RMSE を、棒で描く（モデルごとに、横に棒の幅ずつずらす。np.arange(n) は 0, 1, …, n−1）
        ax.bar(np.arange(len(pv.columns)) + i * w, row.to_numpy(), width=w, label=k)
    # ▶ 横軸の目盛りの位置を、棒のまとまりの真ん中にする
    ax.set_xticks(np.arange(len(pv.columns)) + w * (len(pv.index) - 1) / 2)
    # ▶ 目盛りの名前を、範囲の名前にする
    ax.set_xticklabels(pv.columns)
    # ▶ 縦軸の名前
    ax.set_ylabel("3data の RMSE（ln η）")
    # ▶ タイトル
    ax.set_title(f"{MAT_NAMES[mat]}　3data の予測誤差の比較（小さいほど良い）")
    # ▶ 縦軸の方向だけに、破線の格子を描く
    ax.grid(axis="y", ls="--", alpha=0.5)
    # ▶ 凡例を描く
    ax.legend()
    # ▶ 結果のフォルダに「予測検証_RMSEの比較.png」として保存する
    save_fig(fig, out_dir(mat), "予測検証_RMSEの比較")

## 7. 学習済みモデルの保存

モデルのフォルダに `学習済みモデル_<モデル>.joblib` として保存する（学習データ・検証データの場所も記録する。Git には入れない）。**厚みごとの比較のノートが、このモデルと、3data の予測値（`予測検証_3data_予測値.csv`）を読む**。使い方は、学習用のノートの「12. 学習済みモデルの保存」と同じ。

In [ ]:
# セル11：学習済みモデルの保存
# ▶ joblib：Python の物（学習したモデルなど）を、ファイルに保存したり読み込んだりする部品を読み込む
import joblib
# ▶ scikit-learn を読み込む（バージョンを記録するため）
import sklearn
# ▶ 材料ごとに、下の字下げした行をくり返す
for mat in MATERIALS:
    # ▶ モデルごとに
    for key in KEYS:
        # ▶ 保存するファイルの場所（モデルのフォルダの「学習済みモデル_<モデル名>.joblib」）
        path = os.path.join(out_dir(mat, key), f"学習済みモデル_{key}.joblib")
        # ▶ 学習結果と説明をまとめた辞書を、ファイルに保存する（joblib.dump）。最終モデル（シフトの係数 th・tv も入っている）、材料、モデル名、入力の名前、入力の取り方
        joblib.dump({"fit": FITS[(mat, key)], "材料": MAT_NAMES[mat], "モデル": key, "説明": model_title(key), "入力": input_labels(key), "入力の取り方": INPUT_MODE,
                     # ▶ （続き）シフトファクターの設定と、学習データ・検証データのファイルの場所（show_path で、リポジトリの中ならリポジトリ直下から見た形にする。厚みごとの比較のノートが、この記録からデータを読む）
                     "シフトファクター": CFG[mat], "学習データのファイル": show_path(SOURCES[mat]["5data"]), "検証データのファイル": show_path(SOURCES[mat]["3data"]),
                     # ▶ （続き）scikit-learn のバージョンと、作成日（time.strftime で、今日の日付を「年-月-日」の文字にする）、保存先
                     "scikit-learn": sklearn.__version__, "作成日": time.strftime("%Y-%m-%d")}, path)
        # ▶ 保存した場所を、リポジトリ直下から見た形で表示する
        print("保存:", show_path(path))

## 8. まとめ：どの列をシフトファクターに使うか

判定の一覧（★ の行を確かめる）と、モデルごとの主な結果（θ と 95% 区間、ΔLML、3data の RMSE・被覆率）。`まとめ.csv`・`判定の一覧.csv` に保存する。最後に、材料ごとに **学習用のノートの `SHIFT_COL` の候補**を示す（ΔLML が基準 3.0 以上のシフトのうち、差が最も大きい列。どれも届かなければ、シフトなしの `None` も候補）。これは参考で、3data の誤差（C との比較）や、厚みごとの比較のノートの結果も見て、決めるのはユーザー。

In [ ]:
# セル12：まとめ（どの列をシフトファクターに使うか）
# ▶ これまでの判定の記録を、表にする
JT = pd.DataFrame(JUDGES)
# ▶ 判定の件数と、★要確認の件数を表示する
print(f"判定 {len(JT)} 件のうち、★要確認 {int((JT['判定'] != 'OK').sum())} 件")
# ▶ ★要確認の判定だけを表示する
display(JT[JT["判定"] != "OK"])
# ▶ モデルの名前の温度（"Tset"・"Tff"）→ 学習用のノートの設定 SHIFT_COL に書く列名、の辞書
COL_OF = {"Tset": "加熱シリンダ温度[℃]", "Tff": "流動中樹脂温度(℃)"}      # モデルの記号の温度 → 学習用のノートの SHIFT_COL に書く列名
# ▶ まとめの表の行を入れる、空のリスト
SUM = []
# ▶ 材料ごとに、下の字下げした行をくり返す
for mat in MATERIALS:
    # ▶ 係数の名前と単位
    thn, tvn, un = theta_names(CFG[mat])
    # ▶ 比べるモデルごとに
    for key in KEYS:
        # ▶ まとめの 1 行：材料、モデル、入力の取り方（版の名前）
        row = {"材料": MAT_NAMES[mat], "モデル": key, "入力": VERSION_DIRS[INPUT_MODE],
               # ▶ （続き）学習用のノートの SHIFT_COL と SHIFT_STRUCT に書く値（シフトのあるモデルだけ。C は None）。key.split("_") は、モデルの名前を _ で分けたリスト（例：["B", "Tset"]）
               "SHIFT_COL": COL_OF[key.split("_")[1]] if key in SHIFT_KEYS else None, "SHIFT_STRUCT": key.split("_")[0] if key in SHIFT_KEYS else None}
        # ▶ シフトのあるモデルなら
        if key in SHIFT_KEYS:
            # ▶ 推定の結果
            b = BEST[(mat, key)]
            # ▶ 係数（最良の値と 95% 区間）と LML を足す（:g は余計な 0 を付けない数の書き方）
            row.update({"θ": f"{thn} {b['th_best']:g}［{b['th_lo']:g}〜{b['th_hi']:g}］・{tvn} {b['tv_best']:g}［{b['tv_lo']:g}〜{b['tv_hi']:g}］{un}", "LML": round(b["LML_best"], 1),
                        # ▶ （続き）係数 0 との LML の差（round は四捨五入）と、尤度で支持されたか（差が基準 3.0 以上か。bool は True／False にする）
                        "θ=0 との LML の差": round(b["dLML_vs_0"], 1), "尤度で支持": bool(b["dLML_vs_0"] >= LRT_DLML)})
        # ▶ シフトのないモデル（C）なら
        else:
            # ▶ C の LML を、ハイパーパラメータの表から取る
            row["LML"] = round(float(KT[(KT["材料"] == MAT_NAMES[mat]) & (KT["model"] == key)]["LML"].iloc[0]), 1)
        # ▶ そのモデルの検証の指標の表（範囲の名前から引ける形にする）
        t = MT[(MT["材料"] == MAT_NAMES[mat]) & (MT["モデル"] == key)].set_index("範囲")
        # ▶ 全体・低速側・高速側について
        for s in ["全体", "低速側", "高速側"]:
            # ▶ その範囲の行があれば（index は行の名前）
            if s in t.index:
                # ▶ RMSE を足す
                row[f"RMSE {s}"] = round(float(t.loc[s, "RMSE"]), 4)
        # ▶ 全体の被覆率を足す
        row["被覆率 全体"] = round(float(t.loc["全体", "coverage95"]), 3)
        # ▶ 全体の、粘度の実数値での R² を足す
        row["R²(実数) 全体"] = round(float(t.loc["全体", "R2_real"]), 4)
        # ▶ 1 行をリストに足す
        SUM.append(row)
# ▶ まとめの表にする
SUM = pd.DataFrame(SUM)
# ▶ ノートに表示する
display(SUM)
# ▶ ここから、学習用のノートの SHIFT_COL の候補を表示する（\n は改行。:g は数を短く書く書式）
print(f"\n学習用のノートの SHIFT_COL の候補（参考。θ = 0 との LML の差が基準 {LRT_DLML:g} 以上のシフトのうち、差が最も大きいもの。3data の誤差も見て、決めるのはユーザー）：")
# ▶ 材料ごとに
for mat in MATERIALS:
    # ▶ その材料の、シフトのあるモデルの行（isin は「リストに入っているか」）
    s = SUM[(SUM["材料"] == MAT_NAMES[mat]) & SUM["モデル"].isin(SHIFT_KEYS)]
    # ▶ そのうち、尤度で支持された行（行がなければ、空の表のまま）
    ok = s[s["尤度で支持"].astype(bool)] if len(s) else s
    # ▶ その材料の C の、3data の RMSE（全体）
    rc = SUM[(SUM["材料"] == MAT_NAMES[mat]) & (SUM["モデル"] == "C")]["RMSE 全体"]
    # ▶ C の RMSE を書いた文（C の行がなければ、空の文字）
    c_txt = f"C の 3data の RMSE 全体 {float(rc.iloc[0]):.3f}" if len(rc) else ""
    # ▶ 支持されたシフトがあれば
    if len(ok):
        # ▶ その中で、θ = 0 との LML の差が最も大きい行（idxmax は、最大の値の行の名前）
        r = ok.loc[ok["θ=0 との LML の差"].idxmax()]
        # ▶ 候補の列と形、そのモデルの LML の差を表示する（\" は、文字列の中の " の書き方。次の行に続く）
        print(f"  【{MAT_NAMES[mat]}】SHIFT_COL = \"{r['SHIFT_COL']}\"、SHIFT_STRUCT = \"{r['SHIFT_STRUCT']}\"（{r['モデル']}：θ = 0 との LML の差 {r['θ=0 との LML の差']:.1f}、"
              # ▶ （続き）そのモデルと C の、3data の RMSE
              f"3data の RMSE 全体 {r['RMSE 全体']:.3f}。{c_txt}）")
    # ▶ 支持されたシフトがなければ
    else:
        # ▶ どの列のシフトも支持されないこと、シフトなし（None＝C）も候補であることを表示する
        print(f"  【{MAT_NAMES[mat]}】どの列のシフトも、尤度で支持されない（θ = 0 との LML の差が {LRT_DLML:g} 未満）。SHIFT_COL = None（C）も候補（{c_txt}）")
# ▶ 材料ごとに
for mat in MATERIALS:
    # ▶ その材料（と、材料の指定がない）の判定を「判定の一覧.csv」に保存する（isin は「リストに入っているか」）
    to_csv(JT[JT["材料"].isin([MAT_NAMES[mat], ""])], os.path.join(out_dir(mat), "判定の一覧.csv"))
    # ▶ その材料のまとめを「まとめ.csv」に保存する
    to_csv(SUM[SUM["材料"] == MAT_NAMES[mat]], os.path.join(out_dir(mat), "まとめ.csv"))
# ▶ 結果の置き場所（材料ごとの、このノートの結果のフォルダ）を表示する（show_path で、リポジトリの中なら、リポジトリ直下から見た形）
print("結果の置き場所：" + "、".join(show_path(out_dir(m)) for m in MATERIALS))